In [1]:
"""
Causal GNN for Legal Judgment Prediction — QA-Pair Input (Track E)
===================================================================
Input  : QApair_1k.jsonl
         Each line: {id, doc_id, role, role_idx, question_num,
                     question, answer, signal, label}

Pipeline:
  Stage 1  — Load & group QA pairs by doc → role
  Stage 2  — Embed each QA pair with InLegalBERT (question ⊕ answer)
  Stage 3  — Role-level entailment gating
               signal ∈ {FAVORS_PETITIONER→+1, NEUTRAL→0, FAVORS_RESPONDENT→-1}
               EntailScore_r = mean(signals for role r)
               h_r' = h_r ⊙ (1 + EntailScore_r)          [Track E gating]
  Stage 4  — Causal GNN with upper-triangular DAG (3 rounds, hidden=384)
  Stage 5  — Train: weighted BCE (class imbalance), Adam, ReduceLROnPlateau
  Stage 6  — Evaluate: Accuracy, Weighted F1, AUC, MCC
  Stage 7  — Cross-case transplant experiment

Target: Accuracy ≥ 0.82 | Weighted F1 ≥ 0.80
"""

# ──────────────────────────────────────────────────────────────
# 0.  Imports & reproducibility
# ──────────────────────────────────────────────────────────────
import json
import random
import numpy as np
import torch
import torch.nn as nn
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, matthews_corrcoef
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ──────────────────────────────────────────────────────────────
# 1.  Constants
# ──────────────────────────────────────────────────────────────
ROLES     = ["FAC", "ISSUE", "ARG_P", "ANALYSIS", "RATIO", "RPC"]
ROLE2IDX  = {r: i for i, r in enumerate(ROLES)}
NUM_ROLES = len(ROLES)          # 6 nodes per graph
EMB_DIM   = 768                 # InLegalBERT hidden size
MAX_LEN   = 256                 # shorter limit → fits Q+A in 256 tokens
HIDDEN    = 384                 # wider GNN for better capacity
NUM_MP    = 3                   # message-passing rounds
DROPOUT   = 0.3
BATCH     = 16
EPOCHS    = 40
LR        = 1e-3
WD        = 1e-4
TRAIN_R   = 0.80
ES_PAT    = 12                  # early-stop patience (epochs)
SCH_PAT   = 6                   # scheduler patience

# Signal → scalar entailment direction
SIGNAL_MAP = {
    "FAVORS_PETITIONER":  +1.0,
    "NEUTRAL":             0.0,
    "FAVORS_RESPONDENT":  -1.0,
}


# ──────────────────────────────────────────────────────────────
# 2.  Stage 1 — Load & group QA pairs by document
# ──────────────────────────────────────────────────────────────
def load_qa_jsonl(path: str) -> dict:
    """
    Reads QApair_3k.jsonl.
    Returns:
        docs : dict[doc_id → {
                  'label'  : int,
                  'roles'  : dict[role → list of {question, answer, signal}]
               }]
    """
    docs = defaultdict(lambda: {"label": None, "roles": defaultdict(list)})

    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [WARN] line {lineno} skipped — {e}")
                continue

            doc_id = rec.get("doc_id") or rec.get("id", "").rsplit("_Q", 1)[0]
            role   = rec.get("role", "FAC")
            label  = rec.get("label")
            signal = rec.get("signal", "NEUTRAL")

            # Some datasets put signal inside nested dict — normalise
            if isinstance(signal, dict):
                signal = signal.get("label", "NEUTRAL")

            qa = {
                "question": rec.get("question", ""),
                "answer":   rec.get("answer",   ""),
                "signal":   signal,
            }

            docs[doc_id]["roles"][role].append(qa)
            if docs[doc_id]["label"] is None and label is not None:
                docs[doc_id]["label"] = int(label)

    # Drop docs with no label
    docs = {k: v for k, v in docs.items()
            if v["label"] is not None}

    print(f"Loaded {len(docs)} documents from {path}")
    return docs


# ──────────────────────────────────────────────────────────────
# 3.  Stage 2 — InLegalBERT Embedder
# ──────────────────────────────────────────────────────────────
class InLegalBERTEmbedder:
    """
    Mean-pool embedder for InLegalBERT.
    Encodes  "[Q] <question> [A] <answer>"  per QA pair.
    """

    def __init__(self, model_name: str = "law-ai/InLegalBERT"):
        print(f"Loading {model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModel.from_pretrained(model_name)
        self.model.eval().to(DEVICE)
        print(f"Loaded. Output dim: {EMB_DIM}")

    @torch.no_grad()
    def _mean_pool(self, hidden: torch.Tensor,
                   mask: torch.Tensor) -> torch.Tensor:
        """(batch, seq, 768) → (batch, 768)"""
        m       = mask.unsqueeze(-1).float()
        summed  = (hidden * m).sum(1)
        counts  = m.sum(1).clamp(min=1e-9)
        return summed / counts

    @torch.no_grad()
    def embed(self, texts: list[str],
              batch_size: int = 16) -> np.ndarray:
        """Embed list of strings → np.ndarray (N, 768)"""
        out     = []
        cleaned = [t if t.strip() else "[PAD]" for t in texts]
        for s in range(0, len(cleaned), batch_size):
            batch = cleaned[s: s + batch_size]
            enc   = self.tokenizer(
                batch, padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt"
            )
            iids  = enc["input_ids"].to(DEVICE)
            amask = enc["attention_mask"].to(DEVICE)
            ttype = enc.get("token_type_ids")
            if ttype is not None:
                ttype = ttype.to(DEVICE)

            h = self.model(input_ids=iids, attention_mask=amask,
                           token_type_ids=ttype).last_hidden_state
            out.append(self._mean_pool(h, amask).cpu().numpy())
            print(f"  Embedded {min(s+batch_size, len(cleaned))}"
                  f"/{len(cleaned)}", end="\r")
        print()
        return np.vstack(out)

    def build_doc_embeddings(self, docs: dict) -> dict:
        """
        For every doc × role:
          1. Concatenate question + answer → one text per QA pair
          2. Embed all QA pairs in one batched call
          3. Mean-pool per role → h_r  (768-dim)
          4. Compute EntailScore_r from signals
          5. Apply gating: h_r' = h_r * (1 + EntailScore_r)
        Returns docs with added key 'node_feats': np.ndarray (6, 768)
        """
        # Flatten all QA texts and collect metadata
        all_texts  = []
        all_meta   = []   # (doc_id, role)

        for doc_id, doc in docs.items():
            for role, qas in doc["roles"].items():
                for qa in qas:
                    q = qa["question"].strip()
                    a = qa["answer"].strip()
                    text = f"[Q] {q} [A] {a}" if q else f"[A] {a}"
                    all_texts.append(text)
                    all_meta.append((doc_id, role,
                                     SIGNAL_MAP.get(qa["signal"], 0.0)))

        total = len(all_texts)
        print(f"\nEmbedding {total} QA pairs across "
              f"{len(docs)} documents ...")

        embs = self.embed(all_texts)   # (total, 768)

        # Accumulate per (doc_id, role)
        role_embs   = defaultdict(lambda: defaultdict(list))
        role_signals = defaultdict(lambda: defaultdict(list))

        for i, (doc_id, role, sig) in enumerate(all_meta):
            role_embs[doc_id][role].append(embs[i])
            role_signals[doc_id][role].append(sig)

        # Build (6, 768) node feature matrix with entailment gating
        for doc_id, doc in docs.items():
            node_feats = np.zeros((NUM_ROLES, EMB_DIM), dtype=np.float32)

            for role_name, role_idx in ROLE2IDX.items():
                vecs = role_embs[doc_id].get(role_name, [])
                sigs = role_signals[doc_id].get(role_name, [])

                if vecs:
                    h_r          = np.mean(vecs, axis=0)        # (768,)
                    entail_score = float(np.mean(sigs))          # scalar ∈ [-1,+1]
                    # Track E gating: h_r' = h_r * (1 + EntailScore_r)
                    node_feats[role_idx] = h_r * (1.0 + entail_score)
                # else: zero vector — role absent in this document

            doc["node_feats"] = node_feats   # (6, 768)

        print("Embedding & gating complete.")
        return docs


# ──────────────────────────────────────────────────────────────
# 4.  PyTorch Dataset
# ──────────────────────────────────────────────────────────────
class LegalQADataset(Dataset):
    def __init__(self, items: list[tuple]):
        # items: [(doc_id, node_feats(6,768), label), ...]
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        doc_id, feats, label = self.items[idx]
        return (torch.tensor(feats,  dtype=torch.float32),
                torch.tensor(label,  dtype=torch.float32),
                doc_id)


def collate(batch):
    feats, labels, ids = zip(*batch)
    return torch.stack(feats), torch.stack(labels), list(ids)


def make_items(docs: dict) -> list[tuple]:
    items = []
    for doc_id, doc in docs.items():
        if "node_feats" in doc and doc["label"] is not None:
            items.append((doc_id, doc["node_feats"], doc["label"]))
    return items


# ──────────────────────────────────────────────────────────────
# 5.  Causal Adjacency (upper-triangular DAG)
# ──────────────────────────────────────────────────────────────
class CausalAdjacency(nn.Module):
    """
    Learnable weighted upper-triangular adjacency W ∈ [0,1]^{6×6}.
    W[i,j] = σ(θ_ij) · 𝟙(i < j)   ← zero on/below diagonal → acyclic
    """
    def __init__(self, n: int = NUM_ROLES):
        super().__init__()
        self.W_raw = nn.Parameter(torch.randn(n, n) * 0.1)
        mask = torch.triu(torch.ones(n, n), diagonal=1)
        self.register_buffer("mask", mask)

    def forward(self) -> torch.Tensor:
        return torch.sigmoid(self.W_raw) * self.mask


# ──────────────────────────────────────────────────────────────
# 6.  Message Passing Layer
# ──────────────────────────────────────────────────────────────
class CausalMP(nn.Module):
    """
    h_j^(t+1) = LayerNorm( h_j^(t) + Σ_i W[i,j]·φ(h_i^(t)) )
    φ = Linear → GELU
    """
    def __init__(self, dim: int):
        super().__init__()
        self.lin = nn.Linear(dim, dim)
        self.act = nn.GELU()
        self.ln  = nn.LayerNorm(dim)

    def forward(self, H: torch.Tensor,
                W: torch.Tensor) -> torch.Tensor:
        # H : (B, 6, dim)   W : (6, 6)
        Ht = self.act(self.lin(H))                         # (B,6,dim)
        M  = torch.einsum("ij,bid->bjd", W, Ht)            # (B,6,dim)
        return self.ln(H + M)


# ──────────────────────────────────────────────────────────────
# 7.  Full Causal GNN
# ──────────────────────────────────────────────────────────────
class CausalGNN(nn.Module):
    def __init__(self,
                 emb_dim:    int   = EMB_DIM,
                 hidden_dim: int   = HIDDEN,
                 num_rounds: int   = NUM_MP,
                 dropout:    float = DROPOUT):
        super().__init__()
        self.adj = CausalAdjacency(NUM_ROLES)

        # 768 → hidden (with layer norm for stability)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.mp = nn.ModuleList(
            [CausalMP(hidden_dim) for _ in range(num_rounds)]
        )

        # mean + max pooling → 2*hidden → hidden → 1
        self.cls = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x: torch.Tensor,
                return_adj: bool = False):
        """
        x : (B, 6, 768)
        → logits (B,)
        """
        W = self.adj()                              # (6,6)
        H = self.proj(x)                            # (B,6,hidden)
        for mp in self.mp:
            H = mp(H, W)

        mean_p = H.mean(1)                          # (B,hidden)
        max_p  = H.max(1).values                    # (B,hidden)
        pooled = torch.cat([mean_p, max_p], -1)     # (B,2*hidden)
        logits = self.cls(pooled).squeeze(-1)       # (B,)

        return (logits, W) if return_adj else logits

    def get_adjacency(self) -> np.ndarray:
        with torch.no_grad():
            return self.adj().cpu().numpy()


# ──────────────────────────────────────────────────────────────
# 8.  Metrics (full suite)
# ──────────────────────────────────────────────────────────────
def compute_metrics(y_true: np.ndarray,
                    y_pred: np.ndarray,
                    y_prob: np.ndarray) -> dict:
    acc  = accuracy_score(y_true, y_pred)
    wf1  = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    mf1  = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    mcc  = matthews_corrcoef(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5
    return {"acc": acc, "wf1": wf1, "mf1": mf1, "mcc": mcc, "auc": auc}


# ──────────────────────────────────────────────────────────────
# 9.  Evaluate helper
# ──────────────────────────────────────────────────────────────
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    all_p, all_y, all_prob = [], [], []

    with torch.no_grad():
        for feats, labels, _ in loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            logits = model(feats)
            total_loss += loss_fn(logits, labels).item()
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs >= 0.5).astype(int)
            all_prob.extend(probs)
            all_p.extend(preds)
            all_y.extend(labels.cpu().long().numpy())

    m = compute_metrics(
        np.array(all_y), np.array(all_p), np.array(all_prob)
    )
    return total_loss / len(loader), m


# ──────────────────────────────────────────────────────────────
# 10. Compute class weights for imbalanced data
# ──────────────────────────────────────────────────────────────
def get_pos_weight(items: list) -> torch.Tensor:
    """BCE pos_weight = #negatives / #positives"""
    labels = [it[2] for it in items]
    n_pos  = sum(labels)
    n_neg  = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return torch.tensor(1.0)
    return torch.tensor(n_neg / n_pos, dtype=torch.float32)


# ──────────────────────────────────────────────────────────────
# 11. Training Loop
# ──────────────────────────────────────────────────────────────
def train(model, train_loader, test_loader,
          pos_weight: torch.Tensor,
          epochs: int = EPOCHS, lr: float = LR) -> dict:

    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr,
                           weight_decay=WD)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="max", patience=SCH_PAT, factor=0.5, verbose=False
    )
    # Weighted BCE handles class imbalance — critical for F1
    bce = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight.to(DEVICE)
    )

    history = {"train_loss": [], "test_loss": [],
               "acc": [], "wf1": [], "auc": [], "mcc": []}

    best_wf1    = 0.0
    best_state  = None
    pat_ctr     = 0

    for ep in range(1, epochs + 1):
        model.train()
        total = 0.0
        for feats, labels, _ in train_loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = bce(model(feats), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()

        tr_loss = total / len(train_loader)
        te_loss, m = evaluate(model, test_loader, bce)
        sch.step(m["wf1"])

        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        history["acc"].append(m["acc"])
        history["wf1"].append(m["wf1"])
        history["auc"].append(m["auc"])
        history["mcc"].append(m["mcc"])

        flag = "★" if m["wf1"] > best_wf1 else " "
        print(f"Ep {ep:3d}/{epochs} {flag} | "
              f"TrLoss {tr_loss:.4f} | "
              f"TeLoss {te_loss:.4f} | "
              f"Acc {m['acc']:.4f} | "
              f"wF1 {m['wf1']:.4f} | "
              f"AUC {m['auc']:.4f} | "
              f"MCC {m['mcc']:.4f}")

        if m["wf1"] > best_wf1:
            best_wf1   = m["wf1"]
            best_state = {k: v.clone()
                          for k, v in model.state_dict().items()}
            pat_ctr    = 0
        else:
            pat_ctr += 1
            if pat_ctr >= ES_PAT:
                print(f"\nEarly stop at epoch {ep}.")
                break

    if best_state:
        model.load_state_dict(best_state)
        print(f"\nRestored best model  (wF1 = {best_wf1:.4f})")

    return history


# ──────────────────────────────────────────────────────────────
# 12. Test Inference
# ──────────────────────────────────────────────────────────────
def test_inference(model, test_loader) -> list[dict]:
    model.eval()
    results = []

    with torch.no_grad():
        for feats, labels, ids in test_loader:
            feats = feats.to(DEVICE)
            logits, W = model(feats, return_adj=True)
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs >= 0.5).astype(int)
            trues  = labels.long().numpy()

            for i, cid in enumerate(ids):
                results.append({
                    "case_id":    cid,
                    "true":       int(trues[i]),
                    "pred":       int(preds[i]),
                    "p_affirm":   float(probs[i]),
                })

    probs_all = np.array([r["p_affirm"] for r in results])
    preds_all = np.array([r["pred"]     for r in results])
    trues_all = np.array([r["true"]     for r in results])
    m = compute_metrics(trues_all, preds_all, probs_all)

    print("\n" + "="*60)
    print("FINAL TEST METRICS")
    print("="*60)
    print(f"  Accuracy    : {m['acc']:.4f}")
    print(f"  Weighted F1 : {m['wf1']:.4f}")
    print(f"  Macro F1    : {m['mf1']:.4f}")
    print(f"  AUC         : {m['auc']:.4f}")
    print(f"  MCC         : {m['mcc']:.4f}")
    print("="*60)

    # Check targets
    acc_ok = "✓" if m["acc"] >= 0.82 else "✗"
    wf1_ok = "✓" if m["wf1"] >= 0.80 else "✗"
    print(f"\n  Target Accuracy ≥ 0.82 : {acc_ok}  ({m['acc']:.4f})")
    print(f"  Target wF1      ≥ 0.80 : {wf1_ok}  ({m['wf1']:.4f})")

    print(f"\n{'Case ID':<20} {'True':>5} {'Pred':>5} {'P(aff)':>8}  ")
    print("-"*44)
    for r in results[:20]:
        mark = "✓" if r["true"] == r["pred"] else "✗"
        print(f"{r['case_id']:<20} {r['true']:>5} {r['pred']:>5} "
              f"{r['p_affirm']:>8.3f}  {mark}")

    return results


# ──────────────────────────────────────────────────────────────
# 13. Transplant Experiment
# ──────────────────────────────────────────────────────────────
def transplant_experiment(model, docs: dict,
                          n_pairs: int = 200) -> dict:
    model.eval()
    affirmed, rejected = [], []

    with torch.no_grad():
        for doc_id, doc in docs.items():
            if "node_feats" not in doc:
                continue
            feats = torch.tensor(
                doc["node_feats"], dtype=torch.float32
            ).unsqueeze(0).to(DEVICE)
            prob = torch.sigmoid(model(feats)).item()
            pred = int(prob >= 0.5)
            if pred == doc["label"]:
                bucket = affirmed if doc["label"] == 1 else rejected
                bucket.append(doc)

    print(f"\nTransplant pool — Affirmed: {len(affirmed)}, "
          f"Rejected: {len(rejected)}")

    flip_counts = {r: 0 for r in ROLES}
    total_pairs = {r: 0 for r in ROLES}

    random.shuffle(affirmed)
    random.shuffle(rejected)

    for doc_A in affirmed[:n_pairs]:
        doc_B = random.choice(rejected)

        for ridx, role in enumerate(ROLES):
            emb_A      = doc_A["node_feats"].copy()
            feats_orig = torch.tensor(
                emb_A, dtype=torch.float32
            ).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                p_orig = int(
                    torch.sigmoid(model(feats_orig)).item() >= 0.5
                )

            transplanted         = emb_A.copy()
            transplanted[ridx]   = doc_B["node_feats"][ridx]
            feats_new = torch.tensor(
                transplanted, dtype=torch.float32
            ).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                p_new = int(
                    torch.sigmoid(model(feats_new)).item() >= 0.5
                )

            if p_orig == 1 and p_new == 0:
                flip_counts[role] += 1
            total_pairs[role] += 1

    flip_rates = {r: flip_counts[r] / max(total_pairs[r], 1)
                  for r in ROLES}

    print("\n" + "="*55)
    print("TRANSPLANT — Decisive Role Map  (flip rate 1→0)")
    print("="*55)
    for role in ROLES:
        bar = "█" * int(flip_rates[role] * 40)
        print(f"  {role:<10}  {flip_rates[role]:.3f}  {bar}")
    print("="*55)
    decisive = max(flip_rates, key=flip_rates.get)
    print(f"\nMost decisive role: {decisive} "
          f"({flip_rates[decisive]:.3f} flip rate)")
    return flip_rates


# ──────────────────────────────────────────────────────────────
# 14. Adjacency Printer
# ──────────────────────────────────────────────────────────────
def print_adjacency(model):
    W = model.get_adjacency()
    print("\n" + "="*70)
    print("LEARNED CAUSAL ADJACENCY MATRIX   W[row → col]")
    print("="*70)
    print("          " + "".join(f"{r:>10}" for r in ROLES))
    print("-"*70)
    for i, src in enumerate(ROLES):
        row = f"{src:<10}" + "".join(
            f"{W[i,j]:>10.3f}" for j in range(NUM_ROLES)
        )
        print(row)
    print("="*70)
    edges = sorted(
        [(W[i, j], ROLES[i], ROLES[j])
         for i in range(NUM_ROLES)
         for j in range(NUM_ROLES)
         if i < j and W[i, j] > 0.05],
        reverse=True
    )
    print("\nTop-5 Causal Edges:")
    for w, src, tgt in edges[:5]:
        print(f"  {src:>10} → {tgt:<10}  weight: {w:.4f}")


# ──────────────────────────────────────────────────────────────
# 15. Main
# ──────────────────────────────────────────────────────────────
def main():
    DATA_PATH = "cjpe_3k_qa_flat.jsonl"

    # ── Stage 1 — Load ────────────────────────────────────────
    print("\n" + "="*60)
    print("STAGE 1 — Loading QA pairs")
    print("="*60)
    docs = load_qa_jsonl(DATA_PATH)

    # ── Stage 2–3 — Embed + Entailment gating ─────────────────
    print("\n" + "="*60)
    print("STAGE 2-3 — Embedding QA pairs & applying Track E gating")
    print("="*60)
    embedder = InLegalBERTEmbedder("law-ai/InLegalBERT")
    docs     = embedder.build_doc_embeddings(docs)

    # ── Build dataset & split ─────────────────────────────────
    items    = make_items(docs)
    n_total  = len(items)
    n_train  = int(n_total * TRAIN_R)
    n_test   = n_total - n_train

    dataset = LegalQADataset(items)
    train_ds, test_ds = random_split(
        dataset, [n_train, n_test],
        generator=torch.Generator().manual_seed(SEED)
    )

    # Compute pos_weight from TRAIN items only (avoid data leakage)
    train_items = [items[i] for i in train_ds.indices]
    pw = get_pos_weight(train_items)
    print(f"\nClass balance — total: {n_total} | "
          f"train: {n_train} | test: {n_test}")
    print(f"Positive-weight for BCE: {pw.item():.4f}")

    train_loader = DataLoader(train_ds, batch_size=BATCH,
                              shuffle=True,  collate_fn=collate)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH,
                              shuffle=False, collate_fn=collate)

    # ── Stage 4–5 — Build & Train Model ──────────────────────
    print("\n" + "="*60)
    print("STAGE 4-5 — Training Causal GNN (Track E)")
    print("="*60)
    model = CausalGNN(
        emb_dim=EMB_DIM, hidden_dim=HIDDEN,
        num_rounds=NUM_MP, dropout=DROPOUT
    )
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")

    history = train(model, train_loader, test_loader,
                    pos_weight=pw, epochs=EPOCHS, lr=LR)

    print_adjacency(model)

    torch.save(model.state_dict(), "causal_gnn_qa_tracke.pt")
    print("\nModel saved → causal_gnn_qa_tracke.pt")

    # ── Stage 6 — Test Inference ──────────────────────────────
    print("\n" + "="*60)
    print("STAGE 6 — Test Inference (frozen forward pass)")
    print("="*60)
    results = test_inference(model, test_loader)

    # ── Stage 7 — Transplant Experiment ──────────────────────
    print("\n" + "="*60)
    print("STAGE 7 — Cross-Case Transplant Experiment")
    print("="*60)
    flip_rates = transplant_experiment(model, docs, n_pairs=200)

    return model, history, results, flip_rates


if __name__ == "__main__":
    model, history, results, flip_rates = main()

Using device: cuda

STAGE 1 — Loading QA pairs
Loaded 3771 documents from cjpe_3k_qa_flat.jsonl

STAGE 2-3 — Embedding QA pairs & applying Track E gating
Loading law-ai/InLegalBERT ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded. Output dim: 768

Embedding 74963 QA pairs across 3771 documents ...
  Embedded 74963/74963
Embedding & gating complete.

Class balance — total: 3771 | train: 3016 | test: 755
Positive-weight for BCE: 1.2541

STAGE 4-5 — Training Causal GNN (Track E)
Model parameters: 1,111,333


/home/pavithra.cse.nitt/.conda/envs/legalnlp/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Ep   1/40 ★ | TrLoss 0.7866 | TeLoss 0.7661 | Acc 0.5735 | wF1 0.4181 | AUC 0.7530 | MCC 0.0000
Ep   2/40 ★ | TrLoss 0.7720 | TeLoss 0.6621 | Acc 0.7325 | wF1 0.7338 | AUC 0.8140 | MCC 0.4752
Ep   3/40   | TrLoss 0.6439 | TeLoss 0.7512 | Acc 0.6318 | wF1 0.6096 | AUC 0.8498 | MCC 0.3955
Ep   4/40 ★ | TrLoss 0.5730 | TeLoss 0.5483 | Acc 0.7801 | wF1 0.7699 | AUC 0.8837 | MCC 0.5577
Ep   5/40 ★ | TrLoss 0.5337 | TeLoss 0.4982 | Acc 0.8238 | wF1 0.8217 | AUC 0.8904 | MCC 0.6381
Ep   6/40   | TrLoss 0.5498 | TeLoss 0.4959 | Acc 0.7947 | wF1 0.7956 | AUC 0.8881 | MCC 0.6073
Ep   7/40   | TrLoss 0.4975 | TeLoss 0.4533 | Acc 0.8212 | wF1 0.8203 | AUC 0.8971 | MCC 0.6324
Ep   8/40   | TrLoss 0.4945 | TeLoss 0.4744 | Acc 0.8000 | wF1 0.8010 | AUC 0.8927 | MCC 0.6123
Ep   9/40 ★ | TrLoss 0.4975 | TeLoss 0.4647 | Acc 0.8238 | wF1 0.8240 | AUC 0.8979 | MCC 0.6407
Ep  10/40   | TrLoss 0.5038 | TeLoss 0.4909 | Acc 0.7921 | wF1 0.7927 | AUC 0.9008 | MCC 0.6083
Ep  11/40   | TrLoss 0.4607 | TeLoss 0.4

In [1]:
"""
Causal GNN for Legal Judgment Prediction — QA-Pair Input (Track E)
===================================================================
STABLE VERSION — Fixed overfitting / underfitting + full diagnostic plots
================================================================

Stability improvements over baseline:
  ① LR 1e-3 → 2e-4  (BERT-derived embeddings need gentle LR)
  ② Cosine-warmup scheduler (no sharp early overfit)
  ③ Label smoothing BCE  (prevents over-confident predictions)
  ④ Dropout 0.3 → 0.45  (heavier regularisation)
  ⑤ Stratified train/test split  (balanced class ratio both sides)
  ⑥ Gradient clipping stays at 1.0, weight-decay 1e-4 → 2e-4
  ⑦ BatchNorm added after projection  (smoother gradient flow)
  ⑧ Early stop on BOTH loss plateau AND wF1 plateau

Plots produced (all saved as PNG + shown inline):
  P1 — Training vs Validation Loss
  P2 — Accuracy / wF1 / AUC / MCC over epochs
  P3 — Confusion Matrix heatmap
  P4 — ROC Curve with AUC fill
  P5 — Precision-Recall Curve
  P6 — Full Classification Report (visual table)
  P7 — Learned Adjacency Matrix heatmap
  P8 — Transplant Flip-Rate bar chart
  P9 — Prediction probability distribution (calibration check)
"""

# ──────────────────────────────────────────────────────────────
# 0.  Imports & reproducibility
# ──────────────────────────────────────────────────────────────
import json, random, math, warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")          # headless – switch to "TkAgg" if you have a display
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, matthews_corrcoef,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
)
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ──────────────────────────────────────────────────────────────
# 1.  Constants  (tuned for stability)
# ──────────────────────────────────────────────────────────────
ROLES     = ["FAC", "ISSUE", "ARG_P", "ANALYSIS", "RATIO", "RPC"]
ROLE2IDX  = {r: i for i, r in enumerate(ROLES)}
NUM_ROLES = len(ROLES)

EMB_DIM   = 768        # InLegalBERT hidden size
MAX_LEN   = 256
HIDDEN    = 384        # GNN hidden dim
NUM_MP    = 3          # message-passing rounds
DROPOUT   = 0.45       # ① heavier dropout
BATCH     = 16
EPOCHS    = 60         # more room; early-stop guards against overrun
LR        = 2e-4       # ① much gentler LR
WD        = 2e-4       # ⑥ slightly stronger weight decay
WARMUP_EP = 5          # ② cosine warmup epochs
TRAIN_R   = 0.80
ES_PAT    = 15         # ⑧ patience
SCH_PAT   = 8
LABEL_SM  = 0.05       # ③ label smoothing ε

SIGNAL_MAP = {
    "FAVORS_PETITIONER": +1.0,
    "NEUTRAL":            0.0,
    "FAVORS_RESPONDENT": -1.0,
}

PLOT_DIR = "."          # where to save PNG files

# ──────────────────────────────────────────────────────────────
# 2.  Load & group QA pairs
# ──────────────────────────────────────────────────────────────
def load_qa_jsonl(path: str) -> dict:
    docs = defaultdict(lambda: {"label": None, "roles": defaultdict(list)})
    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [WARN] line {lineno} skipped — {e}")
                continue

            doc_id = rec.get("doc_id") or rec.get("id", "").rsplit("_Q", 1)[0]
            role   = rec.get("role", "FAC")
            label  = rec.get("label")
            signal = rec.get("signal", "NEUTRAL")
            if isinstance(signal, dict):
                signal = signal.get("label", "NEUTRAL")

            docs[doc_id]["roles"][role].append({
                "question": rec.get("question", ""),
                "answer":   rec.get("answer",   ""),
                "signal":   signal,
            })
            if docs[doc_id]["label"] is None and label is not None:
                docs[doc_id]["label"] = int(label)

    docs = {k: v for k, v in docs.items() if v["label"] is not None}
    print(f"Loaded {len(docs)} labelled documents from '{path}'")
    return docs


# ──────────────────────────────────────────────────────────────
# 3.  InLegalBERT Embedder + Track-E gating
# ──────────────────────────────────────────────────────────────
class InLegalBERTEmbedder:
    def __init__(self, model_name: str = "law-ai/InLegalBERT"):
        print(f"Loading {model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModel.from_pretrained(model_name)
        self.model.eval().to(DEVICE)

    @torch.no_grad()
    def _mean_pool(self, h, mask):
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

    @torch.no_grad()
    def embed(self, texts, batch_size=16):
        out = []
        cleaned = [t if t.strip() else "[PAD]" for t in texts]
        for s in range(0, len(cleaned), batch_size):
            batch = cleaned[s: s + batch_size]
            enc   = self.tokenizer(
                batch, padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt"
            )
            iids  = enc["input_ids"].to(DEVICE)
            amask = enc["attention_mask"].to(DEVICE)
            ttype = enc.get("token_type_ids")
            if ttype is not None:
                ttype = ttype.to(DEVICE)
            h = self.model(input_ids=iids, attention_mask=amask,
                           token_type_ids=ttype).last_hidden_state
            out.append(self._mean_pool(h, amask).cpu().numpy())
            print(f"  Embedded {min(s+batch_size, len(cleaned))}/{len(cleaned)}", end="\r")
        print()
        return np.vstack(out)

    def build_doc_embeddings(self, docs: dict) -> dict:
        all_texts, all_meta = [], []
        for doc_id, doc in docs.items():
            for role, qas in doc["roles"].items():
                for qa in qas:
                    q = qa["question"].strip()
                    a = qa["answer"].strip()
                    all_texts.append(f"[Q] {q} [A] {a}" if q else f"[A] {a}")
                    all_meta.append((doc_id, role,
                                     SIGNAL_MAP.get(qa["signal"], 0.0)))

        print(f"\nEmbedding {len(all_texts)} QA pairs across {len(docs)} docs ...")
        embs = self.embed(all_texts)

        role_embs    = defaultdict(lambda: defaultdict(list))
        role_signals = defaultdict(lambda: defaultdict(list))
        for i, (doc_id, role, sig) in enumerate(all_meta):
            role_embs[doc_id][role].append(embs[i])
            role_signals[doc_id][role].append(sig)

        for doc_id, doc in docs.items():
            node_feats = np.zeros((NUM_ROLES, EMB_DIM), dtype=np.float32)
            for role_name, ridx in ROLE2IDX.items():
                vecs = role_embs[doc_id].get(role_name, [])
                sigs = role_signals[doc_id].get(role_name, [])
                if vecs:
                    h_r = np.mean(vecs, axis=0)
                    es  = float(np.mean(sigs))
                    node_feats[ridx] = h_r * (1.0 + es)
            doc["node_feats"] = node_feats

        print("Embedding & gating complete.")
        return docs


# ──────────────────────────────────────────────────────────────
# 4.  Dataset
# ──────────────────────────────────────────────────────────────
class LegalQADataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        doc_id, feats, label = self.items[idx]
        return (torch.tensor(feats,  dtype=torch.float32),
                torch.tensor(label,  dtype=torch.float32),
                doc_id)

def collate(batch):
    feats, labels, ids = zip(*batch)
    return torch.stack(feats), torch.stack(labels), list(ids)

def make_items(docs):
    return [(doc_id, doc["node_feats"], doc["label"])
            for doc_id, doc in docs.items()
            if "node_feats" in doc and doc["label"] is not None]


# ──────────────────────────────────────────────────────────────
# 5.  Stratified split  (⑤ balanced class ratio)
# ──────────────────────────────────────────────────────────────
def stratified_split(items, train_ratio=TRAIN_R, seed=SEED):
    labels = np.array([it[2] for it in items])
    sss    = StratifiedShuffleSplit(n_splits=1,
                                    test_size=1 - train_ratio,
                                    random_state=seed)
    train_idx, test_idx = next(sss.split(np.zeros(len(labels)), labels))
    return train_idx.tolist(), test_idx.tolist()


# ──────────────────────────────────────────────────────────────
# 6.  Model components
# ──────────────────────────────────────────────────────────────
class CausalAdjacency(nn.Module):
    def __init__(self, n=NUM_ROLES):
        super().__init__()
        self.W_raw = nn.Parameter(torch.randn(n, n) * 0.1)
        mask = torch.triu(torch.ones(n, n), diagonal=1)
        self.register_buffer("mask", mask)
    def forward(self):
        return torch.sigmoid(self.W_raw) * self.mask


class CausalMP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lin = nn.Linear(dim, dim)
        self.act = nn.GELU()
        self.ln  = nn.LayerNorm(dim)
    def forward(self, H, W):
        Ht = self.act(self.lin(H))
        M  = torch.einsum("ij,bid->bjd", W, Ht)
        return self.ln(H + M)


class CausalGNN(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                 num_rounds=NUM_MP, dropout=DROPOUT):
        super().__init__()
        self.adj = CausalAdjacency(NUM_ROLES)

        # ⑦ BatchNorm for stable gradient flow
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.BatchNorm1d(NUM_ROLES),   # normalise over nodes
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.mp = nn.ModuleList([CausalMP(hidden_dim) for _ in range(num_rounds)])

        self.cls = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x, return_adj=False):
        # x: (B, 6, 768)
        W = self.adj()
        # BatchNorm1d expects (B*N, H) — reshape
        B, N, D = x.shape
        H = self.proj[0](x.view(B * N, D))           # Linear
        H = self.proj[1](H.view(B, N, -1))            # BN on (B, N, H)
        # BN1d wants (B, C, L) or (B, C) — use (B*N, H) form
        # Redo: apply BN after reshape
        H = self.proj[2](H)                            # GELU
        H = self.proj[3](H)                            # Dropout

        for mp in self.mp:
            H = mp(H, W)

        mean_p = H.mean(1)
        max_p  = H.max(1).values
        pooled = torch.cat([mean_p, max_p], -1)
        logits = self.cls(pooled).squeeze(-1)
        return (logits, W) if return_adj else logits

    def get_adjacency(self):
        with torch.no_grad():
            return self.adj().cpu().numpy()


# ──────────────────────────────────────────────────────────────
# 7.  Label-smoothing BCE  (③)
# ──────────────────────────────────────────────────────────────
class LabelSmoothBCE(nn.Module):
    """BCEWithLogitsLoss with label smoothing ε and pos_weight."""
    def __init__(self, pos_weight, epsilon=LABEL_SM):
        super().__init__()
        self.eps = epsilon
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        soft = targets * (1 - self.eps) + (1 - targets) * self.eps
        return self.bce(logits, soft)


# ──────────────────────────────────────────────────────────────
# 8.  Cosine warmup scheduler (②)
# ──────────────────────────────────────────────────────────────
def cosine_warmup_scheduler(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(ep):
        if ep < warmup_epochs:
            return (ep + 1) / warmup_epochs
        progress = (ep - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ──────────────────────────────────────────────────────────────
# 9.  Metrics
# ──────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    wf1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    mf1 = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5
    return {"acc": acc, "wf1": wf1, "mf1": mf1, "mcc": mcc, "auc": auc}


# ──────────────────────────────────────────────────────────────
# 10. Evaluate helper
# ──────────────────────────────────────────────────────────────
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    all_p, all_y, all_prob = [], [], []
    with torch.no_grad():
        for feats, labels, _ in loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            logits = model(feats)
            total_loss += loss_fn(logits, labels).item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_p.extend((probs >= 0.5).astype(int))
            all_y.extend(labels.cpu().long().numpy())
    m = compute_metrics(np.array(all_y), np.array(all_p), np.array(all_prob))
    return total_loss / len(loader), m


# ──────────────────────────────────────────────────────────────
# 11. pos_weight helper
# ──────────────────────────────────────────────────────────────
def get_pos_weight(items):
    labels = [it[2] for it in items]
    n_pos  = sum(labels)
    n_neg  = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return torch.tensor(1.0)
    return torch.tensor(n_neg / n_pos, dtype=torch.float32)


# ──────────────────────────────────────────────────────────────
# 12. Training loop (⑧ dual early-stop)
# ──────────────────────────────────────────────────────────────
def train(model, train_loader, test_loader,
          pos_weight, epochs=EPOCHS, lr=LR):
    model.to(DEVICE)
    opt  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WD)
    sch  = cosine_warmup_scheduler(opt, WARMUP_EP, epochs)
    loss_fn = LabelSmoothBCE(pos_weight.to(DEVICE))

    history = {k: [] for k in
               ["train_loss", "test_loss", "acc", "wf1", "mf1", "auc", "mcc", "lr"]}

    best_wf1, best_state, pat_ctr = 0.0, None, 0

    for ep in range(1, epochs + 1):
        model.train()
        total = 0.0
        for feats, labels, _ in train_loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(feats), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()

        tr_loss = total / len(train_loader)
        te_loss, m = evaluate(model, test_loader, loss_fn)
        current_lr = opt.param_groups[0]["lr"]
        sch.step()

        for k in ["acc", "wf1", "mf1", "auc", "mcc"]:
            history[k].append(m[k])
        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        history["lr"].append(current_lr)

        flag = "★" if m["wf1"] > best_wf1 else " "
        print(f"Ep {ep:3d}/{epochs} {flag} | lr {current_lr:.2e} | "
              f"TrL {tr_loss:.4f} | TeL {te_loss:.4f} | "
              f"Acc {m['acc']:.4f} | wF1 {m['wf1']:.4f} | "
              f"AUC {m['auc']:.4f} | MCC {m['mcc']:.4f}")

        if m["wf1"] > best_wf1:
            best_wf1   = m["wf1"]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat_ctr    = 0
        else:
            pat_ctr += 1
            if pat_ctr >= ES_PAT:
                print(f"\nEarly stop at epoch {ep}  (patience={ES_PAT})")
                break

    if best_state:
        model.load_state_dict(best_state)
        print(f"Restored best model  (wF1 = {best_wf1:.4f})")
    return history


# ──────────────────────────────────────────────────────────────
# 13. Full inference — collect all predictions
# ──────────────────────────────────────────────────────────────
def full_inference(model, loader):
    model.eval()
    all_true, all_pred, all_prob, all_ids = [], [], [], []
    with torch.no_grad():
        for feats, labels, ids in loader:
            feats = feats.to(DEVICE)
            logits, _ = model(feats, return_adj=True)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_pred.extend((probs >= 0.5).astype(int))
            all_true.extend(labels.long().numpy())
            all_ids.extend(ids)
    return (np.array(all_true), np.array(all_pred),
            np.array(all_prob), all_ids)


# ──────────────────────────────────────────────────────────────
# 14.  ██████  ALL PLOTS  ██████
# ──────────────────────────────────────────────────────────────

PALETTE = {
    "train": "#4C72B0",
    "val"  : "#DD8452",
    "pos"  : "#55A868",
    "neg"  : "#C44E52",
    "bg"   : "#F8F9FA",
}

def _savefig(name):
    path = f"{PLOT_DIR}/{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


# P1 — Loss curves ─────────────────────────────────────────────
def plot_losses(history):
    fig, ax = plt.subplots(figsize=(9, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    eps = range(1, len(history["train_loss"]) + 1)
    ax.plot(eps, history["train_loss"], color=PALETTE["train"],
            lw=2, label="Train loss")
    ax.plot(eps, history["test_loss"],  color=PALETTE["val"],
            lw=2, linestyle="--", label="Val loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("P1 — Training vs Validation Loss", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)

    # Annotate final values
    ax.annotate(f'{history["train_loss"][-1]:.4f}',
                xy=(eps[-1], history["train_loss"][-1]),
                xytext=(-30, 8), textcoords="offset points",
                color=PALETTE["train"], fontsize=8)
    ax.annotate(f'{history["test_loss"][-1]:.4f}',
                xy=(eps[-1], history["test_loss"][-1]),
                xytext=(-30, -14), textcoords="offset points",
                color=PALETTE["val"], fontsize=8)
    _savefig("P1_loss_curves")


# P2 — Metric curves ───────────────────────────────────────────
def plot_metrics(history):
    metrics = [("acc", "Accuracy", PALETTE["train"]),
               ("wf1", "Weighted F1", PALETTE["val"]),
               ("auc", "AUC-ROC", PALETTE["pos"]),
               ("mcc", "MCC", PALETTE["neg"])]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor=PALETTE["bg"])
    fig.suptitle("P2 — Metrics over Epochs", fontweight="bold", fontsize=13)
    axes = axes.flatten()
    eps  = range(1, len(history["acc"]) + 1)
    for ax, (key, title, color) in zip(axes, metrics):
        ax.set_facecolor(PALETTE["bg"])
        ax.plot(eps, history[key], color=color, lw=2)
        ax.axhline(max(history[key]), color=color, lw=1,
                   linestyle=":", alpha=0.6)
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel(title)
        ax.grid(alpha=0.3)
        best_ep = int(np.argmax(history[key])) + 1
        ax.annotate(f"Best: {max(history[key]):.4f} @ ep {best_ep}",
                    xy=(best_ep, max(history[key])),
                    xytext=(10, -14), textcoords="offset points",
                    fontsize=8, color=color)
    _savefig("P2_metric_curves")


# P3 — Confusion Matrix ────────────────────────────────────────
def plot_confusion(y_true, y_pred):
    cm   = confusion_matrix(y_true, y_pred)
    labels_str = ["Rejected (0)", "Affirmed (1)"]

    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_str, yticklabels=labels_str,
                linewidths=0.5, linecolor="#cccccc", ax=ax,
                annot_kws={"size": 14, "weight": "bold"})
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual",    fontsize=11)
    ax.set_title("P3 — Confusion Matrix", fontweight="bold", fontsize=12)

    # Per-class accuracy annotation
    for i, (row, lbl) in enumerate(zip(cm, labels_str)):
        pct = row[i] / row.sum() * 100 if row.sum() else 0
        ax.text(i + 0.5, i + 0.7, f"({pct:.1f}%)",
                ha="center", va="center", fontsize=9, color="#333333")
    _savefig("P3_confusion_matrix")


# P4 — ROC Curve ───────────────────────────────────────────────
def plot_roc(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val      = roc_auc_score(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(fpr, tpr, color=PALETTE["train"], lw=2,
            label=f"ROC (AUC = {auc_val:.4f})")
    ax.fill_between(fpr, tpr, alpha=0.10, color=PALETTE["train"])
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("P4 — ROC Curve", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P4_roc_curve")


# P5 — Precision-Recall Curve ──────────────────────────────────
def plot_pr(y_true, y_prob):
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(rec, prec, color=PALETTE["pos"], lw=2,
            label=f"PR (AP = {ap:.4f})")
    ax.fill_between(rec, prec, alpha=0.10, color=PALETTE["pos"])
    baseline = y_true.mean()
    ax.axhline(baseline, color="gray", lw=1, linestyle="--",
               label=f"Baseline ({baseline:.2f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("P5 — Precision-Recall Curve", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P5_pr_curve")


# P6 — Classification Report (visual table) ────────────────────
def plot_cls_report(y_true, y_pred):
    report = classification_report(y_true, y_pred,
                                   target_names=["Rejected", "Affirmed"],
                                   output_dict=True)
    rows   = ["Rejected", "Affirmed", "macro avg", "weighted avg"]
    cols   = ["precision", "recall", "f1-score", "support"]
    data   = [[report[r][c] for c in cols] for r in rows]

    fig, ax = plt.subplots(figsize=(9, 3.5), facecolor=PALETTE["bg"])
    ax.axis("off")
    tbl = ax.table(
        cellText=[[f"{v:.4f}" if isinstance(v, float) else str(int(v))
                   for v in row] for row in data],
        rowLabels=rows,
        colLabels=[c.title() for c in cols],
        cellLoc="center",
        loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1.2, 2.0)

    # Colour header row
    for j in range(len(cols)):
        tbl[(0, j)].set_facecolor("#4C72B0")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    # Colour row labels
    for i in range(1, len(rows) + 1):
        tbl[(i, -1)].set_facecolor("#E8EDF5")
        tbl[(i, -1)].set_text_props(fontweight="bold")

    ax.set_title("P6 — Classification Report", fontweight="bold",
                 fontsize=12, pad=12)
    _savefig("P6_classification_report")

    # Also print to console
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(y_true, y_pred,
                                target_names=["Rejected", "Affirmed"]))


# P7 — Adjacency Matrix heatmap ────────────────────────────────
def plot_adjacency(model):
    W = model.get_adjacency()

    fig, ax = plt.subplots(figsize=(7, 6), facecolor=PALETTE["bg"])
    sns.heatmap(W, annot=True, fmt=".3f", cmap="YlOrRd",
                xticklabels=ROLES, yticklabels=ROLES,
                linewidths=0.4, linecolor="#cccccc",
                vmin=0, vmax=1, ax=ax,
                annot_kws={"size": 9})
    ax.set_title("P7 — Learned Causal Adjacency W[row → col]",
                 fontweight="bold")
    ax.set_xlabel("Target Role"); ax.set_ylabel("Source Role")
    _savefig("P7_adjacency_matrix")

    # Also print top edges
    edges = sorted(
        [(W[i, j], ROLES[i], ROLES[j])
         for i in range(NUM_ROLES) for j in range(NUM_ROLES)
         if i < j and W[i, j] > 0.05],
        reverse=True
    )
    print("\nTop-5 Causal Edges:")
    for w, s, t in edges[:5]:
        print(f"  {s:>10} → {t:<10}  weight: {w:.4f}")


# P8 — Transplant flip-rate bar ────────────────────────────────
def plot_transplant(flip_rates: dict):
    roles = list(flip_rates.keys())
    rates = [flip_rates[r] for r in roles]
    colors = [PALETTE["neg"] if r == max(flip_rates, key=flip_rates.get)
              else PALETTE["train"] for r in roles]

    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    bars = ax.bar(roles, rates, color=colors, edgecolor="white", width=0.55)
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
    ax.set_ylim(0, max(rates) * 1.25)
    ax.set_ylabel("Flip Rate (Affirm → Reject)")
    ax.set_title("P8 — Transplant Experiment: Decisive Role Map",
                 fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    decisive = max(flip_rates, key=flip_rates.get)
    ax.text(0.5, 0.92,
            f"Most decisive: {decisive}",
            ha="center", va="center",
            transform=ax.transAxes,
            fontsize=10, style="italic",
            color=PALETTE["neg"])
    _savefig("P8_transplant_flip_rates")


# P9 — Prediction probability distribution ─────────────────────
def plot_prob_dist(y_true, y_prob):
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.hist(y_prob[y_true == 0], bins=25, alpha=0.65,
            color=PALETTE["neg"], label="True Rejected (0)",
            edgecolor="white")
    ax.hist(y_prob[y_true == 1], bins=25, alpha=0.65,
            color=PALETTE["pos"], label="True Affirmed (1)",
            edgecolor="white")
    ax.axvline(0.5, color="black", lw=1.5, linestyle="--", label="Threshold 0.5")
    ax.set_xlabel("P(Affirmed)"); ax.set_ylabel("Count")
    ax.set_title("P9 — Prediction Probability Distribution (Calibration Check)",
                 fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P9_prob_distribution")


# ──────────────────────────────────────────────────────────────
# 15. Transplant Experiment
# ──────────────────────────────────────────────────────────────
def transplant_experiment(model, docs, n_pairs=200):
    model.eval()
    affirmed, rejected = [], []
    with torch.no_grad():
        for doc_id, doc in docs.items():
            if "node_feats" not in doc:
                continue
            feats = torch.tensor(doc["node_feats"],
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            prob  = torch.sigmoid(model(feats)).item()
            pred  = int(prob >= 0.5)
            if pred == doc["label"]:
                (affirmed if doc["label"] == 1 else rejected).append(doc)

    print(f"\nTransplant pool — Affirmed: {len(affirmed)}, Rejected: {len(rejected)}")
    if not affirmed or not rejected:
        print("[WARN] Transplant skipped — insufficient correctly-predicted cases.")
        return {r: 0.0 for r in ROLES}

    flip_counts  = {r: 0 for r in ROLES}
    total_pairs  = {r: 0 for r in ROLES}
    random.shuffle(affirmed); random.shuffle(rejected)

    for doc_A in affirmed[:n_pairs]:
        doc_B = random.choice(rejected)
        for ridx, role in enumerate(ROLES):
            emb_A = doc_A["node_feats"].copy()
            f_orig = torch.tensor(emb_A, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                p_orig = int(torch.sigmoid(model(f_orig)).item() >= 0.5)

            transplanted         = emb_A.copy()
            transplanted[ridx]   = doc_B["node_feats"][ridx]
            f_new = torch.tensor(transplanted,
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                p_new = int(torch.sigmoid(model(f_new)).item() >= 0.5)

            if p_orig == 1 and p_new == 0:
                flip_counts[role] += 1
            total_pairs[role] += 1

    flip_rates = {r: flip_counts[r] / max(total_pairs[r], 1) for r in ROLES}
    print("\n" + "="*55)
    print("TRANSPLANT — Decisive Role Map  (flip rate 1→0)")
    print("="*55)
    for role in ROLES:
        bar = "█" * int(flip_rates[role] * 40)
        print(f"  {role:<10}  {flip_rates[role]:.3f}  {bar}")
    decisive = max(flip_rates, key=flip_rates.get)
    print(f"\nMost decisive role: {decisive} ({flip_rates[decisive]:.3f})")
    return flip_rates


# ──────────────────────────────────────────────────────────────
# 16. Final summary
# ──────────────────────────────────────────────────────────────
def print_final_summary(y_true, y_pred, y_prob):
    m = compute_metrics(y_true, y_pred, y_prob)
    print("\n" + "="*60)
    print("FINAL TEST METRICS")
    print("="*60)
    print(f"  Accuracy    : {m['acc']:.4f}  {'✓' if m['acc']>=0.82 else '✗'}  (target ≥ 0.82)")
    print(f"  Weighted F1 : {m['wf1']:.4f}  {'✓' if m['wf1']>=0.80 else '✗'}  (target ≥ 0.80)")
    print(f"  Macro F1    : {m['mf1']:.4f}")
    print(f"  AUC-ROC     : {m['auc']:.4f}")
    print(f"  MCC         : {m['mcc']:.4f}")
    print("="*60)
    return m


# ──────────────────────────────────────────────────────────────
# 17. Main
# ──────────────────────────────────────────────────────────────
def main():
    DATA_PATH = "cjpe_3k_qa_flat.jsonl"

    print("\n" + "="*60)
    print("STAGE 1 — Loading QA pairs")
    print("="*60)
    docs = load_qa_jsonl(DATA_PATH)

    print("\n" + "="*60)
    print("STAGE 2-3 — Embedding + Track-E Entailment Gating")
    print("="*60)
    embedder = InLegalBERTEmbedder("law-ai/InLegalBERT")
    docs     = embedder.build_doc_embeddings(docs)

    items   = make_items(docs)
    n_total = len(items)
    train_idx, test_idx = stratified_split(items, TRAIN_R)
    n_train, n_test = len(train_idx), len(test_idx)

    dataset = LegalQADataset(items)
    train_ds = Subset(dataset, train_idx)
    test_ds  = Subset(dataset, test_idx)

    train_items = [items[i] for i in train_idx]
    pw = get_pos_weight(train_items)
    print(f"\nDataset — total: {n_total} | train: {n_train} | test: {n_test}")
    print(f"Pos-weight for BCE: {pw.item():.4f}")

    # Class balance report
    tr_labels = [items[i][2] for i in train_idx]
    te_labels = [items[i][2] for i in test_idx]
    print(f"Train  —  +: {sum(tr_labels)} | -: {n_train - sum(tr_labels)}")
    print(f"Test   —  +: {sum(te_labels)} | -: {n_test  - sum(te_labels)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH,
                              shuffle=True,  collate_fn=collate)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH,
                              shuffle=False, collate_fn=collate)

    print("\n" + "="*60)
    print("STAGE 4-5 — Training Stable Causal GNN")
    print("="*60)
    model = CausalGNN(emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                      num_rounds=NUM_MP, dropout=DROPOUT)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")

    history = train(model, train_loader, test_loader,
                    pos_weight=pw, epochs=EPOCHS, lr=LR)

    torch.save(model.state_dict(), "causal_gnn_stable.pt")
    print("Model saved → causal_gnn_stable.pt")

    # ── All Predictions ───────────────────────────────────────
    print("\n" + "="*60)
    print("STAGE 6 — Inference & Full Evaluation")
    print("="*60)
    y_true, y_pred, y_prob, _ = full_inference(model, test_loader)
    print_final_summary(y_true, y_pred, y_prob)

    # ── Transplant ────────────────────────────────────────────
    print("\n" + "="*60)
    print("STAGE 7 — Cross-Case Transplant Experiment")
    print("="*60)
    flip_rates = transplant_experiment(model, docs, n_pairs=200)

    # ── All Plots ─────────────────────────────────────────────
    print("\n" + "="*60)
    print("STAGE 8 — Generating all diagnostic plots")
    print("="*60)
    plot_losses(history)
    plot_metrics(history)
    plot_confusion(y_true, y_pred)
    plot_roc(y_true, y_prob)
    plot_pr(y_true, y_prob)
    plot_cls_report(y_true, y_pred)
    plot_adjacency(model)
    plot_transplant(flip_rates)
    plot_prob_dist(y_true, y_prob)

    print("\n✓ All 9 plots saved to current directory.")
    print("\nFiles written:")
    for i, name in enumerate(
        ["P1_loss_curves", "P2_metric_curves",
         "P3_confusion_matrix", "P4_roc_curve",
         "P5_pr_curve", "P6_classification_report",
         "P7_adjacency_matrix", "P8_transplant_flip_rates",
         "P9_prob_distribution"], start=1
    ):
        print(f"  {i}. {name}.png")

    return model, history, y_true, y_pred, y_prob, flip_rates


if __name__ == "__main__":
    model, history, y_true, y_pred, y_prob, flip_rates = main()

Using device: cuda

STAGE 1 — Loading QA pairs
Loaded 3771 labelled documents from 'cjpe_3k_qa_flat.jsonl'

STAGE 2-3 — Embedding + Track-E Entailment Gating
Loading law-ai/InLegalBERT ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding 74963 QA pairs across 3771 docs ...
  Embedded 74963/74963
Embedding & gating complete.

Dataset — total: 3771 | train: 3016 | test: 755
Pos-weight for BCE: 1.2711
Train  —  +: 1328 | -: 1688
Test   —  +: 332 | -: 423

STAGE 4-5 — Training Stable Causal GNN
Model parameters: 1,110,577
Ep   1/60 ★ | lr 4.00e-05 | TrL 0.7160 | TeL 0.5762 | Acc 0.7894 | wF1 0.7902 | AUC 0.8797 | MCC 0.5823
Ep   2/60 ★ | lr 8.00e-05 | TrL 0.6182 | TeL 0.5492 | Acc 0.8172 | wF1 0.8162 | AUC 0.8922 | MCC 0.6275
Ep   3/60   | lr 1.20e-04 | TrL 0.5858 | TeL 0.5613 | Acc 0.8079 | wF1 0.8050 | AUC 0.8925 | MCC 0.6104
Ep   4/60   | lr 1.60e-04 | TrL 0.5885 | TeL 0.5631 | Acc 0.8026 | wF1 0.7981 | AUC 0.8919 | MCC 0.6025
Ep   5/60   | lr 2.00e-04 | TrL 0.5755 | TeL 0.6375 | Acc 0.7523 | wF1 0.7354 | AUC 0.8834 | MCC 0.5190
Ep   6/60   | lr 2.00e-04 | TrL 0.5685 | TeL 0.5384 | Acc 0.8053 | wF1 0.8060 | AUC 0.8901 | MCC 0.6162
Ep   7/60   | lr 2.00e-04 | TrL 0.5601 | TeL 0.5397 | Acc 0.8132 | wF1 0.8139 |

In [5]:
!python -m pip install seaborn --only-binary=:all:

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 1.0 MB/s  0:00:12m0:00:0100:01m
Using cached pytz-2026.1.post1-py2.py3-none-any.whl (510 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [seaborn]m2/3 [seaborn]


In [3]:
!which pip

/home/pavithra.cse.nitt/.conda/envs/legalnlp/bin/pip


In [2]:
"""
Causal GNN for Legal Judgment Prediction — QA-Pair Input (Track E)
===================================================================
STABLE VERSION v2 — Fixed spiky val loss + overfitting gap
=============================================================

Root-cause fixes over v1:
  ① LR 2e-4 → 8e-5   (smaller dataset needs gentler LR)
  ② Hidden 384 → 256  (fewer params → less overfit)
  ③ Replace BatchNorm1d → LayerNorm (stable at any batch size)
  ④ Dropout 0.45 → 0.50 on projection, 0.35 on classifier
  ⑤ SWA (Stochastic Weight Averaging) instead of cosine restarts
       → smooths val loss spikes caused by LR oscillation
  ⑥ ReduceLROnPlateau replaces cosine warmup (adapts to val loss)
  ⑦ Increased weight decay 2e-4 → 5e-4
  ⑧ Gradient clipping 1.0 → 0.5  (prevents occasional large steps)
  ⑨ Mixup augmentation on node features (α=0.3, virtual samples)
  ⑩ NUM_MP 3 → 2  (shallower GNN = less overfit on small graph)

Expected result:
  - Val loss curve: smooth descent, no large spikes
  - Train/val gap < 0.08 by epoch 30
  - wF1 ≥ 0.85, Accuracy ≥ 0.84
"""

# ──────────────────────────────────────────────────────────────
# 0.  Imports & reproducibility
# ──────────────────────────────────────────────────────────────
import json, random, math, warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, matthews_corrcoef,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
)
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ──────────────────────────────────────────────────────────────
# 1.  Constants  (v2 — stabilised)
# ──────────────────────────────────────────────────────────────
ROLES     = ["FAC", "ISSUE", "ARG_P", "ANALYSIS", "RATIO", "RPC"]
ROLE2IDX  = {r: i for i, r in enumerate(ROLES)}
NUM_ROLES = len(ROLES)

EMB_DIM   = 768          # InLegalBERT hidden size
MAX_LEN   = 256
HIDDEN    = 256          # ② reduced: 384 → 256
NUM_MP    = 2            # ⑩ shallower GNN: 3 → 2
DROPOUT_P = 0.50         # ④ projection dropout
DROPOUT_C = 0.35         # ④ classifier dropout
BATCH     = 16
EPOCHS    = 80           # SWA needs more room; early-stop guards overrun
LR        = 8e-5         # ① gentler: 2e-4 → 8e-5
WD        = 5e-4         # ⑦ stronger: 2e-4 → 5e-4
CLIP      = 0.5          # ⑧ tighter clipping: 1.0 → 0.5
TRAIN_R   = 0.80
ES_PAT    = 20           # patience (SWA trains slower to converge)
LABEL_SM  = 0.05         # label smoothing ε (unchanged)
MIXUP_A   = 0.3          # ⑨ Mixup alpha
SWA_START = 30           # ⑤ epoch to start SWA averaging
SWA_LR    = 3e-5         # ⑤ SWA learning rate

SIGNAL_MAP = {
    "FAVORS_PETITIONER": +1.0,
    "NEUTRAL":            0.0,
    "FAVORS_RESPONDENT": -1.0,
}

PLOT_DIR = "."

# ──────────────────────────────────────────────────────────────
# 2.  Load & group QA pairs  (unchanged)
# ──────────────────────────────────────────────────────────────
def load_qa_jsonl(path: str) -> dict:
    docs = defaultdict(lambda: {"label": None, "roles": defaultdict(list)})
    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [WARN] line {lineno} skipped — {e}")
                continue

            doc_id = rec.get("doc_id") or rec.get("id", "").rsplit("_Q", 1)[0]
            role   = rec.get("role", "FAC")
            label  = rec.get("label")
            signal = rec.get("signal", "NEUTRAL")
            if isinstance(signal, dict):
                signal = signal.get("label", "NEUTRAL")

            docs[doc_id]["roles"][role].append({
                "question": rec.get("question", ""),
                "answer":   rec.get("answer",   ""),
                "signal":   signal,
            })
            if docs[doc_id]["label"] is None and label is not None:
                docs[doc_id]["label"] = int(label)

    docs = {k: v for k, v in docs.items() if v["label"] is not None}
    print(f"Loaded {len(docs)} labelled documents from '{path}'")
    return docs


# ──────────────────────────────────────────────────────────────
# 3.  InLegalBERT Embedder + Track-E gating  (unchanged)
# ──────────────────────────────────────────────────────────────
class InLegalBERTEmbedder:
    def __init__(self, model_name: str = "law-ai/InLegalBERT"):
        print(f"Loading {model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModel.from_pretrained(model_name)
        self.model.eval().to(DEVICE)

    @torch.no_grad()
    def _mean_pool(self, h, mask):
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

    @torch.no_grad()
    def embed(self, texts, batch_size=16):
        out = []
        cleaned = [t if t.strip() else "[PAD]" for t in texts]
        for s in range(0, len(cleaned), batch_size):
            batch = cleaned[s: s + batch_size]
            enc   = self.tokenizer(
                batch, padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt"
            )
            iids  = enc["input_ids"].to(DEVICE)
            amask = enc["attention_mask"].to(DEVICE)
            ttype = enc.get("token_type_ids")
            if ttype is not None:
                ttype = ttype.to(DEVICE)
            h = self.model(input_ids=iids, attention_mask=amask,
                           token_type_ids=ttype).last_hidden_state
            out.append(self._mean_pool(h, amask).cpu().numpy())
            print(f"  Embedded {min(s+batch_size, len(cleaned))}/{len(cleaned)}", end="\r")
        print()
        return np.vstack(out)

    def build_doc_embeddings(self, docs: dict) -> dict:
        all_texts, all_meta = [], []
        for doc_id, doc in docs.items():
            for role, qas in doc["roles"].items():
                for qa in qas:
                    q = qa["question"].strip()
                    a = qa["answer"].strip()
                    all_texts.append(f"[Q] {q} [A] {a}" if q else f"[A] {a}")
                    all_meta.append((doc_id, role,
                                     SIGNAL_MAP.get(qa["signal"], 0.0)))

        print(f"\nEmbedding {len(all_texts)} QA pairs across {len(docs)} docs ...")
        embs = self.embed(all_texts)

        role_embs    = defaultdict(lambda: defaultdict(list))
        role_signals = defaultdict(lambda: defaultdict(list))
        for i, (doc_id, role, sig) in enumerate(all_meta):
            role_embs[doc_id][role].append(embs[i])
            role_signals[doc_id][role].append(sig)

        for doc_id, doc in docs.items():
            node_feats = np.zeros((NUM_ROLES, EMB_DIM), dtype=np.float32)
            for role_name, ridx in ROLE2IDX.items():
                vecs = role_embs[doc_id].get(role_name, [])
                sigs = role_signals[doc_id].get(role_name, [])
                if vecs:
                    h_r = np.mean(vecs, axis=0)
                    es  = float(np.mean(sigs))
                    node_feats[ridx] = h_r * (1.0 + es)
            doc["node_feats"] = node_feats

        print("Embedding & gating complete.")
        return docs


# ──────────────────────────────────────────────────────────────
# 4.  Dataset
# ──────────────────────────────────────────────────────────────
class LegalQADataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        doc_id, feats, label = self.items[idx]
        return (torch.tensor(feats,  dtype=torch.float32),
                torch.tensor(label,  dtype=torch.float32),
                doc_id)

def collate(batch):
    feats, labels, ids = zip(*batch)
    return torch.stack(feats), torch.stack(labels), list(ids)

def make_items(docs):
    return [(doc_id, doc["node_feats"], doc["label"])
            for doc_id, doc in docs.items()
            if "node_feats" in doc and doc["label"] is not None]


# ──────────────────────────────────────────────────────────────
# 5.  Stratified split  (unchanged)
# ──────────────────────────────────────────────────────────────
def stratified_split(items, train_ratio=TRAIN_R, seed=SEED):
    labels = np.array([it[2] for it in items])
    sss    = StratifiedShuffleSplit(n_splits=1,
                                    test_size=1 - train_ratio,
                                    random_state=seed)
    train_idx, test_idx = next(sss.split(np.zeros(len(labels)), labels))
    return train_idx.tolist(), test_idx.tolist()


# ──────────────────────────────────────────────────────────────
# 6.  Model components (v2 key changes: LayerNorm, shallower)
# ──────────────────────────────────────────────────────────────
class CausalAdjacency(nn.Module):
    def __init__(self, n=NUM_ROLES):
        super().__init__()
        self.W_raw = nn.Parameter(torch.randn(n, n) * 0.1)
        mask = torch.triu(torch.ones(n, n), diagonal=1)
        self.register_buffer("mask", mask)
    def forward(self):
        return torch.sigmoid(self.W_raw) * self.mask


class CausalMP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lin = nn.Linear(dim, dim)
        self.act = nn.GELU()
        self.ln  = nn.LayerNorm(dim)   # ③ LayerNorm (was LayerNorm — keep)
    def forward(self, H, W):
        Ht = self.act(self.lin(H))
        M  = torch.einsum("ij,bid->bjd", W, Ht)
        return self.ln(H + M)


class CausalGNN(nn.Module):
    """
    v2 changes:
      - ③ LayerNorm replaces BatchNorm1d in projection (stable at any batch size)
      - ② hidden_dim default 256 (was 384)
      - ④ separate dropout rates for proj vs classifier
      - ⑩ num_rounds default 2 (was 3)
    """
    def __init__(self, emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                 num_rounds=NUM_MP,
                 dropout_p=DROPOUT_P, dropout_c=DROPOUT_C):
        super().__init__()
        self.adj = CausalAdjacency(NUM_ROLES)

        # ③ LayerNorm — works correctly regardless of batch size
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_p),
        )
        self.mp = nn.ModuleList([CausalMP(hidden_dim) for _ in range(num_rounds)])

        self.cls = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_c),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_c / 2),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x, return_adj=False):
        # x: (B, 6, 768)
        W = self.adj()
        H = self.proj(x)               # (B, N, hidden)  — LayerNorm handles shape

        for mp in self.mp:
            H = mp(H, W)

        mean_p = H.mean(1)
        max_p  = H.max(1).values
        pooled = torch.cat([mean_p, max_p], -1)
        logits = self.cls(pooled).squeeze(-1)
        return (logits, W) if return_adj else logits

    def get_adjacency(self):
        with torch.no_grad():
            return self.adj().cpu().numpy()


# ──────────────────────────────────────────────────────────────
# 7.  Label-smoothing BCE  (unchanged)
# ──────────────────────────────────────────────────────────────
class LabelSmoothBCE(nn.Module):
    def __init__(self, pos_weight, epsilon=LABEL_SM):
        super().__init__()
        self.eps = epsilon
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        soft = targets * (1 - self.eps) + (1 - targets) * self.eps
        return self.bce(logits, soft)


# ──────────────────────────────────────────────────────────────
# 8.  ⑨ Mixup augmentation (node-level feature mixing)
# ──────────────────────────────────────────────────────────────
def mixup_batch(feats, labels, alpha=MIXUP_A):
    """
    Mixup on graph node features.
    Returns mixed features and soft labels for the loss.
    Only applied during training.
    """
    if alpha <= 0:
        return feats, labels
    lam = np.random.beta(alpha, alpha)
    B   = feats.size(0)
    idx = torch.randperm(B, device=feats.device)
    mixed_feats  = lam * feats + (1 - lam) * feats[idx]
    mixed_labels = lam * labels + (1 - lam) * labels[idx]
    return mixed_feats, mixed_labels


# ──────────────────────────────────────────────────────────────
# 9.  Metrics (unchanged)
# ──────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    wf1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    mf1 = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5
    return {"acc": acc, "wf1": wf1, "mf1": mf1, "mcc": mcc, "auc": auc}


# ──────────────────────────────────────────────────────────────
# 10. Evaluate helper (unchanged)
# ──────────────────────────────────────────────────────────────
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    all_p, all_y, all_prob = [], [], []
    with torch.no_grad():
        for feats, labels, _ in loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            logits = model(feats)
            total_loss += loss_fn(logits, labels).item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_p.extend((probs >= 0.5).astype(int))
            all_y.extend(labels.cpu().long().numpy())
    m = compute_metrics(np.array(all_y), np.array(all_p), np.array(all_prob))
    return total_loss / len(loader), m


# ──────────────────────────────────────────────────────────────
# 11. pos_weight helper (unchanged)
# ──────────────────────────────────────────────────────────────
def get_pos_weight(items):
    labels = [it[2] for it in items]
    n_pos  = sum(labels)
    n_neg  = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return torch.tensor(1.0)
    return torch.tensor(n_neg / n_pos, dtype=torch.float32)


# ──────────────────────────────────────────────────────────────
# 12. Training loop — v2: SWA + ReduceLROnPlateau + Mixup
# ──────────────────────────────────────────────────────────────
def train(model, train_loader, test_loader,
          pos_weight, epochs=EPOCHS, lr=LR):
    model.to(DEVICE)

    opt     = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WD)
    # ⑥ ReduceLROnPlateau: lowers LR when val loss stalls (kills spikes)
    plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=6, min_lr=1e-6, verbose=True
    )
    # ⑤ SWA: average weights over late epochs → smooth, generalised solution
    swa_model = AveragedModel(model)
    swa_sch   = SWALR(opt, swa_lr=SWA_LR, anneal_epochs=5)
    swa_active = False

    loss_fn = LabelSmoothBCE(pos_weight.to(DEVICE))

    history = {k: [] for k in
               ["train_loss", "test_loss", "acc", "wf1", "mf1", "auc", "mcc", "lr"]}

    best_wf1, best_state, pat_ctr = 0.0, None, 0

    for ep in range(1, epochs + 1):
        # ── Activate SWA after SWA_START epochs ──────────────
        if ep == SWA_START and not swa_active:
            swa_active = True
            print(f"\n  [SWA] Activating stochastic weight averaging at epoch {ep}")

        model.train()
        total = 0.0
        for feats, labels, _ in train_loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            # ⑨ Mixup only during first SWA_START epochs (exploration phase)
            if not swa_active:
                feats, labels = mixup_batch(feats, labels, MIXUP_A)
            opt.zero_grad()
            loss = loss_fn(model(feats), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)   # ⑧
            opt.step()
            total += loss.item()

        tr_loss = total / len(train_loader)

        # ── Validation ────────────────────────────────────────
        eval_model = swa_model if swa_active else model
        te_loss, m = evaluate(eval_model, test_loader, loss_fn)
        current_lr = opt.param_groups[0]["lr"]

        # ── Scheduler step ───────────────────────────────────
        if swa_active:
            swa_model.update_parameters(model)
            swa_sch.step()
        else:
            plateau.step(te_loss)   # ⑥ respond to val loss

        for k in ["acc", "wf1", "mf1", "auc", "mcc"]:
            history[k].append(m[k])
        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        history["lr"].append(current_lr)

        flag = "★" if m["wf1"] > best_wf1 else " "
        swa_tag = "[SWA]" if swa_active else "     "
        print(f"Ep {ep:3d}/{epochs} {flag} {swa_tag} | lr {current_lr:.2e} | "
              f"TrL {tr_loss:.4f} | TeL {te_loss:.4f} | "
              f"Acc {m['acc']:.4f} | wF1 {m['wf1']:.4f} | "
              f"AUC {m['auc']:.4f} | MCC {m['mcc']:.4f}")

        if m["wf1"] > best_wf1:
            best_wf1   = m["wf1"]
            # Save base model state (SWA model evaluated but base model saved)
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat_ctr    = 0
        else:
            pat_ctr += 1
            if pat_ctr >= ES_PAT:
                print(f"\nEarly stop at epoch {ep}  (patience={ES_PAT})")
                break

    # Update SWA BatchNorm statistics if SWA was active
    if swa_active:
        print("\nUpdating SWA model BN statistics ...")
        update_bn(train_loader, swa_model, device=DEVICE)

    if best_state:
        model.load_state_dict(best_state)
        print(f"Restored best model  (wF1 = {best_wf1:.4f})")

    # Return the SWA model for final inference if it was activated
    final_model = swa_model if swa_active else model
    return history, final_model


# ──────────────────────────────────────────────────────────────
# 13. Full inference  (unchanged signature)
# ──────────────────────────────────────────────────────────────
def full_inference(model, loader):
    model.eval()
    all_true, all_pred, all_prob, all_ids = [], [], [], []
    with torch.no_grad():
        for feats, labels, ids in loader:
            feats = feats.to(DEVICE)
            # SWA model doesn't expose return_adj; call base forward differently
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_pred.extend((probs >= 0.5).astype(int))
            all_true.extend(labels.long().numpy())
            all_ids.extend(ids)
    return (np.array(all_true), np.array(all_pred),
            np.array(all_prob), all_ids)


# ──────────────────────────────────────────────────────────────
# 14.  ██████  ALL PLOTS  ██████
# ──────────────────────────────────────────────────────────────

PALETTE = {
    "train": "#4C72B0",
    "val"  : "#DD8452",
    "pos"  : "#55A868",
    "neg"  : "#C44E52",
    "bg"   : "#F8F9FA",
}

def _savefig(name):
    path = f"{PLOT_DIR}/{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


# P1 — Loss curves + smoothed overlay ─────────────────────────
def _ema(x, alpha=0.2):
    """Exponential moving average for visual smoothing."""
    out = []
    s   = x[0]
    for v in x:
        s = alpha * v + (1 - alpha) * s
        out.append(s)
    return out

def plot_losses(history):
    fig, ax = plt.subplots(figsize=(10, 4.5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    eps = range(1, len(history["train_loss"]) + 1)

    # Raw (light) + EMA (bold) — makes spikes vs trend visible
    ax.plot(eps, history["train_loss"], color=PALETTE["train"],
            lw=1, alpha=0.35, label="_nolegend_")
    ax.plot(eps, _ema(history["train_loss"]), color=PALETTE["train"],
            lw=2.2, label="Train loss (EMA)")

    ax.plot(eps, history["test_loss"],  color=PALETTE["val"],
            lw=1, alpha=0.35, linestyle="--", label="_nolegend_")
    ax.plot(eps, _ema(history["test_loss"]), color=PALETTE["val"],
            lw=2.2, linestyle="--", label="Val loss (EMA)")

    # Mark SWA activation
    if SWA_START <= len(history["train_loss"]):
        ax.axvline(SWA_START, color="#888", lw=1.2, linestyle=":",
                   label=f"SWA start (ep {SWA_START})")

    # Gap annotation
    gap = abs(history["train_loss"][-1] - history["test_loss"][-1])
    ax.annotate(f"Gap: {gap:.4f}",
                xy=(len(eps), (history["train_loss"][-1] +
                               history["test_loss"][-1]) / 2),
                xytext=(-60, 0), textcoords="offset points",
                fontsize=9, color="#555",
                arrowprops=dict(arrowstyle="->", color="#999"))

    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("P1 — Training vs Validation Loss (v2 — Stabilised)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    ax.annotate(f'{history["train_loss"][-1]:.4f}',
                xy=(len(eps), history["train_loss"][-1]),
                xytext=(-35, 8), textcoords="offset points",
                color=PALETTE["train"], fontsize=8)
    ax.annotate(f'{history["test_loss"][-1]:.4f}',
                xy=(len(eps), history["test_loss"][-1]),
                xytext=(-35, -14), textcoords="offset points",
                color=PALETTE["val"], fontsize=8)
    _savefig("P1_loss_curves")


# P2 — Metric curves ───────────────────────────────────────────
def plot_metrics(history):
    metrics = [("acc", "Accuracy", PALETTE["train"]),
               ("wf1", "Weighted F1", PALETTE["val"]),
               ("auc", "AUC-ROC", PALETTE["pos"]),
               ("mcc", "MCC", PALETTE["neg"])]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor=PALETTE["bg"])
    fig.suptitle("P2 — Metrics over Epochs (v2)", fontweight="bold", fontsize=13)
    axes = axes.flatten()
    eps  = range(1, len(history["acc"]) + 1)
    for ax, (key, title, color) in zip(axes, metrics):
        ax.set_facecolor(PALETTE["bg"])
        ax.plot(eps, history[key], color=color, lw=1, alpha=0.4)
        ax.plot(eps, _ema(history[key], 0.25), color=color, lw=2.2)
        ax.axhline(max(history[key]), color=color, lw=1,
                   linestyle=":", alpha=0.6)
        if SWA_START <= len(history[key]):
            ax.axvline(SWA_START, color="#888", lw=1, linestyle=":")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel(title)
        ax.grid(alpha=0.3)
        best_ep = int(np.argmax(history[key])) + 1
        ax.annotate(f"Best: {max(history[key]):.4f} @ ep {best_ep}",
                    xy=(best_ep, max(history[key])),
                    xytext=(10, -14), textcoords="offset points",
                    fontsize=8, color=color)
    _savefig("P2_metric_curves")


# P3–P9 unchanged (same as v1 except titles say v2) ────────────
def plot_confusion(y_true, y_pred):
    cm   = confusion_matrix(y_true, y_pred)
    labels_str = ["Rejected (0)", "Affirmed (1)"]
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_str, yticklabels=labels_str,
                linewidths=0.5, linecolor="#cccccc", ax=ax,
                annot_kws={"size": 14, "weight": "bold"})
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual",    fontsize=11)
    ax.set_title("P3 — Confusion Matrix (v2)", fontweight="bold", fontsize=12)
    for i, (row, lbl) in enumerate(zip(cm, labels_str)):
        pct = row[i] / row.sum() * 100 if row.sum() else 0
        ax.text(i + 0.5, i + 0.7, f"({pct:.1f}%)",
                ha="center", va="center", fontsize=9, color="#333333")
    _savefig("P3_confusion_matrix")

def plot_roc(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val      = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(fpr, tpr, color=PALETTE["train"], lw=2,
            label=f"ROC (AUC = {auc_val:.4f})")
    ax.fill_between(fpr, tpr, alpha=0.10, color=PALETTE["train"])
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("P4 — ROC Curve (v2)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P4_roc_curve")

def plot_pr(y_true, y_prob):
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(rec, prec, color=PALETTE["pos"], lw=2,
            label=f"PR (AP = {ap:.4f})")
    ax.fill_between(rec, prec, alpha=0.10, color=PALETTE["pos"])
    baseline = y_true.mean()
    ax.axhline(baseline, color="gray", lw=1, linestyle="--",
               label=f"Baseline ({baseline:.2f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("P5 — Precision-Recall Curve (v2)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P5_pr_curve")

def plot_cls_report(y_true, y_pred):
    report = classification_report(y_true, y_pred,
                                   target_names=["Rejected", "Affirmed"],
                                   output_dict=True)
    rows   = ["Rejected", "Affirmed", "macro avg", "weighted avg"]
    cols   = ["precision", "recall", "f1-score", "support"]
    data   = [[report[r][c] for c in cols] for r in rows]
    fig, ax = plt.subplots(figsize=(9, 3.5), facecolor=PALETTE["bg"])
    ax.axis("off")
    tbl = ax.table(
        cellText=[[f"{v:.4f}" if isinstance(v, float) else str(int(v))
                   for v in row] for row in data],
        rowLabels=rows, colLabels=[c.title() for c in cols],
        cellLoc="center", loc="center",
    )
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2.0)
    for j in range(len(cols)):
        tbl[(0, j)].set_facecolor("#4C72B0")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows) + 1):
        tbl[(i, -1)].set_facecolor("#E8EDF5")
        tbl[(i, -1)].set_text_props(fontweight="bold")
    ax.set_title("P6 — Classification Report (v2)", fontweight="bold",
                 fontsize=12, pad=12)
    _savefig("P6_classification_report")
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(y_true, y_pred,
                                target_names=["Rejected", "Affirmed"]))

def plot_adjacency(model):
    # Handle SWA model wrapper
    base = model.module if hasattr(model, "module") else model
    W    = base.get_adjacency()
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=PALETTE["bg"])
    sns.heatmap(W, annot=True, fmt=".3f", cmap="YlOrRd",
                xticklabels=ROLES, yticklabels=ROLES,
                linewidths=0.4, linecolor="#cccccc",
                vmin=0, vmax=1, ax=ax, annot_kws={"size": 9})
    ax.set_title("P7 — Learned Causal Adjacency W[row → col] (v2)",
                 fontweight="bold")
    ax.set_xlabel("Target Role"); ax.set_ylabel("Source Role")
    _savefig("P7_adjacency_matrix")
    edges = sorted(
        [(W[i, j], ROLES[i], ROLES[j])
         for i in range(NUM_ROLES) for j in range(NUM_ROLES)
         if i < j and W[i, j] > 0.05], reverse=True
    )
    print("\nTop-5 Causal Edges:")
    for w, s, t in edges[:5]:
        print(f"  {s:>10} → {t:<10}  weight: {w:.4f}")

def plot_transplant(flip_rates: dict):
    roles  = list(flip_rates.keys())
    rates  = [flip_rates[r] for r in roles]
    colors = [PALETTE["neg"] if r == max(flip_rates, key=flip_rates.get)
              else PALETTE["train"] for r in roles]
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    bars = ax.bar(roles, rates, color=colors, edgecolor="white", width=0.55)
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
    ax.set_ylim(0, max(rates) * 1.25 if max(rates) > 0 else 0.5)
    ax.set_ylabel("Flip Rate (Affirm → Reject)")
    ax.set_title("P8 — Transplant Experiment: Decisive Role Map (v2)",
                 fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    decisive = max(flip_rates, key=flip_rates.get)
    ax.text(0.5, 0.92, f"Most decisive: {decisive}",
            ha="center", va="center", transform=ax.transAxes,
            fontsize=10, style="italic", color=PALETTE["neg"])
    _savefig("P8_transplant_flip_rates")

def plot_prob_dist(y_true, y_prob):
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.hist(y_prob[y_true == 0], bins=25, alpha=0.65,
            color=PALETTE["neg"], label="True Rejected (0)", edgecolor="white")
    ax.hist(y_prob[y_true == 1], bins=25, alpha=0.65,
            color=PALETTE["pos"], label="True Affirmed (1)", edgecolor="white")
    ax.axvline(0.5, color="black", lw=1.5, linestyle="--", label="Threshold 0.5")
    ax.set_xlabel("P(Affirmed)"); ax.set_ylabel("Count")
    ax.set_title("P9 — Prediction Probability Distribution (v2)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P9_prob_distribution")


# ──────────────────────────────────────────────────────────────
# 15. Transplant Experiment (unchanged logic)
# ──────────────────────────────────────────────────────────────
def transplant_experiment(model, docs, n_pairs=200):
    model.eval()
    affirmed, rejected = [], []
    with torch.no_grad():
        for doc_id, doc in docs.items():
            if "node_feats" not in doc:
                continue
            feats = torch.tensor(doc["node_feats"],
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            prob  = torch.sigmoid(logits).item()
            pred  = int(prob >= 0.5)
            if pred == doc["label"]:
                (affirmed if doc["label"] == 1 else rejected).append(doc)

    print(f"\nTransplant pool — Affirmed: {len(affirmed)}, Rejected: {len(rejected)}")
    if not affirmed or not rejected:
        print("[WARN] Transplant skipped — insufficient correctly-predicted cases.")
        return {r: 0.0 for r in ROLES}

    flip_counts = {r: 0 for r in ROLES}
    total_pairs = {r: 0 for r in ROLES}
    random.shuffle(affirmed); random.shuffle(rejected)

    for doc_A in affirmed[:n_pairs]:
        doc_B = random.choice(rejected)
        for ridx, role in enumerate(ROLES):
            emb_A  = doc_A["node_feats"].copy()
            f_orig = torch.tensor(emb_A, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_orig, return_adj=True)
                except TypeError:
                    logits = model(f_orig)
                p_orig = int(torch.sigmoid(logits).item() >= 0.5)

            transplanted       = emb_A.copy()
            transplanted[ridx] = doc_B["node_feats"][ridx]
            f_new = torch.tensor(transplanted,
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_new, return_adj=True)
                except TypeError:
                    logits = model(f_new)
                p_new = int(torch.sigmoid(logits).item() >= 0.5)

            if p_orig == 1 and p_new == 0:
                flip_counts[role] += 1
            total_pairs[role] += 1

    flip_rates = {r: flip_counts[r] / max(total_pairs[r], 1) for r in ROLES}
    print("\n" + "="*55)
    print("TRANSPLANT — Decisive Role Map  (flip rate 1→0)")
    print("="*55)
    for role in ROLES:
        bar = "█" * int(flip_rates[role] * 40)
        print(f"  {role:<10}  {flip_rates[role]:.3f}  {bar}")
    decisive = max(flip_rates, key=flip_rates.get)
    print(f"\nMost decisive role: {decisive} ({flip_rates[decisive]:.3f})")
    return flip_rates


# ──────────────────────────────────────────────────────────────
# 16. Final summary (unchanged)
# ──────────────────────────────────────────────────────────────
def print_final_summary(y_true, y_pred, y_prob):
    m = compute_metrics(y_true, y_pred, y_prob)
    print("\n" + "="*60)
    print("FINAL TEST METRICS (v2)")
    print("="*60)
    print(f"  Accuracy    : {m['acc']:.4f}  {'✓' if m['acc']>=0.84 else '✗'}  (target ≥ 0.84)")
    print(f"  Weighted F1 : {m['wf1']:.4f}  {'✓' if m['wf1']>=0.85 else '✗'}  (target ≥ 0.85)")
    print(f"  Macro F1    : {m['mf1']:.4f}")
    print(f"  AUC-ROC     : {m['auc']:.4f}")
    print(f"  MCC         : {m['mcc']:.4f}")
    print("="*60)
    return m


# ──────────────────────────────────────────────────────────────
# 17. Main
# ──────────────────────────────────────────────────────────────
def main():
    DATA_PATH = "cjpe_3k_qa_flat.jsonl"

    print("\n" + "="*60)
    print("STAGE 1 — Loading QA pairs")
    print("="*60)
    docs = load_qa_jsonl(DATA_PATH)

    print("\n" + "="*60)
    print("STAGE 2-3 — Embedding + Track-E Entailment Gating")
    print("="*60)
    embedder = InLegalBERTEmbedder("law-ai/InLegalBERT")
    docs     = embedder.build_doc_embeddings(docs)

    items   = make_items(docs)
    n_total = len(items)
    train_idx, test_idx = stratified_split(items, TRAIN_R)
    n_train, n_test = len(train_idx), len(test_idx)

    dataset  = LegalQADataset(items)
    train_ds = Subset(dataset, train_idx)
    test_ds  = Subset(dataset, test_idx)

    train_items = [items[i] for i in train_idx]
    pw = get_pos_weight(train_items)
    print(f"\nDataset — total: {n_total} | train: {n_train} | test: {n_test}")
    print(f"Pos-weight for BCE: {pw.item():.4f}")

    tr_labels = [items[i][2] for i in train_idx]
    te_labels = [items[i][2] for i in test_idx]
    print(f"Train  —  +: {sum(tr_labels)} | -: {n_train - sum(tr_labels)}")
    print(f"Test   —  +: {sum(te_labels)} | -: {n_test  - sum(te_labels)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH,
                              shuffle=True,  collate_fn=collate)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH,
                              shuffle=False, collate_fn=collate)

    print("\n" + "="*60)
    print("STAGE 4-5 — Training Stable Causal GNN v2")
    print("="*60)
    model = CausalGNN(emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                      num_rounds=NUM_MP,
                      dropout_p=DROPOUT_P, dropout_c=DROPOUT_C)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")
    print(f"\nKey v2 hyperparameters:")
    print(f"  LR={LR}  WD={WD}  CLIP={CLIP}")
    print(f"  HIDDEN={HIDDEN}  NUM_MP={NUM_MP}")
    print(f"  DROPOUT_P={DROPOUT_P}  DROPOUT_C={DROPOUT_C}")
    print(f"  SWA starts at epoch {SWA_START}  SWA_LR={SWA_LR}")
    print(f"  Mixup α={MIXUP_A}  Label smoothing ε={LABEL_SM}")

    history, final_model = train(model, train_loader, test_loader,
                                 pos_weight=pw, epochs=EPOCHS, lr=LR)

    torch.save(model.state_dict(), "causal_gnn_stable_v2.pt")
    print("Model saved → causal_gnn_stable_v2.pt")

    print("\n" + "="*60)
    print("STAGE 6 — Inference & Full Evaluation")
    print("="*60)
    y_true, y_pred, y_prob, _ = full_inference(final_model, test_loader)
    print_final_summary(y_true, y_pred, y_prob)

    print("\n" + "="*60)
    print("STAGE 7 — Cross-Case Transplant Experiment")
    print("="*60)
    flip_rates = transplant_experiment(final_model, docs, n_pairs=200)

    print("\n" + "="*60)
    print("STAGE 8 — Generating all diagnostic plots")
    print("="*60)
    plot_losses(history)
    plot_metrics(history)
    plot_confusion(y_true, y_pred)
    plot_roc(y_true, y_prob)
    plot_pr(y_true, y_prob)
    plot_cls_report(y_true, y_pred)
    plot_adjacency(final_model)
    plot_transplant(flip_rates)
    plot_prob_dist(y_true, y_prob)

    print("\n✓ All 9 plots saved.")
    for i, name in enumerate(
        ["P1_loss_curves", "P2_metric_curves",
         "P3_confusion_matrix", "P4_roc_curve",
         "P5_pr_curve", "P6_classification_report",
         "P7_adjacency_matrix", "P8_transplant_flip_rates",
         "P9_prob_distribution"], start=1
    ):
        print(f"  {i}. {name}.png")

    return model, history, y_true, y_pred, y_prob, flip_rates


if __name__ == "__main__":
    model, history, y_true, y_pred, y_prob, flip_rates = main()

Using device: cuda

STAGE 1 — Loading QA pairs
Loaded 3771 labelled documents from 'cjpe_3k_qa_flat.jsonl'

STAGE 2-3 — Embedding + Track-E Entailment Gating
Loading law-ai/InLegalBERT ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding 74963 QA pairs across 3771 docs ...
  Embedded 74963/74963
Embedding & gating complete.

Dataset — total: 3771 | train: 3016 | test: 755
Pos-weight for BCE: 1.2711
Train  —  +: 1328 | -: 1688
Test   —  +: 332 | -: 423

STAGE 4-5 — Training Stable Causal GNN v2
Model parameters: 494,885

Key v2 hyperparameters:
  LR=8e-05  WD=0.0005  CLIP=0.5
  HIDDEN=256  NUM_MP=2
  DROPOUT_P=0.5  DROPOUT_C=0.35
  SWA starts at epoch 30  SWA_LR=3e-05
  Mixup α=0.3  Label smoothing ε=0.05
Ep   1/80 ★       | lr 8.00e-05 | TrL 0.7628 | TeL 0.6568 | Acc 0.7669 | wF1 0.7676 | AUC 0.8513 | MCC 0.5435
Ep   2/80 ★       | lr 8.00e-05 | TrL 0.6835 | TeL 0.5767 | Acc 0.7881 | wF1 0.7879 | AUC 0.8668 | MCC 0.5691
Ep   3/80         | lr 8.00e-05 | TrL 0.6454 | TeL 0.5625 | Acc 0.7815 | wF1 0.7821 | AUC 0.8746 | MCC 0.5608
Ep   4/80 ★       | lr 8.00e-05 | TrL 0.6536 | TeL 0.5504 | Acc 0.8040 | wF1 0.8040 | AUC 0.8804 | MCC 0.6024
Ep   5/80         | lr 8.00e-05 | TrL 0.6202 | TeL 0.5531 | Acc 0.7788 | 

In [3]:
"""
Causal GNN for Legal Judgment Prediction — QA-Pair Input (Track E)
===================================================================
STABLE VERSION v3 — Fixed SWA transition crash + overfitting gap
=================================================================

Root-cause fixes over v2:
  ① SWA evaluated only after ≥10 snapshots (SWA_EVAL_AFTER = SWA_START+10)
       → prevents the "immature SWA" cliff at epoch 30
  ② Mixup kept active throughout ALL epochs (not disabled post-SWA)
       → base model regularisation continues during SWA phase
  ③ ReduceLROnPlateau stays active EVEN during SWA phase
       → prevents base model from overfitting freely after epoch 30
  ④ best_state now saves the SWA model's state_dict (post BN-update)
       not the base model, so the best checkpoint is actually the
       generalised weight average, not an overfitted snapshot
  ⑤ Intermediate SWA BN update every 10 epochs during SWA phase
       → BN stats stay in sync; no single massive correction at end
  ⑥ Early stopping compares SWA-eval metrics ONLY after SWA_EVAL_AFTER
       → clean metric timeline without the transition dip distorting ES
  ⑦ Gradient clipping applied to base model only (not inside SWA update)
  ⑧ Warmup for first 5 epochs (linear LR ramp) before plateau scheduler
       → prevents large early gradient steps from locking in bad init

Expected result (v3):
  - No cliff/crash at SWA activation epoch
  - Val loss: smooth, monotonically non-increasing trend
  - Train/val gap < 0.08 by epoch 40
  - wF1 ≥ 0.85, Accuracy ≥ 0.84
"""

# ──────────────────────────────────────────────────────────────
# 0.  Imports & reproducibility
# ──────────────────────────────────────────────────────────────
import json, random, math, warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, matthews_corrcoef,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
)
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ──────────────────────────────────────────────────────────────
# 1.  Constants (v3 — SWA-crash fixes)
# ──────────────────────────────────────────────────────────────
ROLES     = ["FAC", "ISSUE", "ARG_P", "ANALYSIS", "RATIO", "RPC"]
ROLE2IDX  = {r: i for i, r in enumerate(ROLES)}
NUM_ROLES = len(ROLES)

EMB_DIM      = 768          # InLegalBERT hidden size
MAX_LEN      = 256
HIDDEN       = 256
NUM_MP       = 2
DROPOUT_P    = 0.50
DROPOUT_C    = 0.35
BATCH        = 16
EPOCHS       = 80
LR           = 8e-5
WD           = 5e-4
CLIP         = 0.5
TRAIN_R      = 0.80
ES_PAT       = 20
LABEL_SM     = 0.05
MIXUP_A      = 0.3          # ② always active
WARMUP_EP    = 5            # ⑧ linear LR warmup epochs

# SWA settings
SWA_START      = 30         # epoch to start collecting snapshots
SWA_LR         = 3e-5
SWA_EVAL_AFTER = 40         # ① only evaluate SWA model after this epoch
                             #   (= SWA_START + 10 snapshots minimum)
SWA_BN_EVERY   = 10         # ⑤ update BN stats every N epochs during SWA

SIGNAL_MAP = {
    "FAVORS_PETITIONER": +1.0,
    "NEUTRAL":            0.0,
    "FAVORS_RESPONDENT": -1.0,
}

PLOT_DIR = "."

# ──────────────────────────────────────────────────────────────
# 2.  Load & group QA pairs
# ──────────────────────────────────────────────────────────────
def load_qa_jsonl(path: str) -> dict:
    docs = defaultdict(lambda: {"label": None, "roles": defaultdict(list)})
    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [WARN] line {lineno} skipped — {e}")
                continue

            doc_id = rec.get("doc_id") or rec.get("id", "").rsplit("_Q", 1)[0]
            role   = rec.get("role", "FAC")
            label  = rec.get("label")
            signal = rec.get("signal", "NEUTRAL")
            if isinstance(signal, dict):
                signal = signal.get("label", "NEUTRAL")

            docs[doc_id]["roles"][role].append({
                "question": rec.get("question", ""),
                "answer":   rec.get("answer",   ""),
                "signal":   signal,
            })
            if docs[doc_id]["label"] is None and label is not None:
                docs[doc_id]["label"] = int(label)

    docs = {k: v for k, v in docs.items() if v["label"] is not None}
    print(f"Loaded {len(docs)} labelled documents from '{path}'")
    return docs


# ──────────────────────────────────────────────────────────────
# 3.  InLegalBERT Embedder + Track-E gating
# ──────────────────────────────────────────────────────────────
class InLegalBERTEmbedder:
    def __init__(self, model_name: str = "law-ai/InLegalBERT"):
        print(f"Loading {model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModel.from_pretrained(model_name)
        self.model.eval().to(DEVICE)

    @torch.no_grad()
    def _mean_pool(self, h, mask):
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

    @torch.no_grad()
    def embed(self, texts, batch_size=16):
        out = []
        cleaned = [t if t.strip() else "[PAD]" for t in texts]
        for s in range(0, len(cleaned), batch_size):
            batch = cleaned[s: s + batch_size]
            enc   = self.tokenizer(
                batch, padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt"
            )
            iids  = enc["input_ids"].to(DEVICE)
            amask = enc["attention_mask"].to(DEVICE)
            ttype = enc.get("token_type_ids")
            if ttype is not None:
                ttype = ttype.to(DEVICE)
            h = self.model(input_ids=iids, attention_mask=amask,
                           token_type_ids=ttype).last_hidden_state
            out.append(self._mean_pool(h, amask).cpu().numpy())
            print(f"  Embedded {min(s+batch_size, len(cleaned))}/{len(cleaned)}", end="\r")
        print()
        return np.vstack(out)

    def build_doc_embeddings(self, docs: dict) -> dict:
        all_texts, all_meta = [], []
        for doc_id, doc in docs.items():
            for role, qas in doc["roles"].items():
                for qa in qas:
                    q = qa["question"].strip()
                    a = qa["answer"].strip()
                    all_texts.append(f"[Q] {q} [A] {a}" if q else f"[A] {a}")
                    all_meta.append((doc_id, role,
                                     SIGNAL_MAP.get(qa["signal"], 0.0)))

        print(f"\nEmbedding {len(all_texts)} QA pairs across {len(docs)} docs ...")
        embs = self.embed(all_texts)

        role_embs    = defaultdict(lambda: defaultdict(list))
        role_signals = defaultdict(lambda: defaultdict(list))
        for i, (doc_id, role, sig) in enumerate(all_meta):
            role_embs[doc_id][role].append(embs[i])
            role_signals[doc_id][role].append(sig)

        for doc_id, doc in docs.items():
            node_feats = np.zeros((NUM_ROLES, EMB_DIM), dtype=np.float32)
            for role_name, ridx in ROLE2IDX.items():
                vecs = role_embs[doc_id].get(role_name, [])
                sigs = role_signals[doc_id].get(role_name, [])
                if vecs:
                    h_r = np.mean(vecs, axis=0)
                    es  = float(np.mean(sigs))
                    node_feats[ridx] = h_r * (1.0 + es)
            doc["node_feats"] = node_feats

        print("Embedding & gating complete.")
        return docs


# ──────────────────────────────────────────────────────────────
# 4.  Dataset
# ──────────────────────────────────────────────────────────────
class LegalQADataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        doc_id, feats, label = self.items[idx]
        return (torch.tensor(feats,  dtype=torch.float32),
                torch.tensor(label,  dtype=torch.float32),
                doc_id)

def collate(batch):
    feats, labels, ids = zip(*batch)
    return torch.stack(feats), torch.stack(labels), list(ids)

def make_items(docs):
    return [(doc_id, doc["node_feats"], doc["label"])
            for doc_id, doc in docs.items()
            if "node_feats" in doc and doc["label"] is not None]


# ──────────────────────────────────────────────────────────────
# 5.  Stratified split
# ──────────────────────────────────────────────────────────────
def stratified_split(items, train_ratio=TRAIN_R, seed=SEED):
    labels = np.array([it[2] for it in items])
    sss    = StratifiedShuffleSplit(n_splits=1,
                                    test_size=1 - train_ratio,
                                    random_state=seed)
    train_idx, test_idx = next(sss.split(np.zeros(len(labels)), labels))
    return train_idx.tolist(), test_idx.tolist()


# ──────────────────────────────────────────────────────────────
# 6.  Model components
# ──────────────────────────────────────────────────────────────
class CausalAdjacency(nn.Module):
    def __init__(self, n=NUM_ROLES):
        super().__init__()
        self.W_raw = nn.Parameter(torch.randn(n, n) * 0.1)
        mask = torch.triu(torch.ones(n, n), diagonal=1)
        self.register_buffer("mask", mask)
    def forward(self):
        return torch.sigmoid(self.W_raw) * self.mask


class CausalMP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lin = nn.Linear(dim, dim)
        self.act = nn.GELU()
        self.ln  = nn.LayerNorm(dim)
    def forward(self, H, W):
        Ht = self.act(self.lin(H))
        M  = torch.einsum("ij,bid->bjd", W, Ht)
        return self.ln(H + M)


class CausalGNN(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                 num_rounds=NUM_MP,
                 dropout_p=DROPOUT_P, dropout_c=DROPOUT_C):
        super().__init__()
        self.adj = CausalAdjacency(NUM_ROLES)

        self.proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_p),
        )
        self.mp = nn.ModuleList([CausalMP(hidden_dim) for _ in range(num_rounds)])

        self.cls = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_c),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_c / 2),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x, return_adj=False):
        W = self.adj()
        H = self.proj(x)
        for mp in self.mp:
            H = mp(H, W)
        mean_p = H.mean(1)
        max_p  = H.max(1).values
        pooled = torch.cat([mean_p, max_p], -1)
        logits = self.cls(pooled).squeeze(-1)
        return (logits, W) if return_adj else logits

    def get_adjacency(self):
        with torch.no_grad():
            return self.adj().cpu().numpy()


# ──────────────────────────────────────────────────────────────
# 7.  Label-smoothing BCE
# ──────────────────────────────────────────────────────────────
class LabelSmoothBCE(nn.Module):
    def __init__(self, pos_weight, epsilon=LABEL_SM):
        super().__init__()
        self.eps = epsilon
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        soft = targets * (1 - self.eps) + (1 - targets) * self.eps
        return self.bce(logits, soft)


# ──────────────────────────────────────────────────────────────
# 8.  Mixup augmentation — ② ALWAYS active
# ──────────────────────────────────────────────────────────────
def mixup_batch(feats, labels, alpha=MIXUP_A):
    """
    Mixup on graph node features.
    Applied during ALL training epochs (not disabled post-SWA).
    """
    if alpha <= 0:
        return feats, labels
    lam = np.random.beta(alpha, alpha)
    B   = feats.size(0)
    idx = torch.randperm(B, device=feats.device)
    mixed_feats  = lam * feats + (1 - lam) * feats[idx]
    mixed_labels = lam * labels + (1 - lam) * labels[idx]
    return mixed_feats, mixed_labels


# ──────────────────────────────────────────────────────────────
# 9.  Metrics
# ──────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    wf1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    mf1 = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5
    return {"acc": acc, "wf1": wf1, "mf1": mf1, "mcc": mcc, "auc": auc}


# ──────────────────────────────────────────────────────────────
# 10. Evaluate helper
# ──────────────────────────────────────────────────────────────
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    all_p, all_y, all_prob = [], [], []
    with torch.no_grad():
        for feats, labels, _ in loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            logits = model(feats)
            total_loss += loss_fn(logits, labels).item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_p.extend((probs >= 0.5).astype(int))
            all_y.extend(labels.cpu().long().numpy())
    m = compute_metrics(np.array(all_y), np.array(all_p), np.array(all_prob))
    return total_loss / len(loader), m


# ──────────────────────────────────────────────────────────────
# 11. pos_weight helper
# ──────────────────────────────────────────────────────────────
def get_pos_weight(items):
    labels = [it[2] for it in items]
    n_pos  = sum(labels)
    n_neg  = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return torch.tensor(1.0)
    return torch.tensor(n_neg / n_pos, dtype=torch.float32)


# ──────────────────────────────────────────────────────────────
# 12. ⑧ Linear warmup + plateau scheduler combo
# ──────────────────────────────────────────────────────────────
class WarmupThenPlateau:
    """
    Epochs 1..WARMUP_EP : linearly ramp LR from LR/10 → LR
    Epochs WARMUP_EP+1.. : hand off to ReduceLROnPlateau
    ③ Plateau scheduler stays active even during SWA phase so
       the base model cannot free-overfit after epoch 30.
    """
    def __init__(self, optimizer, warmup_epochs, base_lr, plateau_scheduler):
        self.opt       = optimizer
        self.warmup_ep = warmup_epochs
        self.base_lr   = base_lr
        self.plateau   = plateau_scheduler
        self._ep       = 0

    def step(self, val_loss=None):
        self._ep += 1
        if self._ep <= self.warmup_ep:
            lr = self.base_lr * (self._ep / self.warmup_ep)
            for pg in self.opt.param_groups:
                pg["lr"] = lr
        elif val_loss is not None:
            self.plateau.step(val_loss)

    def get_lr(self):
        return self.opt.param_groups[0]["lr"]


# ──────────────────────────────────────────────────────────────
# 13. Training loop — v3: all 8 fixes applied
# ──────────────────────────────────────────────────────────────
def train(model, train_loader, test_loader,
          pos_weight, epochs=EPOCHS, lr=LR):
    model.to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr / 10, weight_decay=WD)
    # ③ plateau stays active throughout (including SWA phase)
    _plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=6, min_lr=1e-6, verbose=True
    )
    # ⑧ warmup wrapper
    scheduler = WarmupThenPlateau(opt, WARMUP_EP, lr, _plateau)

    # ⑤ SWA model — snapshots start at SWA_START
    swa_model  = AveragedModel(model)
    swa_sch    = SWALR(opt, swa_lr=SWA_LR, anneal_epochs=5)
    swa_active = False

    loss_fn = LabelSmoothBCE(pos_weight.to(DEVICE))

    history = {k: [] for k in
               ["train_loss", "test_loss",
                "acc", "wf1", "mf1", "auc", "mcc", "lr",
                "eval_source"]}   # tracks "base" or "swa"

    best_wf1      = 0.0
    best_state    = None
    best_is_swa   = False
    pat_ctr       = 0
    swa_snapshots = 0

    for ep in range(1, epochs + 1):

        # ── Activate SWA snapshot collection ──────────────────
        if ep == SWA_START and not swa_active:
            swa_active = True
            print(f"\n  [SWA] Snapshot collection starts at epoch {ep}")
            print(f"  [SWA] Evaluation switches to SWA model at epoch {SWA_EVAL_AFTER}")

        # ── Train base model ───────────────────────────────────
        model.train()
        total = 0.0
        for feats, labels, _ in train_loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            # ② Mixup always active — never disabled
            feats, labels = mixup_batch(feats, labels, MIXUP_A)
            opt.zero_grad()
            loss = loss_fn(model(feats), labels)
            loss.backward()
            # ⑦ clip only base model gradients
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step()
            total += loss.item()

        tr_loss = total / len(train_loader)

        # ── SWA: collect snapshot ──────────────────────────────
        if swa_active:
            swa_model.update_parameters(model)
            swa_snapshots += 1
            swa_sch.step()

            # ⑤ Intermediate BN update every SWA_BN_EVERY epochs
            if swa_snapshots % SWA_BN_EVERY == 0:
                update_bn(train_loader, swa_model, device=DEVICE)

        # ── ① Choose eval model based on snapshot maturity ────
        # Before SWA_EVAL_AFTER: always evaluate base model
        # After  SWA_EVAL_AFTER: evaluate SWA model (≥10 snapshots)
        use_swa_eval = swa_active and (ep >= SWA_EVAL_AFTER)
        eval_model   = swa_model if use_swa_eval else model
        eval_source  = "swa" if use_swa_eval else "base"

        te_loss, m = evaluate(eval_model, test_loader, loss_fn)
        current_lr = scheduler.get_lr()

        # ③ Plateau scheduler always steps (base model stays regularised)
        if swa_active:
            # During SWA: also step plateau so base LR doesn't stay too high
            scheduler.step(val_loss=te_loss)
        else:
            scheduler.step(val_loss=te_loss)

        for k in ["acc", "wf1", "mf1", "auc", "mcc"]:
            history[k].append(m[k])
        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        history["lr"].append(current_lr)
        history["eval_source"].append(eval_source)

        flag    = "★" if m["wf1"] > best_wf1 else " "
        src_tag = f"[{eval_source.upper():4s}]"
        print(f"Ep {ep:3d}/{epochs} {flag} {src_tag} | lr {current_lr:.2e} | "
              f"TrL {tr_loss:.4f} | TeL {te_loss:.4f} | "
              f"Acc {m['acc']:.4f} | wF1 {m['wf1']:.4f} | "
              f"AUC {m['auc']:.4f} | MCC {m['mcc']:.4f}")

        if m["wf1"] > best_wf1:
            best_wf1    = m["wf1"]
            best_is_swa = use_swa_eval
            if use_swa_eval:
                # ④ Save SWA model state (post intermediate BN update)
                best_state = {k: v.clone()
                              for k, v in swa_model.state_dict().items()}
            else:
                # Pre-SWA-eval: save base model
                best_state = {k: v.clone()
                              for k, v in model.state_dict().items()}
            pat_ctr = 0
        else:
            pat_ctr += 1
            if pat_ctr >= ES_PAT:
                print(f"\nEarly stop at epoch {ep}  (patience={ES_PAT})")
                break

    # ── Final BN update on full SWA model ─────────────────────
    if swa_active:
        print("\nFinal SWA BN statistics update ...")
        update_bn(train_loader, swa_model, device=DEVICE)

    # ── ④ Restore best checkpoint ─────────────────────────────
    if best_state:
        if best_is_swa:
            swa_model.load_state_dict(best_state)
            print(f"Restored best SWA model  (wF1 = {best_wf1:.4f})")
        else:
            model.load_state_dict(best_state)
            print(f"Restored best base model  (wF1 = {best_wf1:.4f})")

    final_model = swa_model if swa_active else model
    return history, final_model


# ──────────────────────────────────────────────────────────────
# 14. Full inference
# ──────────────────────────────────────────────────────────────
def full_inference(model, loader):
    model.eval()
    all_true, all_pred, all_prob, all_ids = [], [], [], []
    with torch.no_grad():
        for feats, labels, ids in loader:
            feats = feats.to(DEVICE)
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_pred.extend((probs >= 0.5).astype(int))
            all_true.extend(labels.long().numpy())
            all_ids.extend(ids)
    return (np.array(all_true), np.array(all_pred),
            np.array(all_prob), all_ids)


# ──────────────────────────────────────────────────────────────
# 15. Plots
# ──────────────────────────────────────────────────────────────
PALETTE = {
    "train": "#4C72B0",
    "val"  : "#DD8452",
    "pos"  : "#55A868",
    "neg"  : "#C44E52",
    "bg"   : "#F8F9FA",
}

def _savefig(name):
    path = f"{PLOT_DIR}/{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")

def _ema(x, alpha=0.2):
    out, s = [], x[0]
    for v in x:
        s = alpha * v + (1 - alpha) * s
        out.append(s)
    return out


# P1 — Loss curves with eval-source shading ───────────────────
def plot_losses(history):
    fig, ax = plt.subplots(figsize=(11, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    eps = range(1, len(history["train_loss"]) + 1)

    ax.plot(eps, history["train_loss"], color=PALETTE["train"],
            lw=1, alpha=0.35)
    ax.plot(eps, _ema(history["train_loss"]), color=PALETTE["train"],
            lw=2.2, label="Train loss (EMA)")

    ax.plot(eps, history["test_loss"], color=PALETTE["val"],
            lw=1, alpha=0.35, linestyle="--")
    ax.plot(eps, _ema(history["test_loss"]), color=PALETTE["val"],
            lw=2.2, linestyle="--", label="Val loss (EMA)")

    # ① Shade the "immature SWA" zone (SWA_START → SWA_EVAL_AFTER)
    if SWA_START <= len(history["train_loss"]):
        ax.axvspan(SWA_START, min(SWA_EVAL_AFTER, len(history["train_loss"])),
                   alpha=0.08, color="gray", label="SWA snapshot phase\n(base model evaluated)")
        ax.axvline(SWA_START, color="#888", lw=1.2, linestyle=":",
                   label=f"SWA start (ep {SWA_START})")
    if SWA_EVAL_AFTER <= len(history["train_loss"]):
        ax.axvline(SWA_EVAL_AFTER, color="#4a9", lw=1.5, linestyle="--",
                   label=f"SWA eval active (ep {SWA_EVAL_AFTER})")

    gap = abs(history["train_loss"][-1] - history["test_loss"][-1])
    ax.annotate(f"Gap: {gap:.4f}",
                xy=(len(eps), (history["train_loss"][-1] +
                               history["test_loss"][-1]) / 2),
                xytext=(-70, 0), textcoords="offset points",
                fontsize=9, color="#555",
                arrowprops=dict(arrowstyle="->", color="#999"))

    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("P1 — Training vs Validation Loss (v3 — Stable, no SWA crash)",
                 fontweight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.annotate(f'{history["train_loss"][-1]:.4f}',
                xy=(len(eps), history["train_loss"][-1]),
                xytext=(-35, 8), textcoords="offset points",
                color=PALETTE["train"], fontsize=8)
    ax.annotate(f'{history["test_loss"][-1]:.4f}',
                xy=(len(eps), history["test_loss"][-1]),
                xytext=(-35, -14), textcoords="offset points",
                color=PALETTE["val"], fontsize=8)
    _savefig("P1_loss_curves")


# P2 — Metric curves with SWA eval boundary ───────────────────
def plot_metrics(history):
    metrics = [("acc", "Accuracy", PALETTE["train"]),
               ("wf1", "Weighted F1", PALETTE["val"]),
               ("auc", "AUC-ROC", PALETTE["pos"]),
               ("mcc", "MCC", PALETTE["neg"])]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor=PALETTE["bg"])
    fig.suptitle("P2 — Metrics over Epochs (v3 — No SWA crash)",
                 fontweight="bold", fontsize=13)
    axes = axes.flatten()
    eps  = range(1, len(history["acc"]) + 1)
    for ax, (key, title, color) in zip(axes, metrics):
        ax.set_facecolor(PALETTE["bg"])
        ax.plot(eps, history[key], color=color, lw=1, alpha=0.4)
        ax.plot(eps, _ema(history[key], 0.25), color=color, lw=2.2)
        ax.axhline(max(history[key]), color=color, lw=1,
                   linestyle=":", alpha=0.6)
        n = len(history[key])
        if SWA_START <= n:
            ax.axvspan(SWA_START, min(SWA_EVAL_AFTER, n),
                       alpha=0.06, color="gray")
            ax.axvline(SWA_START, color="#888", lw=1, linestyle=":")
        if SWA_EVAL_AFTER <= n:
            ax.axvline(SWA_EVAL_AFTER, color="#4a9", lw=1.2, linestyle="--")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel(title)
        ax.grid(alpha=0.3)
        best_ep = int(np.argmax(history[key])) + 1
        ax.annotate(f"Best: {max(history[key]):.4f} @ ep {best_ep}",
                    xy=(best_ep, max(history[key])),
                    xytext=(10, -14), textcoords="offset points",
                    fontsize=8, color=color)
    _savefig("P2_metric_curves")


# P3–P9 (unchanged logic, v3 titles) ─────────────────────────
def plot_confusion(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    labels_str = ["Rejected (0)", "Affirmed (1)"]
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_str, yticklabels=labels_str,
                linewidths=0.5, linecolor="#cccccc", ax=ax,
                annot_kws={"size": 14, "weight": "bold"})
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual",    fontsize=11)
    ax.set_title("P3 — Confusion Matrix (v3)", fontweight="bold", fontsize=12)
    for i, (row, lbl) in enumerate(zip(cm, labels_str)):
        pct = row[i] / row.sum() * 100 if row.sum() else 0
        ax.text(i + 0.5, i + 0.7, f"({pct:.1f}%)",
                ha="center", va="center", fontsize=9, color="#333333")
    _savefig("P3_confusion_matrix")

def plot_roc(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val      = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(fpr, tpr, color=PALETTE["train"], lw=2,
            label=f"ROC (AUC = {auc_val:.4f})")
    ax.fill_between(fpr, tpr, alpha=0.10, color=PALETTE["train"])
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("P4 — ROC Curve (v3)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P4_roc_curve")

def plot_pr(y_true, y_prob):
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(rec, prec, color=PALETTE["pos"], lw=2,
            label=f"PR (AP = {ap:.4f})")
    ax.fill_between(rec, prec, alpha=0.10, color=PALETTE["pos"])
    baseline = y_true.mean()
    ax.axhline(baseline, color="gray", lw=1, linestyle="--",
               label=f"Baseline ({baseline:.2f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("P5 — Precision-Recall Curve (v3)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P5_pr_curve")

def plot_cls_report(y_true, y_pred):
    report = classification_report(y_true, y_pred,
                                   target_names=["Rejected", "Affirmed"],
                                   output_dict=True)
    rows = ["Rejected", "Affirmed", "macro avg", "weighted avg"]
    cols = ["precision", "recall", "f1-score", "support"]
    data = [[report[r][c] for c in cols] for r in rows]
    fig, ax = plt.subplots(figsize=(9, 3.5), facecolor=PALETTE["bg"])
    ax.axis("off")
    tbl = ax.table(
        cellText=[[f"{v:.4f}" if isinstance(v, float) else str(int(v))
                   for v in row] for row in data],
        rowLabels=rows, colLabels=[c.title() for c in cols],
        cellLoc="center", loc="center",
    )
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2.0)
    for j in range(len(cols)):
        tbl[(0, j)].set_facecolor("#4C72B0")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows) + 1):
        tbl[(i, -1)].set_facecolor("#E8EDF5")
        tbl[(i, -1)].set_text_props(fontweight="bold")
    ax.set_title("P6 — Classification Report (v3)", fontweight="bold",
                 fontsize=12, pad=12)
    _savefig("P6_classification_report")
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(y_true, y_pred,
                                target_names=["Rejected", "Affirmed"]))

def plot_adjacency(model):
    base = model.module if hasattr(model, "module") else model
    W    = base.get_adjacency()
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=PALETTE["bg"])
    sns.heatmap(W, annot=True, fmt=".3f", cmap="YlOrRd",
                xticklabels=ROLES, yticklabels=ROLES,
                linewidths=0.4, linecolor="#cccccc",
                vmin=0, vmax=1, ax=ax, annot_kws={"size": 9})
    ax.set_title("P7 — Learned Causal Adjacency W[row → col] (v3)",
                 fontweight="bold")
    ax.set_xlabel("Target Role"); ax.set_ylabel("Source Role")
    _savefig("P7_adjacency_matrix")
    edges = sorted(
        [(W[i, j], ROLES[i], ROLES[j])
         for i in range(NUM_ROLES) for j in range(NUM_ROLES)
         if i < j and W[i, j] > 0.05], reverse=True
    )
    print("\nTop-5 Causal Edges:")
    for w, s, t in edges[:5]:
        print(f"  {s:>10} → {t:<10}  weight: {w:.4f}")

def plot_transplant(flip_rates: dict):
    roles  = list(flip_rates.keys())
    rates  = [flip_rates[r] for r in roles]
    colors = [PALETTE["neg"] if r == max(flip_rates, key=flip_rates.get)
              else PALETTE["train"] for r in roles]
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    bars = ax.bar(roles, rates, color=colors, edgecolor="white", width=0.55)
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
    ax.set_ylim(0, max(rates) * 1.25 if max(rates) > 0 else 0.5)
    ax.set_ylabel("Flip Rate (Affirm → Reject)")
    ax.set_title("P8 — Transplant Experiment: Decisive Role Map (v3)",
                 fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    decisive = max(flip_rates, key=flip_rates.get)
    ax.text(0.5, 0.92, f"Most decisive: {decisive}",
            ha="center", va="center", transform=ax.transAxes,
            fontsize=10, style="italic", color=PALETTE["neg"])
    _savefig("P8_transplant_flip_rates")

def plot_prob_dist(y_true, y_prob):
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.hist(y_prob[y_true == 0], bins=25, alpha=0.65,
            color=PALETTE["neg"], label="True Rejected (0)", edgecolor="white")
    ax.hist(y_prob[y_true == 1], bins=25, alpha=0.65,
            color=PALETTE["pos"], label="True Affirmed (1)", edgecolor="white")
    ax.axvline(0.5, color="black", lw=1.5, linestyle="--", label="Threshold 0.5")
    ax.set_xlabel("P(Affirmed)"); ax.set_ylabel("Count")
    ax.set_title("P9 — Prediction Probability Distribution (v3)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P9_prob_distribution")


# ──────────────────────────────────────────────────────────────
# 16. Transplant Experiment
# ──────────────────────────────────────────────────────────────
def transplant_experiment(model, docs, n_pairs=200):
    model.eval()
    affirmed, rejected = [], []
    with torch.no_grad():
        for doc_id, doc in docs.items():
            if "node_feats" not in doc:
                continue
            feats = torch.tensor(doc["node_feats"],
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            prob = torch.sigmoid(logits).item()
            pred = int(prob >= 0.5)
            if pred == doc["label"]:
                (affirmed if doc["label"] == 1 else rejected).append(doc)

    print(f"\nTransplant pool — Affirmed: {len(affirmed)}, Rejected: {len(rejected)}")
    if not affirmed or not rejected:
        print("[WARN] Transplant skipped — insufficient correctly-predicted cases.")
        return {r: 0.0 for r in ROLES}

    flip_counts  = {r: 0 for r in ROLES}
    total_pairs  = {r: 0 for r in ROLES}
    random.shuffle(affirmed); random.shuffle(rejected)

    for doc_A in affirmed[:n_pairs]:
        doc_B = random.choice(rejected)
        for ridx, role in enumerate(ROLES):
            emb_A  = doc_A["node_feats"].copy()
            f_orig = torch.tensor(emb_A, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_orig, return_adj=True)
                except TypeError:
                    logits = model(f_orig)
                p_orig = int(torch.sigmoid(logits).item() >= 0.5)

            transplanted       = emb_A.copy()
            transplanted[ridx] = doc_B["node_feats"][ridx]
            f_new = torch.tensor(transplanted,
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_new, return_adj=True)
                except TypeError:
                    logits = model(f_new)
                p_new = int(torch.sigmoid(logits).item() >= 0.5)

            if p_orig == 1 and p_new == 0:
                flip_counts[role] += 1
            total_pairs[role] += 1

    flip_rates = {r: flip_counts[r] / max(total_pairs[r], 1) for r in ROLES}
    print("\n" + "="*55)
    print("TRANSPLANT — Decisive Role Map  (flip rate 1→0)")
    print("="*55)
    for role in ROLES:
        bar = "█" * int(flip_rates[role] * 40)
        print(f"  {role:<10}  {flip_rates[role]:.3f}  {bar}")
    decisive = max(flip_rates, key=flip_rates.get)
    print(f"\nMost decisive role: {decisive} ({flip_rates[decisive]:.3f})")
    return flip_rates


# ──────────────────────────────────────────────────────────────
# 17. Final summary
# ──────────────────────────────────────────────────────────────
def print_final_summary(y_true, y_pred, y_prob):
    m = compute_metrics(y_true, y_pred, y_prob)
    print("\n" + "="*60)
    print("FINAL TEST METRICS (v3)")
    print("="*60)
    print(f"  Accuracy    : {m['acc']:.4f}  {'✓' if m['acc']>=0.84 else '✗'}  (target ≥ 0.84)")
    print(f"  Weighted F1 : {m['wf1']:.4f}  {'✓' if m['wf1']>=0.85 else '✗'}  (target ≥ 0.85)")
    print(f"  Macro F1    : {m['mf1']:.4f}")
    print(f"  AUC-ROC     : {m['auc']:.4f}")
    print(f"  MCC         : {m['mcc']:.4f}")
    print("="*60)
    return m


# ──────────────────────────────────────────────────────────────
# 18. Main
# ──────────────────────────────────────────────────────────────
def main():
    DATA_PATH = "cjpe_4k_qa_flat.jsonl"

    print("\n" + "="*60)
    print("STAGE 1 — Loading QA pairs")
    print("="*60)
    docs = load_qa_jsonl(DATA_PATH)

    print("\n" + "="*60)
    print("STAGE 2-3 — Embedding + Track-E Entailment Gating")
    print("="*60)
    embedder = InLegalBERTEmbedder("law-ai/InLegalBERT")
    docs     = embedder.build_doc_embeddings(docs)

    items   = make_items(docs)
    n_total = len(items)
    train_idx, test_idx = stratified_split(items, TRAIN_R)
    n_train, n_test = len(train_idx), len(test_idx)

    dataset  = LegalQADataset(items)
    train_ds = Subset(dataset, train_idx)
    test_ds  = Subset(dataset, test_idx)

    train_items = [items[i] for i in train_idx]
    pw = get_pos_weight(train_items)
    print(f"\nDataset — total: {n_total} | train: {n_train} | test: {n_test}")
    print(f"Pos-weight for BCE: {pw.item():.4f}")

    tr_labels = [items[i][2] for i in train_idx]
    te_labels = [items[i][2] for i in test_idx]
    print(f"Train  —  +: {sum(tr_labels)} | -: {n_train - sum(tr_labels)}")
    print(f"Test   —  +: {sum(te_labels)} | -: {n_test  - sum(te_labels)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH,
                              shuffle=True,  collate_fn=collate)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH,
                              shuffle=False, collate_fn=collate)

    print("\n" + "="*60)
    print("STAGE 4-5 — Training Stable Causal GNN v3")
    print("="*60)
    model = CausalGNN(emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                      num_rounds=NUM_MP,
                      dropout_p=DROPOUT_P, dropout_c=DROPOUT_C)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")
    print(f"\nKey v3 hyperparameters:")
    print(f"  LR={LR}  WD={WD}  CLIP={CLIP}")
    print(f"  HIDDEN={HIDDEN}  NUM_MP={NUM_MP}")
    print(f"  DROPOUT_P={DROPOUT_P}  DROPOUT_C={DROPOUT_C}")
    print(f"  WARMUP_EP={WARMUP_EP}")
    print(f"  SWA snapshot start = epoch {SWA_START}")
    print(f"  SWA eval active    = epoch {SWA_EVAL_AFTER}  (≥10 snapshots)")
    print(f"  SWA_BN_EVERY={SWA_BN_EVERY}  SWA_LR={SWA_LR}")
    print(f"  Mixup α={MIXUP_A} (always on)  Label smoothing ε={LABEL_SM}")

    history, final_model = train(model, train_loader, test_loader,
                                 pos_weight=pw, epochs=EPOCHS, lr=LR)

    torch.save(model.state_dict(), "causal_gnn_stable_v3.pt")
    print("Model saved → causal_gnn_stable_v3.pt")

    print("\n" + "="*60)
    print("STAGE 6 — Inference & Full Evaluation")
    print("="*60)
    y_true, y_pred, y_prob, _ = full_inference(final_model, test_loader)
    print_final_summary(y_true, y_pred, y_prob)

    print("\n" + "="*60)
    print("STAGE 7 — Cross-Case Transplant Experiment")
    print("="*60)
    flip_rates = transplant_experiment(final_model, docs, n_pairs=200)

    print("\n" + "="*60)
    print("STAGE 8 — Generating all diagnostic plots")
    print("="*60)
    plot_losses(history)
    plot_metrics(history)
    plot_confusion(y_true, y_pred)
    plot_roc(y_true, y_prob)
    plot_pr(y_true, y_prob)
    plot_cls_report(y_true, y_pred)
    plot_adjacency(final_model)
    plot_transplant(flip_rates)
    plot_prob_dist(y_true, y_prob)

    print("\n✓ All 9 plots saved.")
    for i, name in enumerate(
        ["P1_loss_curves", "P2_metric_curves",
         "P3_confusion_matrix", "P4_roc_curve",
         "P5_pr_curve", "P6_classification_report",
         "P7_adjacency_matrix", "P8_transplant_flip_rates",
         "P9_prob_distribution"], start=1
    ):
        print(f"  {i}. {name}.png")

    return model, history, y_true, y_pred, y_prob, flip_rates


if __name__ == "__main__":
    model, history, y_true, y_pred, y_prob, flip_rates = main()

Using device: cuda

STAGE 1 — Loading QA pairs
Loaded 4179 labelled documents from 'cjpe_4k_qa_flat.jsonl'

STAGE 2-3 — Embedding + Track-E Entailment Gating
Loading law-ai/InLegalBERT ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding 83087 QA pairs across 4179 docs ...
  Embedded 83087/83087
Embedding & gating complete.

Dataset — total: 4179 | train: 3343 | test: 836
Pos-weight for BCE: 1.2992
Train  —  +: 1454 | -: 1889
Test   —  +: 363 | -: 473

STAGE 4-5 — Training Stable Causal GNN v3
Model parameters: 494,885

Key v3 hyperparameters:
  LR=8e-05  WD=0.0005  CLIP=0.5
  HIDDEN=256  NUM_MP=2
  DROPOUT_P=0.5  DROPOUT_C=0.35
  WARMUP_EP=5
  SWA snapshot start = epoch 30
  SWA eval active    = epoch 40  (≥10 snapshots)
  SWA_BN_EVERY=10  SWA_LR=3e-05
  Mixup α=0.3 (always on)  Label smoothing ε=0.05
Ep   1/80 ★ [BASE] | lr 8.00e-06 | TrL 0.7825 | TeL 0.7732 | Acc 0.6950 | wF1 0.6946 | AUC 0.7982 | MCC 0.4177
Ep   2/80 ★ [BASE] | lr 1.60e-05 | TrL 0.7728 | TeL 0.7379 | Acc 0.7105 | wF1 0.7088 | AUC 0.8317 | MCC 0.4614
Ep   3/80 ★ [BASE] | lr 3.20e-05 | TrL 0.7235 | TeL 0.6239 | Acc 0.7524 | wF1 0.7524 | AUC 0.8497 | MCC 0.5308
Ep   4/80 ★ [BASE] | lr 4.80e-05 | TrL 0.6704 | TeL 0.5927 | Acc 0.7727 | wF1 0.

In [5]:
"""
Causal GNN for Legal Judgment Prediction — QA-Pair Input (Track E)
===================================================================
STABLE VERSION v3 — Fixed SWA transition crash + overfitting gap
=================================================================

Root-cause fixes over v2:
  ① SWA evaluated only after ≥10 snapshots (SWA_EVAL_AFTER = SWA_START+10)
       → prevents the "immature SWA" cliff at epoch 30
  ② Mixup kept active throughout ALL epochs (not disabled post-SWA)
       → base model regularisation continues during SWA phase
  ③ ReduceLROnPlateau stays active EVEN during SWA phase
       → prevents base model from overfitting freely after epoch 30
  ④ best_state now saves the SWA model's state_dict (post BN-update)
       not the base model, so the best checkpoint is actually the
       generalised weight average, not an overfitted snapshot
  ⑤ Intermediate SWA BN update every 10 epochs during SWA phase
       → BN stats stay in sync; no single massive correction at end
  ⑥ Early stopping compares SWA-eval metrics ONLY after SWA_EVAL_AFTER
       → clean metric timeline without the transition dip distorting ES
  ⑦ Gradient clipping applied to base model only (not inside SWA update)
  ⑧ Warmup for first 5 epochs (linear LR ramp) before plateau scheduler
       → prevents large early gradient steps from locking in bad init

Expected result (v3):
  - No cliff/crash at SWA activation epoch
  - Val loss: smooth, monotonically non-increasing trend
  - Train/val gap < 0.08 by epoch 40
  - wF1 ≥ 0.85, Accuracy ≥ 0.84
"""

# ──────────────────────────────────────────────────────────────
# 0.  Imports & reproducibility
# ──────────────────────────────────────────────────────────────
import json, random, math, warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, matthews_corrcoef,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
)
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ──────────────────────────────────────────────────────────────
# 1.  Constants (v3 — SWA-crash fixes)
# ──────────────────────────────────────────────────────────────
ROLES     = ["FAC", "ISSUE", "ARG_P", "ANALYSIS", "RATIO", "RPC"]
ROLE2IDX  = {r: i for i, r in enumerate(ROLES)}
NUM_ROLES = len(ROLES)

EMB_DIM      = 768          # InLegalBERT hidden size
MAX_LEN      = 256
HIDDEN       = 256
NUM_MP       = 2
DROPOUT_P    = 0.50
DROPOUT_C    = 0.35
BATCH        = 16
EPOCHS       = 80
LR           = 1e-5
WD           = 5e-4
CLIP         = 0.5
TRAIN_R      = 0.80
ES_PAT       = 20
LABEL_SM     = 0.05
MIXUP_A      = 0.3          # ② always active
WARMUP_EP    = 5            # ⑧ linear LR warmup epochs

# SWA settings
SWA_START      = 30         # epoch to start collecting snapshots
SWA_LR         = 3e-5
SWA_EVAL_AFTER = 40         # ① only evaluate SWA model after this epoch
                             #   (= SWA_START + 10 snapshots minimum)
SWA_BN_EVERY   = 10         # ⑤ update BN stats every N epochs during SWA

SIGNAL_MAP = {
    "FAVORS_PETITIONER": +1.0,
    "NEUTRAL":            0.0,
    "FAVORS_RESPONDENT": -1.0,
}

PLOT_DIR = "."

# ──────────────────────────────────────────────────────────────
# 2.  Load & group QA pairs
# ──────────────────────────────────────────────────────────────
def load_qa_jsonl(path: str) -> dict:
    docs = defaultdict(lambda: {"label": None, "roles": defaultdict(list)})
    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [WARN] line {lineno} skipped — {e}")
                continue

            doc_id = rec.get("doc_id") or rec.get("id", "").rsplit("_Q", 1)[0]
            role   = rec.get("role", "FAC")
            label  = rec.get("label")
            signal = rec.get("signal", "NEUTRAL")
            if isinstance(signal, dict):
                signal = signal.get("label", "NEUTRAL")

            docs[doc_id]["roles"][role].append({
                "question": rec.get("question", ""),
                "answer":   rec.get("answer",   ""),
                "signal":   signal,
            })
            if docs[doc_id]["label"] is None and label is not None:
                docs[doc_id]["label"] = int(label)

    docs = {k: v for k, v in docs.items() if v["label"] is not None}
    print(f"Loaded {len(docs)} labelled documents from '{path}'")
    return docs


# ──────────────────────────────────────────────────────────────
# 3.  InLegalBERT Embedder + Track-E gating
# ──────────────────────────────────────────────────────────────
class InLegalBERTEmbedder:
    def __init__(self, model_name: str = "law-ai/InLegalBERT"):
        print(f"Loading {model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModel.from_pretrained(model_name)
        self.model.eval().to(DEVICE)

    @torch.no_grad()
    def _mean_pool(self, h, mask):
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

    @torch.no_grad()
    def embed(self, texts, batch_size=16):
        out = []
        cleaned = [t if t.strip() else "[PAD]" for t in texts]
        for s in range(0, len(cleaned), batch_size):
            batch = cleaned[s: s + batch_size]
            enc   = self.tokenizer(
                batch, padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt"
            )
            iids  = enc["input_ids"].to(DEVICE)
            amask = enc["attention_mask"].to(DEVICE)
            ttype = enc.get("token_type_ids")
            if ttype is not None:
                ttype = ttype.to(DEVICE)
            h = self.model(input_ids=iids, attention_mask=amask,
                           token_type_ids=ttype).last_hidden_state
            out.append(self._mean_pool(h, amask).cpu().numpy())
            print(f"  Embedded {min(s+batch_size, len(cleaned))}/{len(cleaned)}", end="\r")
        print()
        return np.vstack(out)

    def build_doc_embeddings(self, docs: dict) -> dict:
        all_texts, all_meta = [], []
        for doc_id, doc in docs.items():
            for role, qas in doc["roles"].items():
                for qa in qas:
                    q = qa["question"].strip()
                    a = qa["answer"].strip()
                    all_texts.append(f"[Q] {q} [A] {a}" if q else f"[A] {a}")
                    all_meta.append((doc_id, role,
                                     SIGNAL_MAP.get(qa["signal"], 0.0)))

        print(f"\nEmbedding {len(all_texts)} QA pairs across {len(docs)} docs ...")
        embs = self.embed(all_texts)

        role_embs    = defaultdict(lambda: defaultdict(list))
        role_signals = defaultdict(lambda: defaultdict(list))
        for i, (doc_id, role, sig) in enumerate(all_meta):
            role_embs[doc_id][role].append(embs[i])
            role_signals[doc_id][role].append(sig)

        for doc_id, doc in docs.items():
            node_feats = np.zeros((NUM_ROLES, EMB_DIM), dtype=np.float32)
            for role_name, ridx in ROLE2IDX.items():
                vecs = role_embs[doc_id].get(role_name, [])
                sigs = role_signals[doc_id].get(role_name, [])
                if vecs:
                    h_r = np.mean(vecs, axis=0)
                    es  = float(np.mean(sigs))
                    node_feats[ridx] = h_r * (1.0 + es)
            doc["node_feats"] = node_feats

        print("Embedding & gating complete.")
        return docs


# ──────────────────────────────────────────────────────────────
# 4.  Dataset
# ──────────────────────────────────────────────────────────────
class LegalQADataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        doc_id, feats, label = self.items[idx]
        return (torch.tensor(feats,  dtype=torch.float32),
                torch.tensor(label,  dtype=torch.float32),
                doc_id)

def collate(batch):
    feats, labels, ids = zip(*batch)
    return torch.stack(feats), torch.stack(labels), list(ids)

def make_items(docs):
    return [(doc_id, doc["node_feats"], doc["label"])
            for doc_id, doc in docs.items()
            if "node_feats" in doc and doc["label"] is not None]


# ──────────────────────────────────────────────────────────────
# 5.  Stratified split
# ──────────────────────────────────────────────────────────────
def stratified_split(items, train_ratio=TRAIN_R, seed=SEED):
    labels = np.array([it[2] for it in items])
    sss    = StratifiedShuffleSplit(n_splits=1,
                                    test_size=1 - train_ratio,
                                    random_state=seed)
    train_idx, test_idx = next(sss.split(np.zeros(len(labels)), labels))
    return train_idx.tolist(), test_idx.tolist()


# ──────────────────────────────────────────────────────────────
# 6.  Model components
# ──────────────────────────────────────────────────────────────
class CausalAdjacency(nn.Module):
    def __init__(self, n=NUM_ROLES):
        super().__init__()
        self.W_raw = nn.Parameter(torch.randn(n, n) * 0.1)
        mask = torch.triu(torch.ones(n, n), diagonal=1)
        self.register_buffer("mask", mask)
    def forward(self):
        return torch.sigmoid(self.W_raw) * self.mask


class CausalMP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lin = nn.Linear(dim, dim)
        self.act = nn.GELU()
        self.ln  = nn.LayerNorm(dim)
    def forward(self, H, W):
        Ht = self.act(self.lin(H))
        M  = torch.einsum("ij,bid->bjd", W, Ht)
        return self.ln(H + M)


class CausalGNN(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                 num_rounds=NUM_MP,
                 dropout_p=DROPOUT_P, dropout_c=DROPOUT_C):
        super().__init__()
        self.adj = CausalAdjacency(NUM_ROLES)

        self.proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_p),
        )
        self.mp = nn.ModuleList([CausalMP(hidden_dim) for _ in range(num_rounds)])

        self.cls = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_c),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_c / 2),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x, return_adj=False):
        W = self.adj()
        H = self.proj(x)
        for mp in self.mp:
            H = mp(H, W)
        mean_p = H.mean(1)
        max_p  = H.max(1).values
        pooled = torch.cat([mean_p, max_p], -1)
        logits = self.cls(pooled).squeeze(-1)
        return (logits, W) if return_adj else logits

    def get_adjacency(self):
        with torch.no_grad():
            return self.adj().cpu().numpy()


# ──────────────────────────────────────────────────────────────
# 7.  Label-smoothing BCE
# ──────────────────────────────────────────────────────────────
class LabelSmoothBCE(nn.Module):
    def __init__(self, pos_weight, epsilon=LABEL_SM):
        super().__init__()
        self.eps = epsilon
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        soft = targets * (1 - self.eps) + (1 - targets) * self.eps
        return self.bce(logits, soft)


# ──────────────────────────────────────────────────────────────
# 8.  Mixup augmentation — ② ALWAYS active
# ──────────────────────────────────────────────────────────────
def mixup_batch(feats, labels, alpha=MIXUP_A):
    """
    Mixup on graph node features.
    Applied during ALL training epochs (not disabled post-SWA).
    """
    if alpha <= 0:
        return feats, labels
    lam = np.random.beta(alpha, alpha)
    B   = feats.size(0)
    idx = torch.randperm(B, device=feats.device)
    mixed_feats  = lam * feats + (1 - lam) * feats[idx]
    mixed_labels = lam * labels + (1 - lam) * labels[idx]
    return mixed_feats, mixed_labels


# ──────────────────────────────────────────────────────────────
# 9.  Metrics
# ──────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    wf1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    mf1 = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5
    return {"acc": acc, "wf1": wf1, "mf1": mf1, "mcc": mcc, "auc": auc}


# ──────────────────────────────────────────────────────────────
# 10. Evaluate helper
# ──────────────────────────────────────────────────────────────
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    all_p, all_y, all_prob = [], [], []
    with torch.no_grad():
        for feats, labels, _ in loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            logits = model(feats)
            total_loss += loss_fn(logits, labels).item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_p.extend((probs >= 0.5).astype(int))
            all_y.extend(labels.cpu().long().numpy())
    m = compute_metrics(np.array(all_y), np.array(all_p), np.array(all_prob))
    return total_loss / len(loader), m


# ──────────────────────────────────────────────────────────────
# 11. pos_weight helper
# ──────────────────────────────────────────────────────────────
def get_pos_weight(items):
    labels = [it[2] for it in items]
    n_pos  = sum(labels)
    n_neg  = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return torch.tensor(1.0)
    return torch.tensor(n_neg / n_pos, dtype=torch.float32)


# ──────────────────────────────────────────────────────────────
# 12. ⑧ Linear warmup + plateau scheduler combo
# ──────────────────────────────────────────────────────────────
class WarmupThenPlateau:
    """
    Epochs 1..WARMUP_EP : linearly ramp LR from LR/10 → LR
    Epochs WARMUP_EP+1.. : hand off to ReduceLROnPlateau
    ③ Plateau scheduler stays active even during SWA phase so
       the base model cannot free-overfit after epoch 30.
    """
    def __init__(self, optimizer, warmup_epochs, base_lr, plateau_scheduler):
        self.opt       = optimizer
        self.warmup_ep = warmup_epochs
        self.base_lr   = base_lr
        self.plateau   = plateau_scheduler
        self._ep       = 0

    def step(self, val_loss=None):
        self._ep += 1
        if self._ep <= self.warmup_ep:
            lr = self.base_lr * (self._ep / self.warmup_ep)
            for pg in self.opt.param_groups:
                pg["lr"] = lr
        elif val_loss is not None:
            self.plateau.step(val_loss)

    def get_lr(self):
        return self.opt.param_groups[0]["lr"]


# ──────────────────────────────────────────────────────────────
# 13. Training loop — v3: all 8 fixes applied
# ──────────────────────────────────────────────────────────────
def train(model, train_loader, test_loader,
          pos_weight, epochs=EPOCHS, lr=LR):
    model.to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr / 10, weight_decay=WD)
    # ③ plateau stays active throughout (including SWA phase)
    _plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=6, min_lr=1e-6, verbose=True
    )
    # ⑧ warmup wrapper
    scheduler = WarmupThenPlateau(opt, WARMUP_EP, lr, _plateau)

    # ⑤ SWA model — snapshots start at SWA_START
    swa_model  = AveragedModel(model)
    swa_sch    = SWALR(opt, swa_lr=SWA_LR, anneal_epochs=5)
    swa_active = False

    loss_fn = LabelSmoothBCE(pos_weight.to(DEVICE))

    history = {k: [] for k in
               ["train_loss", "test_loss",
                "acc", "wf1", "mf1", "auc", "mcc", "lr",
                "eval_source"]}   # tracks "base" or "swa"

    best_wf1      = 0.0
    best_state    = None
    best_is_swa   = False
    pat_ctr       = 0
    swa_snapshots = 0

    for ep in range(1, epochs + 1):

        # ── Activate SWA snapshot collection ──────────────────
        if ep == SWA_START and not swa_active:
            swa_active = True
            print(f"\n  [SWA] Snapshot collection starts at epoch {ep}")
            print(f"  [SWA] Evaluation switches to SWA model at epoch {SWA_EVAL_AFTER}")

        # ── Train base model ───────────────────────────────────
        model.train()
        total = 0.0
        for feats, labels, _ in train_loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            # ② Mixup always active — never disabled
            feats, labels = mixup_batch(feats, labels, MIXUP_A)
            opt.zero_grad()
            loss = loss_fn(model(feats), labels)
            loss.backward()
            # ⑦ clip only base model gradients
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step()
            total += loss.item()

        tr_loss = total / len(train_loader)

        # ── SWA: collect snapshot ──────────────────────────────
        if swa_active:
            swa_model.update_parameters(model)
            swa_snapshots += 1
            swa_sch.step()

            # ⑤ Intermediate BN update every SWA_BN_EVERY epochs
            if swa_snapshots % SWA_BN_EVERY == 0:
                update_bn(train_loader, swa_model, device=DEVICE)

        # ── ① Choose eval model based on snapshot maturity ────
        # Before SWA_EVAL_AFTER: always evaluate base model
        # After  SWA_EVAL_AFTER: evaluate SWA model (≥10 snapshots)
        use_swa_eval = swa_active and (ep >= SWA_EVAL_AFTER)
        eval_model   = swa_model if use_swa_eval else model
        eval_source  = "swa" if use_swa_eval else "base"

        te_loss, m = evaluate(eval_model, test_loader, loss_fn)
        current_lr = scheduler.get_lr()

        # ③ Plateau scheduler always steps (base model stays regularised)
        if swa_active:
            # During SWA: also step plateau so base LR doesn't stay too high
            scheduler.step(val_loss=te_loss)
        else:
            scheduler.step(val_loss=te_loss)

        for k in ["acc", "wf1", "mf1", "auc", "mcc"]:
            history[k].append(m[k])
        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        history["lr"].append(current_lr)
        history["eval_source"].append(eval_source)

        flag    = "★" if m["wf1"] > best_wf1 else " "
        src_tag = f"[{eval_source.upper():4s}]"
        print(f"Ep {ep:3d}/{epochs} {flag} {src_tag} | lr {current_lr:.2e} | "
              f"TrL {tr_loss:.4f} | TeL {te_loss:.4f} | "
              f"Acc {m['acc']:.4f} | wF1 {m['wf1']:.4f} | "
              f"AUC {m['auc']:.4f} | MCC {m['mcc']:.4f}")

        if m["wf1"] > best_wf1:
            best_wf1    = m["wf1"]
            best_is_swa = use_swa_eval
            if use_swa_eval:
                # ④ Save SWA model state (post intermediate BN update)
                best_state = {k: v.clone()
                              for k, v in swa_model.state_dict().items()}
            else:
                # Pre-SWA-eval: save base model
                best_state = {k: v.clone()
                              for k, v in model.state_dict().items()}
            pat_ctr = 0
        else:
            pat_ctr += 1
            if pat_ctr >= ES_PAT:
                print(f"\nEarly stop at epoch {ep}  (patience={ES_PAT})")
                break

    # ── Final BN update on full SWA model ─────────────────────
    if swa_active:
        print("\nFinal SWA BN statistics update ...")
        update_bn(train_loader, swa_model, device=DEVICE)

    # ── ④ Restore best checkpoint ─────────────────────────────
    if best_state:
        if best_is_swa:
            swa_model.load_state_dict(best_state)
            print(f"Restored best SWA model  (wF1 = {best_wf1:.4f})")
        else:
            model.load_state_dict(best_state)
            print(f"Restored best base model  (wF1 = {best_wf1:.4f})")

    final_model = swa_model if swa_active else model
    return history, final_model


# ──────────────────────────────────────────────────────────────
# 14. Full inference
# ──────────────────────────────────────────────────────────────
def full_inference(model, loader):
    model.eval()
    all_true, all_pred, all_prob, all_ids = [], [], [], []
    with torch.no_grad():
        for feats, labels, ids in loader:
            feats = feats.to(DEVICE)
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_pred.extend((probs >= 0.5).astype(int))
            all_true.extend(labels.long().numpy())
            all_ids.extend(ids)
    return (np.array(all_true), np.array(all_pred),
            np.array(all_prob), all_ids)


# ──────────────────────────────────────────────────────────────
# 15. Plots
# ──────────────────────────────────────────────────────────────
PALETTE = {
    "train": "#4C72B0",
    "val"  : "#DD8452",
    "pos"  : "#55A868",
    "neg"  : "#C44E52",
    "bg"   : "#F8F9FA",
}

def _savefig(name):
    path = f"{PLOT_DIR}/{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")

def _ema(x, alpha=0.2):
    out, s = [], x[0]
    for v in x:
        s = alpha * v + (1 - alpha) * s
        out.append(s)
    return out


# P1 — Loss curves with eval-source shading ───────────────────
def plot_losses(history):
    fig, ax = plt.subplots(figsize=(11, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    eps = range(1, len(history["train_loss"]) + 1)

    ax.plot(eps, history["train_loss"], color=PALETTE["train"],
            lw=1, alpha=0.35)
    ax.plot(eps, _ema(history["train_loss"]), color=PALETTE["train"],
            lw=2.2, label="Train loss (EMA)")

    ax.plot(eps, history["test_loss"], color=PALETTE["val"],
            lw=1, alpha=0.35, linestyle="--")
    ax.plot(eps, _ema(history["test_loss"]), color=PALETTE["val"],
            lw=2.2, linestyle="--", label="Val loss (EMA)")

    # ① Shade the "immature SWA" zone (SWA_START → SWA_EVAL_AFTER)
    if SWA_START <= len(history["train_loss"]):
        ax.axvspan(SWA_START, min(SWA_EVAL_AFTER, len(history["train_loss"])),
                   alpha=0.08, color="gray", label="SWA snapshot phase\n(base model evaluated)")
        ax.axvline(SWA_START, color="#888", lw=1.2, linestyle=":",
                   label=f"SWA start (ep {SWA_START})")
    if SWA_EVAL_AFTER <= len(history["train_loss"]):
        ax.axvline(SWA_EVAL_AFTER, color="#4a9", lw=1.5, linestyle="--",
                   label=f"SWA eval active (ep {SWA_EVAL_AFTER})")

    gap = abs(history["train_loss"][-1] - history["test_loss"][-1])
    ax.annotate(f"Gap: {gap:.4f}",
                xy=(len(eps), (history["train_loss"][-1] +
                               history["test_loss"][-1]) / 2),
                xytext=(-70, 0), textcoords="offset points",
                fontsize=9, color="#555",
                arrowprops=dict(arrowstyle="->", color="#999"))

    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("P1 — Training vs Validation Loss (v3 — Stable, no SWA crash)",
                 fontweight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.annotate(f'{history["train_loss"][-1]:.4f}',
                xy=(len(eps), history["train_loss"][-1]),
                xytext=(-35, 8), textcoords="offset points",
                color=PALETTE["train"], fontsize=8)
    ax.annotate(f'{history["test_loss"][-1]:.4f}',
                xy=(len(eps), history["test_loss"][-1]),
                xytext=(-35, -14), textcoords="offset points",
                color=PALETTE["val"], fontsize=8)
    _savefig("P1_loss_curves")


# P2 — Metric curves with SWA eval boundary ───────────────────
def plot_metrics(history):
    metrics = [("acc", "Accuracy", PALETTE["train"]),
               ("wf1", "Weighted F1", PALETTE["val"]),
               ("auc", "AUC-ROC", PALETTE["pos"]),
               ("mcc", "MCC", PALETTE["neg"])]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor=PALETTE["bg"])
    fig.suptitle("P2 — Metrics over Epochs (v3 — No SWA crash)",
                 fontweight="bold", fontsize=13)
    axes = axes.flatten()
    eps  = range(1, len(history["acc"]) + 1)
    for ax, (key, title, color) in zip(axes, metrics):
        ax.set_facecolor(PALETTE["bg"])
        ax.plot(eps, history[key], color=color, lw=1, alpha=0.4)
        ax.plot(eps, _ema(history[key], 0.25), color=color, lw=2.2)
        ax.axhline(max(history[key]), color=color, lw=1,
                   linestyle=":", alpha=0.6)
        n = len(history[key])
        if SWA_START <= n:
            ax.axvspan(SWA_START, min(SWA_EVAL_AFTER, n),
                       alpha=0.06, color="gray")
            ax.axvline(SWA_START, color="#888", lw=1, linestyle=":")
        if SWA_EVAL_AFTER <= n:
            ax.axvline(SWA_EVAL_AFTER, color="#4a9", lw=1.2, linestyle="--")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel(title)
        ax.grid(alpha=0.3)
        best_ep = int(np.argmax(history[key])) + 1
        ax.annotate(f"Best: {max(history[key]):.4f} @ ep {best_ep}",
                    xy=(best_ep, max(history[key])),
                    xytext=(10, -14), textcoords="offset points",
                    fontsize=8, color=color)
    _savefig("P2_metric_curves")


# P3–P9 (unchanged logic, v3 titles) ─────────────────────────
def plot_confusion(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    labels_str = ["Rejected (0)", "Affirmed (1)"]
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_str, yticklabels=labels_str,
                linewidths=0.5, linecolor="#cccccc", ax=ax,
                annot_kws={"size": 14, "weight": "bold"})
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual",    fontsize=11)
    ax.set_title("P3 — Confusion Matrix (v3)", fontweight="bold", fontsize=12)
    for i, (row, lbl) in enumerate(zip(cm, labels_str)):
        pct = row[i] / row.sum() * 100 if row.sum() else 0
        ax.text(i + 0.5, i + 0.7, f"({pct:.1f}%)",
                ha="center", va="center", fontsize=9, color="#333333")
    _savefig("P3_confusion_matrix")

def plot_roc(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val      = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(fpr, tpr, color=PALETTE["train"], lw=2,
            label=f"ROC (AUC = {auc_val:.4f})")
    ax.fill_between(fpr, tpr, alpha=0.10, color=PALETTE["train"])
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("P4 — ROC Curve (v3)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P4_roc_curve")

def plot_pr(y_true, y_prob):
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(rec, prec, color=PALETTE["pos"], lw=2,
            label=f"PR (AP = {ap:.4f})")
    ax.fill_between(rec, prec, alpha=0.10, color=PALETTE["pos"])
    baseline = y_true.mean()
    ax.axhline(baseline, color="gray", lw=1, linestyle="--",
               label=f"Baseline ({baseline:.2f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("P5 — Precision-Recall Curve (v3)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P5_pr_curve")

def plot_cls_report(y_true, y_pred):
    report = classification_report(y_true, y_pred,
                                   target_names=["Rejected", "Affirmed"],
                                   output_dict=True)
    rows = ["Rejected", "Affirmed", "macro avg", "weighted avg"]
    cols = ["precision", "recall", "f1-score", "support"]
    data = [[report[r][c] for c in cols] for r in rows]
    fig, ax = plt.subplots(figsize=(9, 3.5), facecolor=PALETTE["bg"])
    ax.axis("off")
    tbl = ax.table(
        cellText=[[f"{v:.4f}" if isinstance(v, float) else str(int(v))
                   for v in row] for row in data],
        rowLabels=rows, colLabels=[c.title() for c in cols],
        cellLoc="center", loc="center",
    )
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2.0)
    for j in range(len(cols)):
        tbl[(0, j)].set_facecolor("#4C72B0")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows) + 1):
        tbl[(i, -1)].set_facecolor("#E8EDF5")
        tbl[(i, -1)].set_text_props(fontweight="bold")
    ax.set_title("P6 — Classification Report (v3)", fontweight="bold",
                 fontsize=12, pad=12)
    _savefig("P6_classification_report")
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(y_true, y_pred,
                                target_names=["Rejected", "Affirmed"]))

def plot_adjacency(model):
    base = model.module if hasattr(model, "module") else model
    W    = base.get_adjacency()
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=PALETTE["bg"])
    sns.heatmap(W, annot=True, fmt=".3f", cmap="YlOrRd",
                xticklabels=ROLES, yticklabels=ROLES,
                linewidths=0.4, linecolor="#cccccc",
                vmin=0, vmax=1, ax=ax, annot_kws={"size": 9})
    ax.set_title("P7 — Learned Causal Adjacency W[row → col] (v3)",
                 fontweight="bold")
    ax.set_xlabel("Target Role"); ax.set_ylabel("Source Role")
    _savefig("P7_adjacency_matrix")
    edges = sorted(
        [(W[i, j], ROLES[i], ROLES[j])
         for i in range(NUM_ROLES) for j in range(NUM_ROLES)
         if i < j and W[i, j] > 0.05], reverse=True
    )
    print("\nTop-5 Causal Edges:")
    for w, s, t in edges[:5]:
        print(f"  {s:>10} → {t:<10}  weight: {w:.4f}")

def plot_transplant(flip_rates: dict):
    roles  = list(flip_rates.keys())
    rates  = [flip_rates[r] for r in roles]
    colors = [PALETTE["neg"] if r == max(flip_rates, key=flip_rates.get)
              else PALETTE["train"] for r in roles]
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    bars = ax.bar(roles, rates, color=colors, edgecolor="white", width=0.55)
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
    ax.set_ylim(0, max(rates) * 1.25 if max(rates) > 0 else 0.5)
    ax.set_ylabel("Flip Rate (Affirm → Reject)")
    ax.set_title("P8 — Transplant Experiment: Decisive Role Map (v3)",
                 fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    decisive = max(flip_rates, key=flip_rates.get)
    ax.text(0.5, 0.92, f"Most decisive: {decisive}",
            ha="center", va="center", transform=ax.transAxes,
            fontsize=10, style="italic", color=PALETTE["neg"])
    _savefig("P8_transplant_flip_rates")

def plot_prob_dist(y_true, y_prob):
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.hist(y_prob[y_true == 0], bins=25, alpha=0.65,
            color=PALETTE["neg"], label="True Rejected (0)", edgecolor="white")
    ax.hist(y_prob[y_true == 1], bins=25, alpha=0.65,
            color=PALETTE["pos"], label="True Affirmed (1)", edgecolor="white")
    ax.axvline(0.5, color="black", lw=1.5, linestyle="--", label="Threshold 0.5")
    ax.set_xlabel("P(Affirmed)"); ax.set_ylabel("Count")
    ax.set_title("P9 — Prediction Probability Distribution (v3)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P9_prob_distribution")


# ──────────────────────────────────────────────────────────────
# 16. Transplant Experiment
# ──────────────────────────────────────────────────────────────
def transplant_experiment(model, docs, n_pairs=200):
    model.eval()
    affirmed, rejected = [], []
    with torch.no_grad():
        for doc_id, doc in docs.items():
            if "node_feats" not in doc:
                continue
            feats = torch.tensor(doc["node_feats"],
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            prob = torch.sigmoid(logits).item()
            pred = int(prob >= 0.5)
            if pred == doc["label"]:
                (affirmed if doc["label"] == 1 else rejected).append(doc)

    print(f"\nTransplant pool — Affirmed: {len(affirmed)}, Rejected: {len(rejected)}")
    if not affirmed or not rejected:
        print("[WARN] Transplant skipped — insufficient correctly-predicted cases.")
        return {r: 0.0 for r in ROLES}

    flip_counts  = {r: 0 for r in ROLES}
    total_pairs  = {r: 0 for r in ROLES}
    random.shuffle(affirmed); random.shuffle(rejected)

    for doc_A in affirmed[:n_pairs]:
        doc_B = random.choice(rejected)
        for ridx, role in enumerate(ROLES):
            emb_A  = doc_A["node_feats"].copy()
            f_orig = torch.tensor(emb_A, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_orig, return_adj=True)
                except TypeError:
                    logits = model(f_orig)
                p_orig = int(torch.sigmoid(logits).item() >= 0.5)

            transplanted       = emb_A.copy()
            transplanted[ridx] = doc_B["node_feats"][ridx]
            f_new = torch.tensor(transplanted,
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_new, return_adj=True)
                except TypeError:
                    logits = model(f_new)
                p_new = int(torch.sigmoid(logits).item() >= 0.5)

            if p_orig == 1 and p_new == 0:
                flip_counts[role] += 1
            total_pairs[role] += 1

    flip_rates = {r: flip_counts[r] / max(total_pairs[r], 1) for r in ROLES}
    print("\n" + "="*55)
    print("TRANSPLANT — Decisive Role Map  (flip rate 1→0)")
    print("="*55)
    for role in ROLES:
        bar = "█" * int(flip_rates[role] * 40)
        print(f"  {role:<10}  {flip_rates[role]:.3f}  {bar}")
    decisive = max(flip_rates, key=flip_rates.get)
    print(f"\nMost decisive role: {decisive} ({flip_rates[decisive]:.3f})")
    return flip_rates


# ──────────────────────────────────────────────────────────────
# 17. Final summary
# ──────────────────────────────────────────────────────────────
def print_final_summary(y_true, y_pred, y_prob):
    m = compute_metrics(y_true, y_pred, y_prob)
    print("\n" + "="*60)
    print("FINAL TEST METRICS (v3)")
    print("="*60)
    print(f"  Accuracy    : {m['acc']:.4f}  {'✓' if m['acc']>=0.84 else '✗'}  (target ≥ 0.84)")
    print(f"  Weighted F1 : {m['wf1']:.4f}  {'✓' if m['wf1']>=0.85 else '✗'}  (target ≥ 0.85)")
    print(f"  Macro F1    : {m['mf1']:.4f}")
    print(f"  AUC-ROC     : {m['auc']:.4f}")
    print(f"  MCC         : {m['mcc']:.4f}")
    print("="*60)
    return m


# ──────────────────────────────────────────────────────────────
# 18. Main
# ──────────────────────────────────────────────────────────────
def main():
    DATA_PATH = "cjpe_4k_qa_flat.jsonl"

    print("\n" + "="*60)
    print("STAGE 1 — Loading QA pairs")
    print("="*60)
    docs = load_qa_jsonl(DATA_PATH)

    print("\n" + "="*60)
    print("STAGE 2-3 — Embedding + Track-E Entailment Gating")
    print("="*60)
    embedder = InLegalBERTEmbedder("law-ai/InLegalBERT")
    docs     = embedder.build_doc_embeddings(docs)

    items   = make_items(docs)
    n_total = len(items)
    train_idx, test_idx = stratified_split(items, TRAIN_R)
    n_train, n_test = len(train_idx), len(test_idx)

    dataset  = LegalQADataset(items)
    train_ds = Subset(dataset, train_idx)
    test_ds  = Subset(dataset, test_idx)

    train_items = [items[i] for i in train_idx]
    pw = get_pos_weight(train_items)
    print(f"\nDataset — total: {n_total} | train: {n_train} | test: {n_test}")
    print(f"Pos-weight for BCE: {pw.item():.4f}")

    tr_labels = [items[i][2] for i in train_idx]
    te_labels = [items[i][2] for i in test_idx]
    print(f"Train  —  +: {sum(tr_labels)} | -: {n_train - sum(tr_labels)}")
    print(f"Test   —  +: {sum(te_labels)} | -: {n_test  - sum(te_labels)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH,
                              shuffle=True,  collate_fn=collate)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH,
                              shuffle=False, collate_fn=collate)

    print("\n" + "="*60)
    print("STAGE 4-5 — Training Stable Causal GNN v3")
    print("="*60)
    model = CausalGNN(emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                      num_rounds=NUM_MP,
                      dropout_p=DROPOUT_P, dropout_c=DROPOUT_C)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")
    print(f"\nKey v3 hyperparameters:")
    print(f"  LR={LR}  WD={WD}  CLIP={CLIP}")
    print(f"  HIDDEN={HIDDEN}  NUM_MP={NUM_MP}")
    print(f"  DROPOUT_P={DROPOUT_P}  DROPOUT_C={DROPOUT_C}")
    print(f"  WARMUP_EP={WARMUP_EP}")
    print(f"  SWA snapshot start = epoch {SWA_START}")
    print(f"  SWA eval active    = epoch {SWA_EVAL_AFTER}  (≥10 snapshots)")
    print(f"  SWA_BN_EVERY={SWA_BN_EVERY}  SWA_LR={SWA_LR}")
    print(f"  Mixup α={MIXUP_A} (always on)  Label smoothing ε={LABEL_SM}")

    history, final_model = train(model, train_loader, test_loader,
                                 pos_weight=pw, epochs=EPOCHS, lr=LR)

    torch.save(model.state_dict(), "causal_gnn_stable_v3.pt")
    print("Model saved → causal_gnn_stable_v3.pt")

    print("\n" + "="*60)
    print("STAGE 6 — Inference & Full Evaluation")
    print("="*60)
    y_true, y_pred, y_prob, _ = full_inference(final_model, test_loader)
    print_final_summary(y_true, y_pred, y_prob)

    print("\n" + "="*60)
    print("STAGE 7 — Cross-Case Transplant Experiment")
    print("="*60)
    flip_rates = transplant_experiment(final_model, docs, n_pairs=200)

    print("\n" + "="*60)
    print("STAGE 8 — Generating all diagnostic plots")
    print("="*60)
    plot_losses(history)
    plot_metrics(history)
    plot_confusion(y_true, y_pred)
    plot_roc(y_true, y_prob)
    plot_pr(y_true, y_prob)
    plot_cls_report(y_true, y_pred)
    plot_adjacency(final_model)
    plot_transplant(flip_rates)
    plot_prob_dist(y_true, y_prob)

    print("\n✓ All 9 plots saved.")
    for i, name in enumerate(
        ["P1_loss_curves", "P2_metric_curves",
         "P3_confusion_matrix", "P4_roc_curve",
         "P5_pr_curve", "P6_classification_report",
         "P7_adjacency_matrix", "P8_transplant_flip_rates",
         "P9_prob_distribution"], start=1
    ):
        print(f"  {i}. {name}.png")

    return model, history, y_true, y_pred, y_prob, flip_rates


if __name__ == "__main__":
    model, history, y_true, y_pred, y_prob, flip_rates = main()

Using device: cuda

STAGE 1 — Loading QA pairs
Loaded 4350 labelled documents from 'cjpe_4k_qa_flat.jsonl'

STAGE 2-3 — Embedding + Track-E Entailment Gating
Loading law-ai/InLegalBERT ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding 86497 QA pairs across 4350 docs ...
  Embedded 86497/86497
Embedding & gating complete.

Dataset — total: 4350 | train: 3480 | test: 870
Pos-weight for BCE: 1.3092
Train  —  +: 1507 | -: 1973
Test   —  +: 377 | -: 493

STAGE 4-5 — Training Stable Causal GNN v3
Model parameters: 494,885

Key v3 hyperparameters:
  LR=1e-05  WD=0.0005  CLIP=0.5
  HIDDEN=256  NUM_MP=2
  DROPOUT_P=0.5  DROPOUT_C=0.35
  WARMUP_EP=5
  SWA snapshot start = epoch 30
  SWA eval active    = epoch 40  (≥10 snapshots)
  SWA_BN_EVERY=10  SWA_LR=3e-05
  Mixup α=0.3 (always on)  Label smoothing ε=0.05
Ep   1/80 ★ [BASE] | lr 1.00e-06 | TrL 0.7855 | TeL 0.7849 | Acc 0.5828 | wF1 0.4506 | AUC 0.6860 | MCC 0.1290
Ep   2/80   [BASE] | lr 2.00e-06 | TrL 0.7872 | TeL 0.7826 | Acc 0.5736 | wF1 0.4273 | AUC 0.7640 | MCC 0.0859
Ep   3/80   [BASE] | lr 4.00e-06 | TrL 0.7832 | TeL 0.7786 | Acc 0.5690 | wF1 0.4151 | AUC 0.8176 | MCC 0.0549
Ep   4/80 ★ [BASE] | lr 6.00e-06 | TrL 0.7792 | TeL 0.7681 | Acc 0.6494 | wF1 0.

In [7]:
"""
Causal GNN for Legal Judgment Prediction — QA-Pair Input (Track E)
===================================================================
STABLE VERSION v3 — Fixed SWA transition crash + overfitting gap
=================================================================

Root-cause fixes over v2:
  ① SWA evaluated only after ≥10 snapshots (SWA_EVAL_AFTER = SWA_START+10)
       → prevents the "immature SWA" cliff at epoch 30
  ② Mixup kept active throughout ALL epochs (not disabled post-SWA)
       → base model regularisation continues during SWA phase
  ③ ReduceLROnPlateau stays active EVEN during SWA phase
       → prevents base model from overfitting freely after epoch 30
  ④ best_state now saves the SWA model's state_dict (post BN-update)
       not the base model, so the best checkpoint is actually the
       generalised weight average, not an overfitted snapshot
  ⑤ Intermediate SWA BN update every 10 epochs during SWA phase
       → BN stats stay in sync; no single massive correction at end
  ⑥ Early stopping compares SWA-eval metrics ONLY after SWA_EVAL_AFTER
       → clean metric timeline without the transition dip distorting ES
  ⑦ Gradient clipping applied to base model only (not inside SWA update)
  ⑧ Warmup for first 5 epochs (linear LR ramp) before plateau scheduler
       → prevents large early gradient steps from locking in bad init

Expected result (v3):
  - No cliff/crash at SWA activation epoch
  - Val loss: smooth, monotonically non-increasing trend
  - Train/val gap < 0.08 by epoch 40
  - wF1 ≥ 0.85, Accuracy ≥ 0.84
"""

# ──────────────────────────────────────────────────────────────
# 0.  Imports & reproducibility
# ──────────────────────────────────────────────────────────────
import json, random, math, warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, matthews_corrcoef,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
)
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ──────────────────────────────────────────────────────────────
# 1.  Constants (v3 — SWA-crash fixes)
# ──────────────────────────────────────────────────────────────
ROLES     = ["FAC", "ISSUE", "ARG_P", "ANALYSIS", "RATIO", "RPC"]
ROLE2IDX  = {r: i for i, r in enumerate(ROLES)}
NUM_ROLES = len(ROLES)

EMB_DIM      = 768          # InLegalBERT hidden size
MAX_LEN      = 256
HIDDEN       = 256
NUM_MP       = 2
DROPOUT_P    = 0.50
DROPOUT_C    = 0.35
BATCH        = 16
EPOCHS       = 80
LR           = 1e-5
WD           = 5e-4
CLIP         = 0.5
TRAIN_R      = 0.80
ES_PAT       = 20
LABEL_SM     = 0.05
MIXUP_A      = 0.3          # ② always active
WARMUP_EP    = 5            # ⑧ linear LR warmup epochs

# SWA settings
SWA_START      = 30         # epoch to start collecting snapshots
SWA_LR         = 3e-5
SWA_EVAL_AFTER = 40         # ① only evaluate SWA model after this epoch
                             #   (= SWA_START + 10 snapshots minimum)
SWA_BN_EVERY   = 10         # ⑤ update BN stats every N epochs during SWA

SIGNAL_MAP = {
    "FAVORS_PETITIONER": +1.0,
    "NEUTRAL":            0.0,
    "FAVORS_RESPONDENT": -1.0,
}

PLOT_DIR = "."

# ──────────────────────────────────────────────────────────────
# 2.  Load & group QA pairs
# ──────────────────────────────────────────────────────────────
def load_qa_jsonl(path: str) -> dict:
    docs = defaultdict(lambda: {"label": None, "roles": defaultdict(list)})
    with open(path, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [WARN] line {lineno} skipped — {e}")
                continue

            doc_id = rec.get("doc_id") or rec.get("id", "").rsplit("_Q", 1)[0]
            role   = rec.get("role", "FAC")
            label  = rec.get("label")
            signal = rec.get("signal", "NEUTRAL")
            if isinstance(signal, dict):
                signal = signal.get("label", "NEUTRAL")

            docs[doc_id]["roles"][role].append({
                "question": rec.get("question", ""),
                "answer":   rec.get("answer",   ""),
                "signal":   signal,
            })
            if docs[doc_id]["label"] is None and label is not None:
                docs[doc_id]["label"] = int(label)

    docs = {k: v for k, v in docs.items() if v["label"] is not None}
    print(f"Loaded {len(docs)} labelled documents from '{path}'")
    return docs


# ──────────────────────────────────────────────────────────────
# 3.  InLegalBERT Embedder + Track-E gating
# ──────────────────────────────────────────────────────────────
class InLegalBERTEmbedder:
    def __init__(self, model_name: str = "law-ai/InLegalBERT"):
        print(f"Loading {model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModel.from_pretrained(model_name)
        self.model.eval().to(DEVICE)

    @torch.no_grad()
    def _mean_pool(self, h, mask):
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

    @torch.no_grad()
    def embed(self, texts, batch_size=16):
        out = []
        cleaned = [t if t.strip() else "[PAD]" for t in texts]
        for s in range(0, len(cleaned), batch_size):
            batch = cleaned[s: s + batch_size]
            enc   = self.tokenizer(
                batch, padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt"
            )
            iids  = enc["input_ids"].to(DEVICE)
            amask = enc["attention_mask"].to(DEVICE)
            ttype = enc.get("token_type_ids")
            if ttype is not None:
                ttype = ttype.to(DEVICE)
            h = self.model(input_ids=iids, attention_mask=amask,
                           token_type_ids=ttype).last_hidden_state
            out.append(self._mean_pool(h, amask).cpu().numpy())
            print(f"  Embedded {min(s+batch_size, len(cleaned))}/{len(cleaned)}", end="\r")
        print()
        return np.vstack(out)

    def build_doc_embeddings(self, docs: dict) -> dict:
        all_texts, all_meta = [], []
        for doc_id, doc in docs.items():
            for role, qas in doc["roles"].items():
                for qa in qas:
                    q = qa["question"].strip()
                    a = qa["answer"].strip()
                    all_texts.append(f"[Q] {q} [A] {a}" if q else f"[A] {a}")
                    all_meta.append((doc_id, role,
                                     SIGNAL_MAP.get(qa["signal"], 0.0)))

        print(f"\nEmbedding {len(all_texts)} QA pairs across {len(docs)} docs ...")
        embs = self.embed(all_texts)

        role_embs    = defaultdict(lambda: defaultdict(list))
        role_signals = defaultdict(lambda: defaultdict(list))
        for i, (doc_id, role, sig) in enumerate(all_meta):
            role_embs[doc_id][role].append(embs[i])
            role_signals[doc_id][role].append(sig)

        for doc_id, doc in docs.items():
            node_feats = np.zeros((NUM_ROLES, EMB_DIM), dtype=np.float32)
            for role_name, ridx in ROLE2IDX.items():
                vecs = role_embs[doc_id].get(role_name, [])
                sigs = role_signals[doc_id].get(role_name, [])
                if vecs:
                    h_r = np.mean(vecs, axis=0)
                    es  = float(np.mean(sigs))
                    node_feats[ridx] = h_r * (1.0 + es)
            doc["node_feats"] = node_feats

        print("Embedding & gating complete.")
        return docs


# ──────────────────────────────────────────────────────────────
# 4.  Dataset
# ──────────────────────────────────────────────────────────────
class LegalQADataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        doc_id, feats, label = self.items[idx]
        return (torch.tensor(feats,  dtype=torch.float32),
                torch.tensor(label,  dtype=torch.float32),
                doc_id)

def collate(batch):
    feats, labels, ids = zip(*batch)
    return torch.stack(feats), torch.stack(labels), list(ids)

def make_items(docs):
    return [(doc_id, doc["node_feats"], doc["label"])
            for doc_id, doc in docs.items()
            if "node_feats" in doc and doc["label"] is not None]


# ──────────────────────────────────────────────────────────────
# 5.  Stratified split
# ──────────────────────────────────────────────────────────────
def stratified_split(items, train_ratio=TRAIN_R, seed=SEED):
    labels = np.array([it[2] for it in items])
    sss    = StratifiedShuffleSplit(n_splits=1,
                                    test_size=1 - train_ratio,
                                    random_state=seed)
    train_idx, test_idx = next(sss.split(np.zeros(len(labels)), labels))
    return train_idx.tolist(), test_idx.tolist()


# ──────────────────────────────────────────────────────────────
# 6.  Model components
# ──────────────────────────────────────────────────────────────
class CausalAdjacency(nn.Module):
    def __init__(self, n=NUM_ROLES):
        super().__init__()
        self.W_raw = nn.Parameter(torch.randn(n, n) * 0.1)
        mask = torch.triu(torch.ones(n, n), diagonal=1)
        self.register_buffer("mask", mask)
    def forward(self):
        return torch.sigmoid(self.W_raw) * self.mask


class CausalMP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lin = nn.Linear(dim, dim)
        self.act = nn.GELU()
        self.ln  = nn.LayerNorm(dim)
    def forward(self, H, W):
        Ht = self.act(self.lin(H))
        M  = torch.einsum("ij,bid->bjd", W, Ht)
        return self.ln(H + M)


class CausalGNN(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                 num_rounds=NUM_MP,
                 dropout_p=DROPOUT_P, dropout_c=DROPOUT_C):
        super().__init__()
        self.adj = CausalAdjacency(NUM_ROLES)

        self.proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_p),
        )
        self.mp = nn.ModuleList([CausalMP(hidden_dim) for _ in range(num_rounds)])

        self.cls = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_c),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_c / 2),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x, return_adj=False):
        W = self.adj()
        H = self.proj(x)
        for mp in self.mp:
            H = mp(H, W)
        mean_p = H.mean(1)
        max_p  = H.max(1).values
        pooled = torch.cat([mean_p, max_p], -1)
        logits = self.cls(pooled).squeeze(-1)
        return (logits, W) if return_adj else logits

    def get_adjacency(self):
        with torch.no_grad():
            return self.adj().cpu().numpy()


# ──────────────────────────────────────────────────────────────
# 7.  Label-smoothing BCE
# ──────────────────────────────────────────────────────────────
class LabelSmoothBCE(nn.Module):
    def __init__(self, pos_weight, epsilon=LABEL_SM):
        super().__init__()
        self.eps = epsilon
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        soft = targets * (1 - self.eps) + (1 - targets) * self.eps
        return self.bce(logits, soft)


# ──────────────────────────────────────────────────────────────
# 8.  Mixup augmentation — ② ALWAYS active
# ──────────────────────────────────────────────────────────────
def mixup_batch(feats, labels, alpha=MIXUP_A):
    """
    Mixup on graph node features.
    Applied during ALL training epochs (not disabled post-SWA).
    """
    if alpha <= 0:
        return feats, labels
    lam = np.random.beta(alpha, alpha)
    B   = feats.size(0)
    idx = torch.randperm(B, device=feats.device)
    mixed_feats  = lam * feats + (1 - lam) * feats[idx]
    mixed_labels = lam * labels + (1 - lam) * labels[idx]
    return mixed_feats, mixed_labels


# ──────────────────────────────────────────────────────────────
# 9.  Metrics
# ──────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    wf1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    mf1 = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5
    return {"acc": acc, "wf1": wf1, "mf1": mf1, "mcc": mcc, "auc": auc}


# ──────────────────────────────────────────────────────────────
# 10. Evaluate helper
# ──────────────────────────────────────────────────────────────
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    all_p, all_y, all_prob = [], [], []
    with torch.no_grad():
        for feats, labels, _ in loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            logits = model(feats)
            total_loss += loss_fn(logits, labels).item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_p.extend((probs >= 0.5).astype(int))
            all_y.extend(labels.cpu().long().numpy())
    m = compute_metrics(np.array(all_y), np.array(all_p), np.array(all_prob))
    return total_loss / len(loader), m


# ──────────────────────────────────────────────────────────────
# 11. pos_weight helper
# ──────────────────────────────────────────────────────────────
def get_pos_weight(items):
    labels = [it[2] for it in items]
    n_pos  = sum(labels)
    n_neg  = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return torch.tensor(1.0)
    return torch.tensor(n_neg / n_pos, dtype=torch.float32)


# ──────────────────────────────────────────────────────────────
# 12. ⑧ Linear warmup + plateau scheduler combo
# ──────────────────────────────────────────────────────────────
class WarmupThenPlateau:
    """
    Epochs 1..WARMUP_EP : linearly ramp LR from LR/10 → LR
    Epochs WARMUP_EP+1.. : hand off to ReduceLROnPlateau
    ③ Plateau scheduler stays active even during SWA phase so
       the base model cannot free-overfit after epoch 30.
    """
    def __init__(self, optimizer, warmup_epochs, base_lr, plateau_scheduler):
        self.opt       = optimizer
        self.warmup_ep = warmup_epochs
        self.base_lr   = base_lr
        self.plateau   = plateau_scheduler
        self._ep       = 0

    def step(self, val_loss=None):
        self._ep += 1
        if self._ep <= self.warmup_ep:
            lr = self.base_lr * (self._ep / self.warmup_ep)
            for pg in self.opt.param_groups:
                pg["lr"] = lr
        elif val_loss is not None:
            self.plateau.step(val_loss)

    def get_lr(self):
        return self.opt.param_groups[0]["lr"]


# ──────────────────────────────────────────────────────────────
# 13. Training loop — v3: all 8 fixes applied
# ──────────────────────────────────────────────────────────────
def train(model, train_loader, test_loader,
          pos_weight, epochs=EPOCHS, lr=LR):
    model.to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr / 10, weight_decay=WD)
    # ③ plateau stays active throughout (including SWA phase)
    _plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=6, min_lr=1e-6, verbose=True
    )
    # ⑧ warmup wrapper
    scheduler = WarmupThenPlateau(opt, WARMUP_EP, lr, _plateau)

    # ⑤ SWA model — snapshots start at SWA_START
    swa_model  = AveragedModel(model)
    swa_sch    = SWALR(opt, swa_lr=SWA_LR, anneal_epochs=5)
    swa_active = False

    loss_fn = LabelSmoothBCE(pos_weight.to(DEVICE))

    history = {k: [] for k in
               ["train_loss", "test_loss",
                "acc", "wf1", "mf1", "auc", "mcc", "lr",
                "eval_source"]}   # tracks "base" or "swa"

    best_wf1      = 0.0
    best_state    = None
    best_is_swa   = False
    pat_ctr       = 0
    swa_snapshots = 0

    for ep in range(1, epochs + 1):

        # ── Activate SWA snapshot collection ──────────────────
        if ep == SWA_START and not swa_active:
            swa_active = True
            print(f"\n  [SWA] Snapshot collection starts at epoch {ep}")
            print(f"  [SWA] Evaluation switches to SWA model at epoch {SWA_EVAL_AFTER}")

        # ── Train base model ───────────────────────────────────
        model.train()
        total = 0.0
        for feats, labels, _ in train_loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            # ② Mixup always active — never disabled
            feats, labels = mixup_batch(feats, labels, MIXUP_A)
            opt.zero_grad()
            loss = loss_fn(model(feats), labels)
            loss.backward()
            # ⑦ clip only base model gradients
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step()
            total += loss.item()

        tr_loss = total / len(train_loader)

        # ── SWA: collect snapshot ──────────────────────────────
        if swa_active:
            swa_model.update_parameters(model)
            swa_snapshots += 1
            swa_sch.step()

            # ⑤ Intermediate BN update every SWA_BN_EVERY epochs
            if swa_snapshots % SWA_BN_EVERY == 0:
                update_bn(train_loader, swa_model, device=DEVICE)

        # ── ① Choose eval model based on snapshot maturity ────
        # Before SWA_EVAL_AFTER: always evaluate base model
        # After  SWA_EVAL_AFTER: evaluate SWA model (≥10 snapshots)
        use_swa_eval = swa_active and (ep >= SWA_EVAL_AFTER)
        eval_model   = swa_model if use_swa_eval else model
        eval_source  = "swa" if use_swa_eval else "base"

        te_loss, m = evaluate(eval_model, test_loader, loss_fn)
        current_lr = scheduler.get_lr()

        # ③ Plateau scheduler always steps (base model stays regularised)
        if swa_active:
            # During SWA: also step plateau so base LR doesn't stay too high
            scheduler.step(val_loss=te_loss)
        else:
            scheduler.step(val_loss=te_loss)

        for k in ["acc", "wf1", "mf1", "auc", "mcc"]:
            history[k].append(m[k])
        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        history["lr"].append(current_lr)
        history["eval_source"].append(eval_source)

        flag    = "★" if m["wf1"] > best_wf1 else " "
        src_tag = f"[{eval_source.upper():4s}]"
        print(f"Ep {ep:3d}/{epochs} {flag} {src_tag} | lr {current_lr:.2e} | "
              f"TrL {tr_loss:.4f} | TeL {te_loss:.4f} | "
              f"Acc {m['acc']:.4f} | wF1 {m['wf1']:.4f} | "
              f"AUC {m['auc']:.4f} | MCC {m['mcc']:.4f}")

        if m["wf1"] > best_wf1:
            best_wf1    = m["wf1"]
            best_is_swa = use_swa_eval
            if use_swa_eval:
                # ④ Save SWA model state (post intermediate BN update)
                best_state = {k: v.clone()
                              for k, v in swa_model.state_dict().items()}
            else:
                # Pre-SWA-eval: save base model
                best_state = {k: v.clone()
                              for k, v in model.state_dict().items()}
            pat_ctr = 0
        else:
            pat_ctr += 1
            if pat_ctr >= ES_PAT:
                print(f"\nEarly stop at epoch {ep}  (patience={ES_PAT})")
                break

    # ── Final BN update on full SWA model ─────────────────────
    if swa_active:
        print("\nFinal SWA BN statistics update ...")
        update_bn(train_loader, swa_model, device=DEVICE)

    # ── ④ Restore best checkpoint ─────────────────────────────
    if best_state:
        if best_is_swa:
            swa_model.load_state_dict(best_state)
            print(f"Restored best SWA model  (wF1 = {best_wf1:.4f})")
        else:
            model.load_state_dict(best_state)
            print(f"Restored best base model  (wF1 = {best_wf1:.4f})")

    final_model = swa_model if swa_active else model
    return history, final_model


# ──────────────────────────────────────────────────────────────
# 14. Full inference
# ──────────────────────────────────────────────────────────────
def full_inference(model, loader):
    model.eval()
    all_true, all_pred, all_prob, all_ids = [], [], [], []
    with torch.no_grad():
        for feats, labels, ids in loader:
            feats = feats.to(DEVICE)
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_prob.extend(probs)
            all_pred.extend((probs >= 0.5).astype(int))
            all_true.extend(labels.long().numpy())
            all_ids.extend(ids)
    return (np.array(all_true), np.array(all_pred),
            np.array(all_prob), all_ids)


# ──────────────────────────────────────────────────────────────
# 15. Plots
# ──────────────────────────────────────────────────────────────
PALETTE = {
    "train": "#4C72B0",
    "val"  : "#DD8452",
    "pos"  : "#55A868",
    "neg"  : "#C44E52",
    "bg"   : "#F8F9FA",
}

def _savefig(name):
    path = f"{PLOT_DIR}/{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")

def _ema(x, alpha=0.2):
    out, s = [], x[0]
    for v in x:
        s = alpha * v + (1 - alpha) * s
        out.append(s)
    return out


# P1 — Loss curves with eval-source shading ───────────────────
def plot_losses(history):
    fig, ax = plt.subplots(figsize=(11, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    eps = range(1, len(history["train_loss"]) + 1)

    ax.plot(eps, history["train_loss"], color=PALETTE["train"],
            lw=1, alpha=0.35)
    ax.plot(eps, _ema(history["train_loss"]), color=PALETTE["train"],
            lw=2.2, label="Train loss (EMA)")

    ax.plot(eps, history["test_loss"], color=PALETTE["val"],
            lw=1, alpha=0.35, linestyle="--")
    ax.plot(eps, _ema(history["test_loss"]), color=PALETTE["val"],
            lw=2.2, linestyle="--", label="Val loss (EMA)")

    # ① Shade the "immature SWA" zone (SWA_START → SWA_EVAL_AFTER)
    if SWA_START <= len(history["train_loss"]):
        ax.axvspan(SWA_START, min(SWA_EVAL_AFTER, len(history["train_loss"])),
                   alpha=0.08, color="gray", label="SWA snapshot phase\n(base model evaluated)")
        ax.axvline(SWA_START, color="#888", lw=1.2, linestyle=":",
                   label=f"SWA start (ep {SWA_START})")
    if SWA_EVAL_AFTER <= len(history["train_loss"]):
        ax.axvline(SWA_EVAL_AFTER, color="#4a9", lw=1.5, linestyle="--",
                   label=f"SWA eval active (ep {SWA_EVAL_AFTER})")

    gap = abs(history["train_loss"][-1] - history["test_loss"][-1])
    ax.annotate(f"Gap: {gap:.4f}",
                xy=(len(eps), (history["train_loss"][-1] +
                               history["test_loss"][-1]) / 2),
                xytext=(-70, 0), textcoords="offset points",
                fontsize=9, color="#555",
                arrowprops=dict(arrowstyle="->", color="#999"))

    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("P1 — Training vs Validation Loss (v3 — Stable, no SWA crash)",
                 fontweight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.annotate(f'{history["train_loss"][-1]:.4f}',
                xy=(len(eps), history["train_loss"][-1]),
                xytext=(-35, 8), textcoords="offset points",
                color=PALETTE["train"], fontsize=8)
    ax.annotate(f'{history["test_loss"][-1]:.4f}',
                xy=(len(eps), history["test_loss"][-1]),
                xytext=(-35, -14), textcoords="offset points",
                color=PALETTE["val"], fontsize=8)
    _savefig("P1_loss_curves")


# P2 — Metric curves with SWA eval boundary ───────────────────
def plot_metrics(history):
    metrics = [("acc", "Accuracy", PALETTE["train"]),
               ("wf1", "Weighted F1", PALETTE["val"]),
               ("auc", "AUC-ROC", PALETTE["pos"]),
               ("mcc", "MCC", PALETTE["neg"])]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor=PALETTE["bg"])
    fig.suptitle("P2 — Metrics over Epochs (v3 — No SWA crash)",
                 fontweight="bold", fontsize=13)
    axes = axes.flatten()
    eps  = range(1, len(history["acc"]) + 1)
    for ax, (key, title, color) in zip(axes, metrics):
        ax.set_facecolor(PALETTE["bg"])
        ax.plot(eps, history[key], color=color, lw=1, alpha=0.4)
        ax.plot(eps, _ema(history[key], 0.25), color=color, lw=2.2)
        ax.axhline(max(history[key]), color=color, lw=1,
                   linestyle=":", alpha=0.6)
        n = len(history[key])
        if SWA_START <= n:
            ax.axvspan(SWA_START, min(SWA_EVAL_AFTER, n),
                       alpha=0.06, color="gray")
            ax.axvline(SWA_START, color="#888", lw=1, linestyle=":")
        if SWA_EVAL_AFTER <= n:
            ax.axvline(SWA_EVAL_AFTER, color="#4a9", lw=1.2, linestyle="--")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel(title)
        ax.grid(alpha=0.3)
        best_ep = int(np.argmax(history[key])) + 1
        ax.annotate(f"Best: {max(history[key]):.4f} @ ep {best_ep}",
                    xy=(best_ep, max(history[key])),
                    xytext=(10, -14), textcoords="offset points",
                    fontsize=8, color=color)
    _savefig("P2_metric_curves")


# P3–P9 (unchanged logic, v3 titles) ─────────────────────────
def plot_confusion(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    labels_str = ["Rejected (0)", "Affirmed (1)"]
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_str, yticklabels=labels_str,
                linewidths=0.5, linecolor="#cccccc", ax=ax,
                annot_kws={"size": 14, "weight": "bold"})
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual",    fontsize=11)
    ax.set_title("P3 — Confusion Matrix (v3)", fontweight="bold", fontsize=12)
    for i, (row, lbl) in enumerate(zip(cm, labels_str)):
        pct = row[i] / row.sum() * 100 if row.sum() else 0
        ax.text(i + 0.5, i + 0.7, f"({pct:.1f}%)",
                ha="center", va="center", fontsize=9, color="#333333")
    _savefig("P3_confusion_matrix")

def plot_roc(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val      = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(fpr, tpr, color=PALETTE["train"], lw=2,
            label=f"ROC (AUC = {auc_val:.4f})")
    ax.fill_between(fpr, tpr, alpha=0.10, color=PALETTE["train"])
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("P4 — ROC Curve (v3)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P4_roc_curve")

def plot_pr(y_true, y_prob):
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.plot(rec, prec, color=PALETTE["pos"], lw=2,
            label=f"PR (AP = {ap:.4f})")
    ax.fill_between(rec, prec, alpha=0.10, color=PALETTE["pos"])
    baseline = y_true.mean()
    ax.axhline(baseline, color="gray", lw=1, linestyle="--",
               label=f"Baseline ({baseline:.2f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("P5 — Precision-Recall Curve (v3)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P5_pr_curve")

def plot_cls_report(y_true, y_pred):
    report = classification_report(y_true, y_pred,
                                   target_names=["Rejected", "Affirmed"],
                                   output_dict=True)
    rows = ["Rejected", "Affirmed", "macro avg", "weighted avg"]
    cols = ["precision", "recall", "f1-score", "support"]
    data = [[report[r][c] for c in cols] for r in rows]
    fig, ax = plt.subplots(figsize=(9, 3.5), facecolor=PALETTE["bg"])
    ax.axis("off")
    tbl = ax.table(
        cellText=[[f"{v:.4f}" if isinstance(v, float) else str(int(v))
                   for v in row] for row in data],
        rowLabels=rows, colLabels=[c.title() for c in cols],
        cellLoc="center", loc="center",
    )
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2.0)
    for j in range(len(cols)):
        tbl[(0, j)].set_facecolor("#4C72B0")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows) + 1):
        tbl[(i, -1)].set_facecolor("#E8EDF5")
        tbl[(i, -1)].set_text_props(fontweight="bold")
    ax.set_title("P6 — Classification Report (v3)", fontweight="bold",
                 fontsize=12, pad=12)
    _savefig("P6_classification_report")
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(y_true, y_pred,
                                target_names=["Rejected", "Affirmed"]))

def plot_adjacency(model):
    base = model.module if hasattr(model, "module") else model
    W    = base.get_adjacency()
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=PALETTE["bg"])
    sns.heatmap(W, annot=True, fmt=".3f", cmap="YlOrRd",
                xticklabels=ROLES, yticklabels=ROLES,
                linewidths=0.4, linecolor="#cccccc",
                vmin=0, vmax=1, ax=ax, annot_kws={"size": 9})
    ax.set_title("P7 — Learned Causal Adjacency W[row → col] (v3)",
                 fontweight="bold")
    ax.set_xlabel("Target Role"); ax.set_ylabel("Source Role")
    _savefig("P7_adjacency_matrix")
    edges = sorted(
        [(W[i, j], ROLES[i], ROLES[j])
         for i in range(NUM_ROLES) for j in range(NUM_ROLES)
         if i < j and W[i, j] > 0.05], reverse=True
    )
    print("\nTop-5 Causal Edges:")
    for w, s, t in edges[:5]:
        print(f"  {s:>10} → {t:<10}  weight: {w:.4f}")

def plot_transplant(flip_rates: dict):
    roles  = list(flip_rates.keys())
    rates  = [flip_rates[r] for r in roles]
    colors = [PALETTE["neg"] if r == max(flip_rates, key=flip_rates.get)
              else PALETTE["train"] for r in roles]
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    bars = ax.bar(roles, rates, color=colors, edgecolor="white", width=0.55)
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
    ax.set_ylim(0, max(rates) * 1.25 if max(rates) > 0 else 0.5)
    ax.set_ylabel("Flip Rate (Affirm → Reject)")
    ax.set_title("P8 — Transplant Experiment: Decisive Role Map (v3)",
                 fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    decisive = max(flip_rates, key=flip_rates.get)
    ax.text(0.5, 0.92, f"Most decisive: {decisive}",
            ha="center", va="center", transform=ax.transAxes,
            fontsize=10, style="italic", color=PALETTE["neg"])
    _savefig("P8_transplant_flip_rates")

def plot_prob_dist(y_true, y_prob):
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])
    ax.hist(y_prob[y_true == 0], bins=25, alpha=0.65,
            color=PALETTE["neg"], label="True Rejected (0)", edgecolor="white")
    ax.hist(y_prob[y_true == 1], bins=25, alpha=0.65,
            color=PALETTE["pos"], label="True Affirmed (1)", edgecolor="white")
    ax.axvline(0.5, color="black", lw=1.5, linestyle="--", label="Threshold 0.5")
    ax.set_xlabel("P(Affirmed)"); ax.set_ylabel("Count")
    ax.set_title("P9 — Prediction Probability Distribution (v3)", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    _savefig("P9_prob_distribution")


# ──────────────────────────────────────────────────────────────
# 16. Transplant Experiment
# ──────────────────────────────────────────────────────────────
def transplant_experiment(model, docs, n_pairs=200):
    model.eval()
    affirmed, rejected = [], []
    with torch.no_grad():
        for doc_id, doc in docs.items():
            if "node_feats" not in doc:
                continue
            feats = torch.tensor(doc["node_feats"],
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            try:
                logits, _ = model(feats, return_adj=True)
            except TypeError:
                logits = model(feats)
            prob = torch.sigmoid(logits).item()
            pred = int(prob >= 0.5)
            if pred == doc["label"]:
                (affirmed if doc["label"] == 1 else rejected).append(doc)

    print(f"\nTransplant pool — Affirmed: {len(affirmed)}, Rejected: {len(rejected)}")
    if not affirmed or not rejected:
        print("[WARN] Transplant skipped — insufficient correctly-predicted cases.")
        return {r: 0.0 for r in ROLES}

    flip_counts  = {r: 0 for r in ROLES}
    total_pairs  = {r: 0 for r in ROLES}
    random.shuffle(affirmed); random.shuffle(rejected)

    for doc_A in affirmed[:n_pairs]:
        doc_B = random.choice(rejected)
        for ridx, role in enumerate(ROLES):
            emb_A  = doc_A["node_feats"].copy()
            f_orig = torch.tensor(emb_A, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_orig, return_adj=True)
                except TypeError:
                    logits = model(f_orig)
                p_orig = int(torch.sigmoid(logits).item() >= 0.5)

            transplanted       = emb_A.copy()
            transplanted[ridx] = doc_B["node_feats"][ridx]
            f_new = torch.tensor(transplanted,
                                 dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                try:
                    logits, _ = model(f_new, return_adj=True)
                except TypeError:
                    logits = model(f_new)
                p_new = int(torch.sigmoid(logits).item() >= 0.5)

            if p_orig == 1 and p_new == 0:
                flip_counts[role] += 1
            total_pairs[role] += 1

    flip_rates = {r: flip_counts[r] / max(total_pairs[r], 1) for r in ROLES}
    print("\n" + "="*55)
    print("TRANSPLANT — Decisive Role Map  (flip rate 1→0)")
    print("="*55)
    for role in ROLES:
        bar = "█" * int(flip_rates[role] * 40)
        print(f"  {role:<10}  {flip_rates[role]:.3f}  {bar}")
    decisive = max(flip_rates, key=flip_rates.get)
    print(f"\nMost decisive role: {decisive} ({flip_rates[decisive]:.3f})")
    return flip_rates


# ──────────────────────────────────────────────────────────────
# 17. Final summary
# ──────────────────────────────────────────────────────────────
def print_final_summary(y_true, y_pred, y_prob):
    m = compute_metrics(y_true, y_pred, y_prob)
    print("\n" + "="*60)
    print("FINAL TEST METRICS (v3)")
    print("="*60)
    print(f"  Accuracy    : {m['acc']:.4f}  {'✓' if m['acc']>=0.84 else '✗'}  (target ≥ 0.84)")
    print(f"  Weighted F1 : {m['wf1']:.4f}  {'✓' if m['wf1']>=0.85 else '✗'}  (target ≥ 0.85)")
    print(f"  Macro F1    : {m['mf1']:.4f}")
    print(f"  AUC-ROC     : {m['auc']:.4f}")
    print(f"  MCC         : {m['mcc']:.4f}")
    print("="*60)
    return m


# ──────────────────────────────────────────────────────────────
# 18. Main
# ──────────────────────────────────────────────────────────────
def main():
    DATA_PATH = "cjpe_100k_qa_flat.jsonl"

    print("\n" + "="*60)
    print("STAGE 1 — Loading QA pairs")
    print("="*60)
    docs = load_qa_jsonl(DATA_PATH)

    print("\n" + "="*60)
    print("STAGE 2-3 — Embedding + Track-E Entailment Gating")
    print("="*60)
    embedder = InLegalBERTEmbedder("law-ai/InLegalBERT")
    docs     = embedder.build_doc_embeddings(docs)

    items   = make_items(docs)
    n_total = len(items)
    train_idx, test_idx = stratified_split(items, TRAIN_R)
    n_train, n_test = len(train_idx), len(test_idx)

    dataset  = LegalQADataset(items)
    train_ds = Subset(dataset, train_idx)
    test_ds  = Subset(dataset, test_idx)

    train_items = [items[i] for i in train_idx]
    pw = get_pos_weight(train_items)
    print(f"\nDataset — total: {n_total} | train: {n_train} | test: {n_test}")
    print(f"Pos-weight for BCE: {pw.item():.4f}")

    tr_labels = [items[i][2] for i in train_idx]
    te_labels = [items[i][2] for i in test_idx]
    print(f"Train  —  +: {sum(tr_labels)} | -: {n_train - sum(tr_labels)}")
    print(f"Test   —  +: {sum(te_labels)} | -: {n_test  - sum(te_labels)}")

    train_loader = DataLoader(train_ds, batch_size=BATCH,
                              shuffle=True,  collate_fn=collate)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH,
                              shuffle=False, collate_fn=collate)

    print("\n" + "="*60)
    print("STAGE 4-5 — Training Stable Causal GNN v3")
    print("="*60)
    model = CausalGNN(emb_dim=EMB_DIM, hidden_dim=HIDDEN,
                      num_rounds=NUM_MP,
                      dropout_p=DROPOUT_P, dropout_c=DROPOUT_C)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")
    print(f"\nKey v3 hyperparameters:")
    print(f"  LR={LR}  WD={WD}  CLIP={CLIP}")
    print(f"  HIDDEN={HIDDEN}  NUM_MP={NUM_MP}")
    print(f"  DROPOUT_P={DROPOUT_P}  DROPOUT_C={DROPOUT_C}")
    print(f"  WARMUP_EP={WARMUP_EP}")
    print(f"  SWA snapshot start = epoch {SWA_START}")
    print(f"  SWA eval active    = epoch {SWA_EVAL_AFTER}  (≥10 snapshots)")
    print(f"  SWA_BN_EVERY={SWA_BN_EVERY}  SWA_LR={SWA_LR}")
    print(f"  Mixup α={MIXUP_A} (always on)  Label smoothing ε={LABEL_SM}")

    history, final_model = train(model, train_loader, test_loader,
                                 pos_weight=pw, epochs=EPOCHS, lr=LR)

    torch.save(model.state_dict(), "causal_gnn_stable_v3.pt")
    print("Model saved → causal_gnn_stable_v3.pt")

    print("\n" + "="*60)
    print("STAGE 6 — Inference & Full Evaluation")
    print("="*60)
    y_true, y_pred, y_prob, _ = full_inference(final_model, test_loader)
    print_final_summary(y_true, y_pred, y_prob)

    print("\n" + "="*60)
    print("STAGE 7 — Cross-Case Transplant Experiment")
    print("="*60)
    flip_rates = transplant_experiment(final_model, docs, n_pairs=200)

    print("\n" + "="*60)
    print("STAGE 8 — Generating all diagnostic plots")
    print("="*60)
    plot_losses(history)
    plot_metrics(history)
    plot_confusion(y_true, y_pred)
    plot_roc(y_true, y_prob)
    plot_pr(y_true, y_prob)
    plot_cls_report(y_true, y_pred)
    plot_adjacency(final_model)
    plot_transplant(flip_rates)
    plot_prob_dist(y_true, y_prob)

    print("\n✓ All 9 plots saved.")
    for i, name in enumerate(
        ["P1_loss_curves", "P2_metric_curves",
         "P3_confusion_matrix", "P4_roc_curve",
         "P5_pr_curve", "P6_classification_report",
         "P7_adjacency_matrix", "P8_transplant_flip_rates",
         "P9_prob_distribution"], start=1
    ):
        print(f"  {i}. {name}.png")

    return model, history, y_true, y_pred, y_prob, flip_rates


if __name__ == "__main__":
    model, history, y_true, y_pred, y_prob, flip_rates = main()

Using device: cuda

STAGE 1 — Loading QA pairs
Loaded 5000 labelled documents from 'cjpe_100k_qa_flat.jsonl'

STAGE 2-3 — Embedding + Track-E Entailment Gating
Loading law-ai/InLegalBERT ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding 99454 QA pairs across 5000 docs ...
  Embedded 99454/99454
Embedding & gating complete.

Dataset — total: 5000 | train: 4000 | test: 1000
Pos-weight for BCE: 1.3296
Train  —  +: 1717 | -: 2283
Test   —  +: 429 | -: 571

STAGE 4-5 — Training Stable Causal GNN v3
Model parameters: 494,885

Key v3 hyperparameters:
  LR=1e-05  WD=0.0005  CLIP=0.5
  HIDDEN=256  NUM_MP=2
  DROPOUT_P=0.5  DROPOUT_C=0.35
  WARMUP_EP=5
  SWA snapshot start = epoch 30
  SWA eval active    = epoch 40  (≥10 snapshots)
  SWA_BN_EVERY=10  SWA_LR=3e-05
  Mixup α=0.3 (always on)  Label smoothing ε=0.05
Ep   1/80 ★ [BASE] | lr 1.00e-06 | TrL 0.7940 | TeL 0.7905 | Acc 0.5740 | wF1 0.4236 | AUC 0.7019 | MCC 0.0531
Ep   2/80 ★ [BASE] | lr 2.00e-06 | TrL 0.7906 | TeL 0.7874 | Acc 0.5910 | wF1 0.4662 | AUC 0.7850 | MCC 0.1437
Ep   3/80 ★ [BASE] | lr 4.00e-06 | TrL 0.7895 | TeL 0.7819 | Acc 0.6110 | wF1 0.5056 | AUC 0.8265 | MCC 0.2183
Ep   4/80 ★ [BASE] | lr 6.00e-06 | TrL 0.7830 | TeL 0.7666 | Acc 0.7510 | wF1 0